# K — Explanation Layer

This notebook implements the explanation and model-evidence component of the
KGRec-inspired multimodal recommender study.

The recommendation models, hyperparameters and test results are treated as
frozen inputs. No model retraining or hyperparameter selection is performed
within the explanation layer.

The analysis addresses RQ4 by examining whether patterns identified in
quantitative recommendation evaluation are consistent with evidence exposed
through model-analysis and explanation methods.

Explanation outputs are interpreted as evidence about model behaviour and
sensitivity. Attention, fusion weights, graph evidence and perturbation
responses are not automatically treated as faithful causal explanations.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

## K1 — Explanation Layer Setup and Frozen-Model Readiness

K1 establishes the frozen inputs required by the explanation analysis,
including the final KGRec-NV and KGRec-MM checkpoints, paired test results,
RQ3 analysis outputs and aligned business modality features.

The purpose of this stage is to verify provenance and consistency before
extracting explanation evidence.

In [3]:
# --------------------------------------------------
# Locate processed_data
# --------------------------------------------------

possible_processed_paths = [
    Path.cwd() / "processed_data",
    Path.cwd().parent / "processed_data",
    Path.cwd().parent.parent / "processed_data"
]


existing_processed_paths = [
    path
    for path
    in possible_processed_paths
    if path.exists()
]


if len(existing_processed_paths) == 0:
    raise FileNotFoundError(
        "Could not locate the processed_data directory."
    )


processed_data_root = (
    existing_processed_paths[0]
)


final_model_dir = (
    processed_data_root
    /
    "new_orleans_model_outputs"
)


model_inputs_dir = (
    processed_data_root
    /
    "new_orleans_model_inputs"
)


explanation_dir = (
    final_model_dir
    /
    "explanations"
)


explanation_dir.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Processed data:",
    processed_data_root
)

print(
    "Model outputs:",
    final_model_dir
)

print(
    "Explanation outputs:",
    explanation_dir
)

Processed data: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data
Model outputs: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_model_outputs
Explanation outputs: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_model_outputs/explanations


In [4]:
nv_checkpoint_path = (
    final_model_dir
    /
    "kgrec_nv_final_checkpoint.pt"
)


mm_checkpoint_path = (
    final_model_dir
    /
    "kgrec_mm_final_checkpoint.pt"
)


checkpoint_readiness = pd.Series({

    "KGRec-NV checkpoint":
        nv_checkpoint_path.exists(),

    "KGRec-MM checkpoint":
        mm_checkpoint_path.exists()
})


checkpoint_readiness

KGRec-NV checkpoint    True
KGRec-MM checkpoint    True
dtype: bool

In [5]:
required_explanation_inputs = {

    "Paired test results":
        final_model_dir
        /
        "paired_user_test_results.parquet",

    "RQ3 paired analysis":
        final_model_dir
        /
        "rq3_paired_test_analysis.parquet",

    "Paired bootstrap summary":
        final_model_dir
        /
        "paired_test_bootstrap_summary.csv",

    "Visual perturbation metrics":
        final_model_dir
        /
        "rq3_mm_visual_perturbation_metrics.csv",

    "Business-level RQ3 analysis":
        final_model_dir
        /
        "rq3_visual_business_level_analysis.parquet",

    "Adjusted rank visual effects":
        final_model_dir
        /
        "rq3_adjusted_rank_visual_effects.csv",

    "Adjusted top-K visual effects":
        final_model_dir
        /
        "rq3_clustered_topk_visual_effects.csv",

    "RQ3 synthesis":
        final_model_dir
        /
        "rq3_quantitative_synthesis.csv"
}


explanation_input_check = pd.Series({
    name:
        path.exists()

    for name, path
    in required_explanation_inputs.items()
})


explanation_input_check

Paired test results              True
RQ3 paired analysis              True
Paired bootstrap summary         True
Visual perturbation metrics      True
Business-level RQ3 analysis      True
Adjusted rank visual effects     True
Adjusted top-K visual effects    True
RQ3 synthesis                    True
dtype: bool

In [6]:
rq3_analysis = pd.read_parquet(
    final_model_dir
    /
    "rq3_paired_test_analysis.parquet"
)


print(
    "RQ3 rows:",
    len(rq3_analysis)
)

print(
    "Unique users:",
    rq3_analysis[
        "user_row"
    ].nunique()
)

print(
    "Unique test businesses:",
    rq3_analysis[
        "business_id"
    ].nunique()
)

RQ3 rows: 14991
Unique users: 14991
Unique test businesses: 1966


In [7]:
explanation_columns = [
    "user_row",
    "business_id",
    "business_row",
    "target_has_visual",
    "nv_rank",
    "mm_rank",
    "rank_improvement",
    "target_visual_score_gain",
    "target_visual_rank_gain",
    "mm_all_visual_off_rank",
    "mm_target_visual_off_rank"
]


explanation_column_check = pd.Series({

    column:
        column
        in rq3_analysis.columns

    for column
    in explanation_columns
})


explanation_column_check

user_row                     True
business_id                  True
business_row                 True
target_has_visual            True
nv_rank                      True
mm_rank                      True
rank_improvement             True
target_visual_score_gain     True
target_visual_rank_gain      True
mm_all_visual_off_rank       True
mm_target_visual_off_rank    True
dtype: bool

In [8]:
business_index = pd.read_parquet(
    model_inputs_dir
    /
    "new_orleans_business_entity_index.parquet"
)


user_index = pd.read_parquet(
    model_inputs_dir
    /
    "new_orleans_user_entity_index.parquet"
)


entity_index = pd.read_parquet(
    model_inputs_dir
    /
    "new_orleans_kg_entity_index.parquet"
)


relation_index = pd.read_parquet(
    model_inputs_dir
    /
    "new_orleans_kg_relation_index.parquet"
)

In [9]:
kg_index_check = pd.Series({

    "Business rows":
        len(
            business_index
        ),

    "User rows":
        len(
            user_index
        ),

    "Entity rows":
        len(
            entity_index
        ),

    "Relation rows":
        len(
            relation_index
        ),

    "Expected businesses":
        (
            len(
                business_index
            )
            ==
            2516
        ),

    "Expected users":
        (
            len(
                user_index
            )
            ==
            14991
        )
})


kg_index_check

Business rows           2516
User rows              14991
Entity rows            17819
Relation rows             64
Expected businesses     True
Expected users          True
dtype: object

In [10]:
mm_checkpoint = torch.load(
    mm_checkpoint_path,
    map_location="cpu",
    weights_only=False
)


print(
    "Checkpoint type:",
    type(mm_checkpoint)
)

Checkpoint type: <class 'dict'>


In [11]:
if (
    isinstance(
        mm_checkpoint,
        dict
    )
    and
    "model_state_dict"
    in mm_checkpoint
):

    mm_state_dict = (
        mm_checkpoint[
            "model_state_dict"
        ]
    )

elif (
    isinstance(
        mm_checkpoint,
        dict
    )
    and
    "state_dict"
    in mm_checkpoint
):

    mm_state_dict = (
        mm_checkpoint[
            "state_dict"
        ]
    )

else:

    mm_state_dict = (
        mm_checkpoint
    )


print(
    "State-dict tensors:",
    len(
        mm_state_dict
    )
)

State-dict tensors: 8


In [12]:
explanation_parameter_keys = [
    key
    for key
    in mm_state_dict.keys()
    if (
        "fusion"
        in key.lower()
        or
        "visual_projection"
        in key.lower()
        or
        "text_projection"
        in key.lower()
        or
        "relation"
        in key.lower()
    )
]


explanation_parameter_keys

['fusion_logits',
 'graph_encoder.relation_embedding.weight',
 'graph_encoder.relation_gate.weight',
 'text_projection.weight',
 'visual_projection.weight']

In [13]:
k1_readiness_check = pd.Series({

    "NV checkpoint available":
        nv_checkpoint_path.exists(),

    "MM checkpoint available":
        mm_checkpoint_path.exists(),

    "Paired RQ3 rows complete":
        (
            len(
                rq3_analysis
            )
            ==
            14991
        ),

    "One test row per user":
        (
            rq3_analysis[
                "user_row"
            ].nunique()
            ==
            14991
        ),

    "Business index complete":
        (
            len(
                business_index
            )
            ==
            2516
        ),

    "MM state dictionary loaded":
        (
            len(
                mm_state_dict
            )
            >
            0
        ),

    "Fusion parameter present":
        any(
            "fusion"
            in key.lower()

            for key
            in mm_state_dict.keys()
        ),

    "Visual projection present":
        any(
            "visual_projection"
            in key.lower()

            for key
            in mm_state_dict.keys()
        ),

    "Text projection present":
        any(
            "text_projection"
            in key.lower()

            for key
            in mm_state_dict.keys()
        )
})


k1_readiness_check

NV checkpoint available       True
MM checkpoint available       True
Paired RQ3 rows complete      True
One test row per user         True
Business index complete       True
MM state dictionary loaded    True
Fusion parameter present      True
Visual projection present     True
Text projection present       True
dtype: bool

## K2 — Global Modality and Fusion Behaviour

The first model-level explanation examines the learned fusion parameters of
the frozen KGRec-MM checkpoint.

For businesses with genuine visual features, graph, textual and visual
representations are combined using learned fusion logits. For businesses
without visual features, the visual branch is masked and the remaining
graph and text weights are renormalised.

These global fusion weights describe how the architecture allocates
representation weight across modalities. They are not recommendation-specific
feature importances and are therefore interpreted as model-level evidence
rather than causal attribution.

In [14]:
# --------------------------------------------------
# K2.1 — Locate learned fusion logits
# --------------------------------------------------

fusion_logit_keys = [
    key
    for key
    in mm_state_dict.keys()
    if key.endswith("fusion_logits")
]


print(
    "Fusion-logit keys:",
    fusion_logit_keys
)


assert len(fusion_logit_keys) == 1, (
    "Expected exactly one fusion_logits parameter."
)

Fusion-logit keys: ['fusion_logits']


In [15]:
fusion_logits = (
    mm_state_dict[
        fusion_logit_keys[0]
    ]
    .detach()
    .cpu()
    .float()
)


print(
    "Fusion logits:",
    fusion_logits
)

print(
    "Shape:",
    tuple(
        fusion_logits.shape
    )
)

Fusion logits: tensor([ 0.0054, -0.0098,  0.0033])
Shape: (3,)


### K2.2 — Convert logits to fusion weights

For businesses with visual evidence:

In [16]:
visual_supported_weights = (
    torch.softmax(
        fusion_logits,
        dim=0
    )
    .numpy()
)

In [17]:
image_less_weights = np.zeros(
    3,
    dtype=float
)


image_less_weights[
    :2
] = (
    torch.softmax(
        fusion_logits[:2],
        dim=0
    )
    .numpy()
)

In [18]:
global_fusion_summary = pd.DataFrame({

    "modality": [
        "Graph",
        "Text",
        "Visual"
    ],

    "fusion_logit":
        fusion_logits.numpy(),

    "visual_supported_weight":
        visual_supported_weights,

    "image_less_weight":
        image_less_weights
})


global_fusion_summary

,modality,fusion_logit,visual_supported_weight,image_less_weight
0,Graph,0.005374,0.335256,0.503802
1,Text,-0.009833,0.330196,0.496198
2,Visual,0.003258,0.334547,0.000000


In [19]:
k2_fusion_check = pd.Series({

    "Three modalities":
        (
            len(
                fusion_logits
            )
            ==
            3
        ),

    "Visual-supported weights sum to 1":
        np.isclose(
            visual_supported_weights.sum(),
            1.0
        ),

    "Image-less weights sum to 1":
        np.isclose(
            image_less_weights.sum(),
            1.0
        ),

    "Image-less visual weight is zero":
        np.isclose(
            image_less_weights[2],
            0.0
        ),

    "Visual-supported weights positive":
        bool(
            (
                visual_supported_weights
                >
                0
            ).all()
        ),

    "All values finite":
        bool(
            np.isfinite(
                np.concatenate([
                    fusion_logits.numpy(),
                    visual_supported_weights,
                    image_less_weights
                ])
            ).all()
        )
})


k2_fusion_check

Three modalities                     True
Visual-supported weights sum to 1    True
Image-less weights sum to 1          True
Image-less visual weight is zero     True
Visual-supported weights positive    True
All values finite                    True
dtype: bool

In [20]:
fusion_balance_summary = pd.Series({

    "Graph weight":
        float(
            visual_supported_weights[0]
        ),

    "Text weight":
        float(
            visual_supported_weights[1]
        ),

    "Visual weight":
        float(
            visual_supported_weights[2]
        ),

    "Maximum weight":
        float(
            visual_supported_weights.max()
        ),

    "Minimum weight":
        float(
            visual_supported_weights.min()
        ),

    "Maximum-minus-minimum":
        float(
            visual_supported_weights.max()
            -
            visual_supported_weights.min()
        ),

    "Graph-text difference":
        float(
            visual_supported_weights[0]
            -
            visual_supported_weights[1]
        ),

    "Graph-visual difference":
        float(
            visual_supported_weights[0]
            -
            visual_supported_weights[2]
        ),

    "Text-visual difference":
        float(
            visual_supported_weights[1]
            -
            visual_supported_weights[2]
        )
})


fusion_balance_summary

Graph weight               0.335256
Text weight                0.330196
Visual weight              0.334547
Maximum weight             0.335256
Minimum weight             0.330196
Maximum-minus-minimum      0.005060
Graph-text difference      0.005060
Graph-visual difference    0.000709
Text-visual difference    -0.004351
dtype: float64

In [21]:
fusion_masking_summary = pd.DataFrame({

    "modality": [
        "Graph",
        "Text",
        "Visual"
    ],

    "weight_with_visual":
        visual_supported_weights,

    "weight_without_visual":
        image_less_weights,

    "weight_change":
        (
            image_less_weights
            -
            visual_supported_weights
        )
})


fusion_masking_summary

,modality,weight_with_visual,weight_without_visual,weight_change
0,Graph,0.335256,0.503802,0.168546
1,Text,0.330196,0.496198,0.166002
2,Visual,0.334547,0.000000,-0.334547


In [22]:
def normalised_entropy(
    weights
):
    weights = np.asarray(
        weights,
        dtype=float
    )

    weights = weights[
        weights > 0
    ]

    entropy = -np.sum(
        weights
        *
        np.log(
            weights
        )
    )

    maximum_entropy = np.log(
        len(
            weights
        )
    )

    return (
        entropy
        /
        maximum_entropy
    )

In [23]:
fusion_entropy_summary = pd.Series({

    "Visual-supported normalised entropy":
        normalised_entropy(
            visual_supported_weights
        ),

    "Image-less graph-text normalised entropy":
        normalised_entropy(
            image_less_weights[:2]
        )
})


fusion_entropy_summary

Visual-supported normalised entropy         0.999979
Image-less graph-text normalised entropy    0.999958
dtype: float64

In [24]:
global_fusion_summary.to_csv(
    explanation_dir
    /
    "k2_global_fusion_summary.csv",

    index=False
)


fusion_masking_summary.to_csv(
    explanation_dir
    /
    "k2_fusion_masking_summary.csv",

    index=False
)


fusion_balance_summary.to_csv(
    explanation_dir
    /
    "k2_fusion_balance_summary.csv",
    header=[
        "value"
    ]
)


fusion_entropy_summary.to_csv(
    explanation_dir
    /
    "k2_fusion_entropy_summary.csv",
    header=[
        "value"
    ]
)

In [25]:
k2_save_check = pd.Series({

    "Global fusion saved":
        (
            explanation_dir
            /
            "k2_global_fusion_summary.csv"
        ).exists(),

    "Masking summary saved":
        (
            explanation_dir
            /
            "k2_fusion_masking_summary.csv"
        ).exists(),

    "Balance summary saved":
        (
            explanation_dir
            /
            "k2_fusion_balance_summary.csv"
        ).exists(),

    "Entropy summary saved":
        (
            explanation_dir
            /
            "k2_fusion_entropy_summary.csv"
        ).exists()
})


k2_save_check

Global fusion saved      True
Masking summary saved    True
Balance summary saved    True
Entropy summary saved    True
dtype: bool

## K3 — Frozen-MM Visual Perturbation Evidence

Visual perturbation analysis is used to examine how the frozen KGRec-MM model
responds when visual information is removed at inference time.

Three complementary forms of evidence are considered:

1. catalogue-wide visual ablation, comparing the original multimodal ranking
   with the same frozen model when visual information is disabled globally;

2. target-specific visual ablation, examining changes in target score and rank
   when visual information is removed only from a visually supported target;

3. competitive missing-modality behaviour, examining image-less targets when
   competing businesses retain or lose their visual information.

No model parameters are retrained or updated. These perturbations therefore
measure sensitivity of the trained model to modality availability rather than
the performance of a separately trained non-visual system.

Because removal of modalities used during training creates an out-of-
distribution representation, perturbation effects are interpreted as
model-behaviour evidence rather than causal estimates of modality importance.

In [26]:
# --------------------------------------------------
# K3.1 — Frozen-MM catalogue-wide visual ablation
# --------------------------------------------------

required_k3_columns = [
    "mm_rank",
    "mm_all_visual_off_rank",
    "mm_target_visual_off_rank",
    "target_has_visual",
    "target_visual_score_gain",
    "target_visual_rank_gain",
    "business_id"
]


k3_column_check = pd.Series({

    column:
        column in rq3_analysis.columns

    for column
    in required_k3_columns
})


k3_column_check

mm_rank                      True
mm_all_visual_off_rank       True
mm_target_visual_off_rank    True
target_has_visual            True
target_visual_score_gain     True
target_visual_rank_gain      True
business_id                  True
dtype: bool

In [27]:
def rank_metric_contributions(
    ranks,
    k
):

    ranks = np.asarray(
        ranks,
        dtype=float
    )

    hit = (
        ranks <= k
    )


    recall = (
        hit.astype(float)
    )


    ndcg = np.where(
        hit,
        1.0
        /
        np.log2(
            ranks + 1.0
        ),
        0.0
    )


    average_precision = np.where(
        hit,
        1.0
        /
        ranks,
        0.0
    )


    return {
        "Recall":
            recall,

        "NDCG":
            ndcg,

        "MAP":
            average_precision
    }

In [28]:
full_mm_ranks = (
    rq3_analysis[
        "mm_rank"
    ]
    .to_numpy()
)


all_visual_off_ranks = (
    rq3_analysis[
        "mm_all_visual_off_rank"
    ]
    .to_numpy()
)

In [29]:
catalogue_ablation_rows = []


for k in [
    5,
    10,
    20
]:

    full_contributions = (
        rank_metric_contributions(
            full_mm_ranks,
            k
        )
    )


    off_contributions = (
        rank_metric_contributions(
            all_visual_off_ranks,
            k
        )
    )


    for metric in [
        "Recall",
        "NDCG",
        "MAP"
    ]:

        full_values = (
            full_contributions[
                metric
            ]
        )


        off_values = (
            off_contributions[
                metric
            ]
        )


        catalogue_ablation_rows.append({

            "metric":
                f"{metric}@{k}",

            "full_mm":
                float(
                    full_values.mean()
                ),

            "all_visuals_off":
                float(
                    off_values.mean()
                ),

            "full_minus_all_off":
                float(
                    (
                        full_values
                        -
                        off_values
                    ).mean()
                )
        })


catalogue_visual_ablation_summary = pd.DataFrame(
    catalogue_ablation_rows
)


catalogue_visual_ablation_summary

,metric,full_mm,all_visuals_off,full_minus_all_off
0,Recall@5,0.041692,0.035821,0.005870
1,NDCG@5,0.026689,0.022576,0.004113
2,MAP@5,0.021785,0.018307,0.003479
3,Recall@10,0.067574,0.064639,0.002935
4,NDCG@10,0.034965,0.031701,0.003264
5,MAP@10,0.025146,0.021960,0.003186
6,Recall@20,0.111000,0.108999,0.002001
7,NDCG@20,0.045860,0.042840,0.003021
8,MAP@20,0.028093,0.024976,0.003117


In [30]:
def paired_bootstrap_mean_difference(
    differences,
    n_bootstrap=2000,
    seed=42
):

    differences = np.asarray(
        differences,
        dtype=float
    )


    observed = float(
        differences.mean()
    )


    n = len(
        differences
    )


    rng = np.random.default_rng(
        seed
    )


    bootstrap_means = np.empty(
        n_bootstrap,
        dtype=float
    )


    for bootstrap_index in range(
        n_bootstrap
    ):

        sampled_indices = rng.integers(
            0,
            n,
            size=n
        )


        bootstrap_means[
            bootstrap_index
        ] = (
            differences[
                sampled_indices
            ]
            .mean()
        )


    ci_lower, ci_upper = np.quantile(
        bootstrap_means,
        [
            0.025,
            0.975
        ]
    )


    return {
        "effect":
            observed,

        "CI_95_lower":
            float(
                ci_lower
            ),

        "CI_95_upper":
            float(
                ci_upper
            ),

        "CI_excludes_zero":
            bool(
                (
                    ci_lower > 0
                )
                or
                (
                    ci_upper < 0
                )
            )
    }

In [31]:
catalogue_ablation_bootstrap_rows = []


bootstrap_index = 0


for k in [
    5,
    10,
    20
]:

    full_contributions = (
        rank_metric_contributions(
            full_mm_ranks,
            k
        )
    )


    off_contributions = (
        rank_metric_contributions(
            all_visual_off_ranks,
            k
        )
    )


    for metric in [
        "Recall",
        "NDCG",
        "MAP"
    ]:

        differences = (
            full_contributions[
                metric
            ]
            -
            off_contributions[
                metric
            ]
        )


        result = (
            paired_bootstrap_mean_difference(
                differences=
                    differences,

                n_bootstrap=
                    2000,

                seed=
                    42
                    +
                    bootstrap_index * 100
            )
        )


        catalogue_ablation_bootstrap_rows.append({

            "metric":
                f"{metric}@{k}",

            "full_minus_all_off":
                result[
                    "effect"
                ],

            "CI_95_lower":
                result[
                    "CI_95_lower"
                ],

            "CI_95_upper":
                result[
                    "CI_95_upper"
                ],

            "CI_excludes_zero":
                result[
                    "CI_excludes_zero"
                ]
        })


        bootstrap_index += 1


catalogue_visual_ablation_bootstrap = (
    pd.DataFrame(
        catalogue_ablation_bootstrap_rows
    )
)


catalogue_visual_ablation_bootstrap

,metric,full_minus_all_off,CI_95_lower,CI_95_upper,CI_excludes_zero
0,Recall@5,0.005870,0.002668,0.009274,True
1,NDCG@5,0.004113,0.002308,0.006007,True
2,MAP@5,0.003479,0.002064,0.005100,True
3,Recall@10,0.002935,-0.000867,0.007273,False
4,NDCG@10,0.003264,0.001448,0.005158,True
5,MAP@10,0.003186,0.001796,0.004622,True
6,Recall@20,0.002001,-0.002937,0.007206,False
7,NDCG@20,0.003021,0.001142,0.004942,True
8,MAP@20,0.003117,0.001732,0.004608,True


In [32]:
visual_target_cases = (
    rq3_analysis.loc[
        rq3_analysis[
            "target_has_visual"
        ].astype(bool)
    ]
    .copy()
)


print(
    "Visual-supported test targets:",
    len(
        visual_target_cases
    )
)

Visual-supported test targets: 13266


In [33]:
target_visual_score_summary = pd.Series({

    "Cases":
        len(
            visual_target_cases
        ),

    "Mean score gain":
        visual_target_cases[
            "target_visual_score_gain"
        ].mean(),

    "Median score gain":
        visual_target_cases[
            "target_visual_score_gain"
        ].median(),

    "Minimum score gain":
        visual_target_cases[
            "target_visual_score_gain"
        ].min(),

    "Maximum score gain":
        visual_target_cases[
            "target_visual_score_gain"
        ].max(),

    "Visual increased target score":
        int(
            (
                visual_target_cases[
                    "target_visual_score_gain"
                ]
                >
                0
            ).sum()
        ),

    "Visual decreased target score":
        int(
            (
                visual_target_cases[
                    "target_visual_score_gain"
                ]
                <
                0
            ).sum()
        )
})


target_visual_score_summary

Cases                            13266.000000
Mean score gain                      0.723890
Median score gain                    0.775547
Minimum score gain                   0.058696
Maximum score gain                   1.100740
Visual increased target score    13266.000000
Visual decreased target score        0.000000
dtype: float64

In [34]:
target_visual_rank_summary = pd.Series({

    "Cases":
        len(
            visual_target_cases
        ),

    "Mean rank gain":
        visual_target_cases[
            "target_visual_rank_gain"
        ].mean(),

    "Median rank gain":
        visual_target_cases[
            "target_visual_rank_gain"
        ].median(),

    "Rank improved with target visual":
        int(
            (
                visual_target_cases[
                    "target_visual_rank_gain"
                ]
                >
                0
            ).sum()
        ),

    "Rank worsened with target visual":
        int(
            (
                visual_target_cases[
                    "target_visual_rank_gain"
                ]
                <
                0
            ).sum()
        ),

    "Rank unchanged":
        int(
            (
                visual_target_cases[
                    "target_visual_rank_gain"
                ]
                ==
                0
            ).sum()
        )
})


target_visual_rank_summary

Cases                               13266.000000
Mean rank gain                        678.173225
Median rank gain                      650.000000
Rank improved with target visual    13266.000000
Rank worsened with target visual        0.000000
Rank unchanged                          0.000000
dtype: float64

In [35]:
image_less_target_cases = (
    rq3_analysis.loc[
        ~rq3_analysis[
            "target_has_visual"
        ].astype(bool)
    ]
    .copy()
)


print(
    "Image-less target cases:",
    len(
        image_less_target_cases
    )
)

Image-less target cases: 1725


In [36]:
image_less_target_cases[
    "catalogue_visual_rank_effect"
] = (
    image_less_target_cases[
        "mm_all_visual_off_rank"
    ]
    -
    image_less_target_cases[
        "mm_rank"
    ]
)

In [37]:
image_less_competitive_summary = pd.Series({

    "Cases":
        len(
            image_less_target_cases
        ),

    "Mean catalogue visual rank effect":
        image_less_target_cases[
            "catalogue_visual_rank_effect"
        ].mean(),

    "Median catalogue visual rank effect":
        image_less_target_cases[
            "catalogue_visual_rank_effect"
        ].median(),

    "Image-less target improved":
        int(
            (
                image_less_target_cases[
                    "catalogue_visual_rank_effect"
                ]
                >
                0
            ).sum()
        ),

    "Image-less target worsened":
        int(
            (
                image_less_target_cases[
                    "catalogue_visual_rank_effect"
                ]
                <
                0
            ).sum()
        ),

    "Image-less target unchanged":
        int(
            (
                image_less_target_cases[
                    "catalogue_visual_rank_effect"
                ]
                ==
                0
            ).sum()
        )
})


image_less_competitive_summary

Cases                                  1725.000000
Mean catalogue visual rank effect      -580.342609
Median catalogue visual rank effect    -578.000000
Image-less target improved                0.000000
Image-less target worsened             1723.000000
Image-less target unchanged               2.000000
dtype: float64

In [38]:
k3_perturbation_synthesis = pd.DataFrame([
    {
        "evidence_type":
            "Catalogue-wide visual ablation",

        "comparison":
            "Full MM vs all visuals masked",

        "observed_pattern":
            (
                "Full multimodal inference improves top-K "
                "effectiveness relative to the same frozen "
                "model with visual evidence disabled."
            ),

        "interpretation":
            (
                "The visual pathway materially affects "
                "recommendation behaviour at catalogue level."
            ),

        "caveat":
            (
                "Global masking is an out-of-distribution "
                "inference perturbation and is not equivalent "
                "to a retrained non-visual model."
            )
    },

    {
        "evidence_type":
            "Target-specific visual ablation",

        "comparison":
            "Full target representation vs target visual masked",

        "observed_pattern":
            (
                "Removing visual evidence reduces target score "
                "and rank for visually supported targets."
            ),

        "interpretation":
            (
                "The frozen model is highly sensitive to visual "
                "information associated with the target business."
            ),

        "caveat":
            (
                "Masking changes both representation content and "
                "fusion renormalisation; the resulting magnitude "
                "must not be interpreted as causal attribution."
            )
    },

    {
        "evidence_type":
            "Competitive missing-modality effect",

        "comparison":
            (
                "Image-less target with visual competitors "
                "vs all visuals masked"
            ),

        "observed_pattern":
            (
                "Image-less targets generally rank lower when "
                "competing businesses retain visual evidence."
            ),

        "interpretation":
            (
                "Visual representations influence relative "
                "competition across the recommendation catalogue."
            ),

        "caveat":
            (
                "Image availability is associated with business "
                "interaction support and metadata richness."
            )
    }
])


k3_perturbation_synthesis

,evidence_type,comparison,observed_pattern,interpretation,caveat
0,Catalogue-wide visual ablation,Full MM vs all visuals masked,Full multimodal inference improves top-K effec...,The visual pathway materially affects recommen...,Global masking is an out-of-distribution infer...
1,Target-specific visual ablation,Full target representation vs target visual ma...,Removing visual evidence reduces target score ...,The frozen model is highly sensitive to visual...,Masking changes both representation content an...
2,Competitive missing-modality effect,Image-less target with visual competitors vs a...,Image-less targets generally rank lower when c...,Visual representations influence relative comp...,Image availability is associated with business...


In [39]:
catalogue_visual_ablation_summary.to_csv(
    explanation_dir
    /
    "k3_catalogue_visual_ablation_summary.csv",

    index=False
)


catalogue_visual_ablation_bootstrap.to_csv(
    explanation_dir
    /
    "k3_catalogue_visual_ablation_bootstrap.csv",

    index=False
)


target_visual_score_summary.to_csv(
    explanation_dir
    /
    "k3_target_visual_score_summary.csv",
    header=[
        "value"
    ]
)


target_visual_rank_summary.to_csv(
    explanation_dir
    /
    "k3_target_visual_rank_summary.csv",
    header=[
        "value"
    ]
)


image_less_competitive_summary.to_csv(
    explanation_dir
    /
    "k3_image_less_competitive_summary.csv",
    header=[
        "value"
    ]
)


k3_perturbation_synthesis.to_csv(
    explanation_dir
    /
    "k3_perturbation_synthesis.csv",

    index=False
)

In [40]:
k3_save_check = pd.Series({

    "Catalogue ablation saved":
        (
            explanation_dir
            /
            "k3_catalogue_visual_ablation_summary.csv"
        ).exists(),

    "Bootstrap results saved":
        (
            explanation_dir
            /
            "k3_catalogue_visual_ablation_bootstrap.csv"
        ).exists(),

    "Target score sensitivity saved":
        (
            explanation_dir
            /
            "k3_target_visual_score_summary.csv"
        ).exists(),

    "Target rank sensitivity saved":
        (
            explanation_dir
            /
            "k3_target_visual_rank_summary.csv"
        ).exists(),

    "Image-less competitive effect saved":
        (
            explanation_dir
            /
            "k3_image_less_competitive_summary.csv"
        ).exists(),

    "K3 synthesis saved":
        (
            explanation_dir
            /
            "k3_perturbation_synthesis.csv"
        ).exists()
})


k3_save_check

Catalogue ablation saved               True
Bootstrap results saved                True
Target score sensitivity saved         True
Target rank sensitivity saved          True
Image-less competitive effect saved    True
K3 synthesis saved                     True
dtype: bool

## K4 — Recommendation Evidence Extraction

The explanation layer now moves from global model behaviour to
human-interpretable evidence associated with individual recommendations.

Three evidence sources are considered:

1. structured knowledge-graph evidence, including business categories and
   descriptive attributes;

2. textual evidence derived exclusively from training-period reviews used by
   the recommender pipeline;

3. visual evidence describing the photographs available to the multimodal
   business representation.

These evidence sources support recommendation interpretation but are not
automatically treated as faithful causal explanations of individual model
predictions.

In [41]:
# --------------------------------------------------
# K4.1 — Evidence-source discovery
# --------------------------------------------------

import pyarrow.parquet as pq


def parquet_columns(
    path
):
    return set(
        pq.ParquetFile(
            path
        ).schema.names
    )


def find_parquets_with_columns(
    root,
    required_columns
):

    matches = []

    required_columns = set(
        required_columns
    )


    for path in root.rglob(
        "*.parquet"
    ):

        # Restrict to New Orleans artefacts
        if (
            "new_orleans"
            not in str(path).lower()
        ):
            continue


        try:

            columns = parquet_columns(
                path
            )


            if required_columns.issubset(
                columns
            ):

                matches.append(
                    path
                )

        except Exception:
            pass


    return sorted(
        matches
    )

In [42]:
business_source_candidates = (
    find_parquets_with_columns(
        processed_data_root,
        [
            "business_id",
            "name"
        ]
    )
)


print(
    "BUSINESS SOURCES"
)


for path in business_source_candidates:

    print(
        path.relative_to(
            processed_data_root
        )
    )

BUSINESS SOURCES
new_orleans_subset/_archive_before_final_preparation/new_orleans_businesses_5core.parquet
new_orleans_subset/_archive_before_final_preparation/new_orleans_businesses_5core_from_active_20260803_133113.parquet
new_orleans_subset/new_orleans_food_hospitality_businesses.parquet
new_orleans_subset/new_orleans_personalisation_businesses.parquet


In [43]:
review_source_candidates = (
    find_parquets_with_columns(
        processed_data_root,
        [
            "business_id",
            "text"
        ]
    )
)


print(
    "\nREVIEW-TEXT SOURCES"
)


for path in review_source_candidates:

    print(
        path.relative_to(
            processed_data_root
        )
    )


REVIEW-TEXT SOURCES
new_orleans_model_outputs/explanations/k4_training_review_evidence.parquet
new_orleans_subset/_archive_before_final_preparation/new_orleans_positive_reviews_5core.parquet
new_orleans_subset/_archive_before_final_preparation/new_orleans_positive_reviews_5core_from_active_20260803_133113.parquet
new_orleans_subset/_archive_before_final_preparation/new_orleans_reviews_5core.parquet
new_orleans_subset/_archive_before_final_preparation/new_orleans_reviews_5core_from_active_20260803_133113.parquet
new_orleans_subset/_archive_before_final_preparation/reviews_test.parquet
new_orleans_subset/_archive_before_final_preparation/reviews_train.parquet
new_orleans_subset/_archive_before_final_preparation/reviews_validation.parquet
new_orleans_subset/new_orleans_food_hospitality_reviews_raw.parquet


In [44]:
photo_source_candidates = (
    find_parquets_with_columns(
        processed_data_root,
        [
            "business_id",
            "label"
        ]
    )
)


print(
    "\nPHOTO SOURCES"
)


for path in photo_source_candidates:

    print(
        path.relative_to(
            processed_data_root
        )
    )


PHOTO SOURCES
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0000.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0001.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0002.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0003.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0004.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0005.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0006.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0007.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0008.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0009.parquet
new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0010.parquet
new_orleans_image_pipeline/c

In [45]:
metadata_kg_path = (
    processed_data_root
    /
    "new_orleans_knowledge_graph"
    /
    "new_orleans_frozen_metadata_kg.parquet"
)


print(
    "Metadata KG exists:",
    metadata_kg_path.exists()
)

Metadata KG exists: True


In [46]:
if metadata_kg_path.exists():

    print(
        parquet_columns(
            metadata_kg_path
        )
    )

{'relation', 'source', 'head', 'head_type', 'tail', 'tail_type'}


In [47]:
train_interaction_path = (
    processed_data_root
    /
    "new_orleans_subset"
    /
    "new_orleans_positive_train.parquet"
)


print(
    "Training interactions exist:",
    train_interaction_path.exists()
)

Training interactions exist: True


In [48]:
if train_interaction_path.exists():

    print(
        parquet_columns(
            train_interaction_path
        )
    )

{'interaction_sentiment', 'user_id', 'review_id', 'business_id', 'stars', 'date'}


In [49]:
def show_candidate_schemas(
    candidates,
    title
):

    print(
        "\n",
        title
    )


    for path in candidates:

        print(
            "\n",
            path.relative_to(
                processed_data_root
            )
        )

        print(
            sorted(
                parquet_columns(
                    path
                )
            )
        )

In [50]:
show_candidate_schemas(
    business_source_candidates,
    "BUSINESS CANDIDATE SCHEMAS"
)


 BUSINESS CANDIDATE SCHEMAS

 new_orleans_subset/_archive_before_final_preparation/new_orleans_businesses_5core.parquet
['AcceptsInsurance', 'AgesAllowed', 'Alcohol', 'Ambience', 'BYOB', 'BYOBCorkage', 'BestNights', 'BikeParking', 'BusinessAcceptsBitcoin', 'BusinessAcceptsCreditCards', 'BusinessParking', 'ByAppointmentOnly', 'Caters', 'CoatCheck', 'Corkage', 'DietaryRestrictions', 'DogsAllowed', 'DriveThru', 'Friday', 'GoodForDancing', 'GoodForKids', 'GoodForMeal', 'HappyHour', 'HasTV', 'Monday', 'Music', 'NoiseLevel', 'Open24Hours', 'OutdoorSeating', 'RestaurantsAttire', 'RestaurantsCounterService', 'RestaurantsDelivery', 'RestaurantsGoodForGroups', 'RestaurantsPriceRange2', 'RestaurantsReservations', 'RestaurantsTableService', 'RestaurantsTakeOut', 'Saturday', 'Smoking', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday', 'WheelchairAccessible', 'WiFi', 'address', 'business_id', 'categories', 'city', 'city_clean', 'element', 'is_open', 'is_operational_at_snapshot', 'is_target_business', '

In [51]:
show_candidate_schemas(
    review_source_candidates,
    "REVIEW CANDIDATE SCHEMAS"
)


 REVIEW CANDIDATE SCHEMAS

 new_orleans_model_outputs/explanations/k4_training_review_evidence.parquet
['business_id', 'cool', 'date', 'funny', 'review_id', 'stars', 'text', 'useful', 'user_id']

 new_orleans_subset/_archive_before_final_preparation/new_orleans_positive_reviews_5core.parquet
['business_id', 'cool', 'date', 'funny', 'interaction_sentiment', 'is_negative', 'is_neutral', 'is_positive', 'review_id', 'stars', 'text', 'useful', 'user_id']

 new_orleans_subset/_archive_before_final_preparation/new_orleans_positive_reviews_5core_from_active_20260803_133113.parquet
['business_id', 'cool', 'date', 'funny', 'interaction_sentiment', 'is_negative', 'is_neutral', 'is_positive', 'review_id', 'stars', 'text', 'useful', 'user_id']

 new_orleans_subset/_archive_before_final_preparation/new_orleans_reviews_5core.parquet
['business_id', 'cool', 'date', 'funny', 'interaction_sentiment', 'is_negative', 'is_neutral', 'is_positive', 'review_id', 'stars', 'text', 'useful', 'user_id']

 new_or

In [52]:
show_candidate_schemas(
    photo_source_candidates,
    "PHOTO CANDIDATE SCHEMAS"
)


 PHOTO CANDIDATE SCHEMAS

 new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0000.parquet
['business_id', 'in_personalisation_core', 'label', 'manifest_position', 'photo_filename', 'photo_id', 'selection_rank']

 new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0001.parquet
['business_id', 'in_personalisation_core', 'label', 'manifest_position', 'photo_filename', 'photo_id', 'selection_rank']

 new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0002.parquet
['business_id', 'in_personalisation_core', 'label', 'manifest_position', 'photo_filename', 'photo_id', 'selection_rank']

 new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0003.parquet
['business_id', 'in_personalisation_core', 'label', 'manifest_position', 'photo_filename', 'photo_id', 'selection_rank']

 new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints/checkpoint_0004.parquet
['business_id', 'in_personalisation_core', 'l

### K4.2 — Authoritative Evidence Sources

Human-readable explanation evidence is reconstructed from the same data
sources underlying the frozen recommendation experiment.

Business identity is obtained from the current personalisation catalogue.
Structured evidence is taken from the frozen metadata knowledge graph.

Textual evidence is restricted to reviews belonging to the frozen training
interaction split. Review text is recovered by joining training review IDs
to the current raw New Orleans review table, preventing validation or test
reviews from entering the explanation evidence.

Visual evidence is taken from the final image-embedding manifest rather than
raw or intermediate photo inventories. Consequently, the visual evidence
shown by the explanation layer corresponds to photographs actually selected
for the CLIP business representation.

Archived and checkpoint artefacts are excluded from the explanation source
set.

In [53]:
# --------------------------------------------------
# K4.2 — Authoritative explanation sources
# --------------------------------------------------

business_catalogue_path = (
    processed_data_root
    /
    "new_orleans_subset"
    /
    "new_orleans_personalisation_businesses.parquet"
)


training_interaction_path = (
    processed_data_root
    /
    "new_orleans_subset"
    /
    "new_orleans_positive_train.parquet"
)


raw_review_path = (
    processed_data_root
    /
    "new_orleans_subset"
    /
    "new_orleans_food_hospitality_reviews_raw.parquet"
)


photo_manifest_path = (
    processed_data_root
    /
    "new_orleans_image_pipeline"
    /
    "new_orleans_image_embedding_manifest.parquet"
)


metadata_kg_path = (
    processed_data_root
    /
    "new_orleans_knowledge_graph"
    /
    "new_orleans_frozen_metadata_kg.parquet"
)

In [54]:
k4_source_path_check = pd.Series({

    "Business catalogue":
        business_catalogue_path.exists(),

    "Training interactions":
        training_interaction_path.exists(),

    "Raw review text":
        raw_review_path.exists(),

    "Image embedding manifest":
        photo_manifest_path.exists(),

    "Frozen metadata KG":
        metadata_kg_path.exists()
})


k4_source_path_check

Business catalogue          True
Training interactions       True
Raw review text             True
Image embedding manifest    True
Frozen metadata KG          True
dtype: bool

In [55]:
business_catalogue = pd.read_parquet(
    business_catalogue_path
)

In [56]:
business_display = (
    business_catalogue[
        [
            "business_id",
            "name",
            "categories",
            "address",
            "city",
            "state"
        ]
    ]
    .copy()
)

In [57]:
business_display_check = pd.Series({

    "Rows":
        len(
            business_display
        ),

    "Unique business IDs":
        business_display[
            "business_id"
        ].nunique(),

    "Expected 2516 businesses":
        (
            len(
                business_display
            )
            ==
            2516
        ),

    "No duplicate business IDs":
        (
            business_display[
                "business_id"
            ].is_unique
        ),

    "Missing names":
        int(
            business_display[
                "name"
            ]
            .isna()
            .sum()
        )
})


business_display_check

Rows                         2516
Unique business IDs          2516
Expected 2516 businesses     True
No duplicate business IDs    True
Missing names                   0
dtype: object

In [58]:
training_interactions = pd.read_parquet(
    training_interaction_path
)


raw_reviews = pd.read_parquet(
    raw_review_path,
    columns=[
        "review_id",
        "business_id",
        "user_id",
        "date",
        "stars",
        "text",
        "useful",
        "funny",
        "cool"
    ]
)

In [59]:
training_review_id_check = pd.Series({

    "Training interactions":
        len(
            training_interactions
        ),

    "Unique training review IDs":
        training_interactions[
            "review_id"
        ].nunique(),

    "Training review IDs unique":
        training_interactions[
            "review_id"
        ].is_unique,

    "Raw review IDs unique":
        raw_reviews[
            "review_id"
        ].is_unique
})


training_review_id_check

Training interactions         122233
Unique training review IDs    122233
Training review IDs unique      True
Raw review IDs unique           True
dtype: object

In [60]:
training_review_evidence = (
    training_interactions[
        [
            "review_id",
            "business_id",
            "user_id"
        ]
    ]
    .merge(
        raw_reviews[
            [
                "review_id",
                "business_id",
                "user_id",
                "date",
                "stars",
                "text",
                "useful",
                "funny",
                "cool"
            ]
        ],

        on=[
            "review_id",
            "business_id",
            "user_id"
        ],

        how="left",

        validate="one_to_one"
    )
)

In [61]:
training_review_evidence_check = pd.Series({

    "Rows":
        len(
            training_review_evidence
        ),

    "Expected 122233":
        (
            len(
                training_review_evidence
            )
            ==
            122233
        ),

    "Unique review IDs":
        training_review_evidence[
            "review_id"
        ].nunique(),

    "All training reviews matched text":
        (
            training_review_evidence[
                "text"
            ]
            .notna()
            .all()
        ),

    "Businesses represented":
        training_review_evidence[
            "business_id"
        ].nunique(),

    "Users represented":
        training_review_evidence[
            "user_id"
        ].nunique(),

    "No duplicated review IDs":
        training_review_evidence[
            "review_id"
        ].is_unique
})


training_review_evidence_check

Rows                                 122233
Expected 122233                        True
Unique review IDs                    122233
All training reviews matched text      True
Businesses represented                 2516
Users represented                     14991
No duplicated review IDs               True
dtype: object

In [62]:
photo_manifest = pd.read_parquet(
    photo_manifest_path
)

In [63]:
visual_evidence = (
    photo_manifest.loc[
        (
            photo_manifest[
                "selected_for_embedding"
            ].astype(bool)
        )
        &
        (
            photo_manifest[
                "in_personalisation_core"
            ].astype(bool)
        )
    ]
    [
        [
            "business_id",
            "photo_id",
            "photo_filename",
            "label",
            "caption_clean",
            "selection_rank",
            "selection_strategy"
        ]
    ]
    .copy()
)

In [64]:
visual_evidence_check = pd.Series({

    "Selected image rows":
        len(
            visual_evidence
        ),

    "Expected 6122":
        (
            len(
                visual_evidence
            )
            ==
            6122
        ),

    "Visual businesses":
        visual_evidence[
            "business_id"
        ].nunique(),

    "Expected 1729 businesses":
        (
            visual_evidence[
                "business_id"
            ].nunique()
            ==
            1729
        ),

    "Maximum images per business":
        visual_evidence
        .groupby(
            "business_id"
        )
        .size()
        .max(),

    "Unique photo IDs":
        visual_evidence[
            "photo_id"
        ].is_unique,

    "Missing labels":
        int(
            visual_evidence[
                "label"
            ]
            .isna()
            .sum()
        )
})


visual_evidence_check

Selected image rows            6122
Expected 6122                  True
Visual businesses              1729
Expected 1729 businesses       True
Maximum images per business       5
Unique photo IDs               True
Missing labels                    0
dtype: object

In [65]:
metadata_evidence = pd.read_parquet(
    metadata_kg_path
)

In [66]:
metadata_evidence_check = pd.Series({

    "Metadata triples":
        len(
            metadata_evidence
        ),

    "Expected 83989":
        (
            len(
                metadata_evidence
            )
            ==
            83989
        ),

    "Business heads":
        metadata_evidence[
            "head"
        ].nunique(),

    "Expected 2516 businesses":
        (
            metadata_evidence[
                "head"
            ].nunique()
            ==
            2516
        ),

    "Relation types":
        metadata_evidence[
            "relation"
        ].nunique(),

    "Missing heads":
        int(
            metadata_evidence[
                "head"
            ]
            .isna()
            .sum()
        ),

    "Missing tails":
        int(
            metadata_evidence[
                "tail"
            ]
            .isna()
            .sum()
        )
})


metadata_evidence_check

Metadata triples            83989
Expected 83989               True
Business heads               2516
Expected 2516 businesses     True
Relation types                 63
Missing heads                   0
Missing tails                   0
dtype: object

In [67]:
metadata_business_ids = (
    metadata_evidence[
        "head"
    ]
    .str.replace(
        "business::",
        "",
        regex=False
    )
    .unique()
)

In [68]:
model_business_ids = set(
    business_index[
        "business_id"
    ].astype(str)
)


display_business_ids = set(
    business_display[
        "business_id"
    ].astype(str)
)


metadata_business_id_set = set(
    pd.Series(
        metadata_business_ids
    ).astype(str)
)


review_business_ids = set(
    training_review_evidence[
        "business_id"
    ].astype(str)
)


visual_business_ids = set(
    visual_evidence[
        "business_id"
    ].astype(str)
)

In [69]:
k4_alignment_check = pd.Series({

    "Display exactly matches model businesses":
        (
            display_business_ids
            ==
            model_business_ids
        ),

    "Metadata exactly matches model businesses":
        (
            metadata_business_id_set
            ==
            model_business_ids
        ),

    "Training reviews exactly cover model businesses":
        (
            review_business_ids
            ==
            model_business_ids
        ),

    "Visual businesses subset of model catalogue":
        visual_business_ids.issubset(
            model_business_ids
        ),

    "Visual business count":
        len(
            visual_business_ids
        ),

    "Image-less model businesses":
        len(
            model_business_ids
            -
            visual_business_ids
        )
})


k4_alignment_check

Display exactly matches model businesses           True
Metadata exactly matches model businesses          True
Training reviews exactly cover model businesses    True
Visual businesses subset of model catalogue        True
Visual business count                              1729
Image-less model businesses                         787
dtype: object

In [70]:
k4_evidence_registry = pd.DataFrame([
    {
        "evidence_type":
            "Business identity",

        "source":
            str(
                business_catalogue_path.relative_to(
                    processed_data_root
                )
            ),

        "scope":
            "2516 personalisation businesses",

        "model_relationship":
            "Display/context only"
    },

    {
        "evidence_type":
            "Structured KG",

        "source":
            str(
                metadata_kg_path.relative_to(
                    processed_data_root
                )
            ),

        "scope":
            "Frozen metadata KG",

        "model_relationship":
            "Direct structured model evidence"
    },

    {
        "evidence_type":
            "Review text",

        "source":
            (
                "Frozen positive_train review IDs joined to "
                "new_orleans_food_hospitality_reviews_raw.parquet"
            ),

        "scope":
            "122233 training interactions only",

        "model_relationship":
            "Same training-review evidence used to construct BGE business text"
    },

    {
        "evidence_type":
            "Visual evidence",

        "source":
            str(
                photo_manifest_path.relative_to(
                    processed_data_root
                )
            ),

        "scope":
            "6122 selected photos across 1729 model businesses",

        "model_relationship":
            "Exact photos selected for CLIP business representation"
    }
])


k4_evidence_registry

,evidence_type,source,scope,model_relationship
0,Business identity,new_orleans_subset/new_orleans_personalisation...,2516 personalisation businesses,Display/context only
1,Structured KG,new_orleans_knowledge_graph/new_orleans_frozen...,Frozen metadata KG,Direct structured model evidence
2,Review text,Frozen positive_train review IDs joined to new...,122233 training interactions only,Same training-review evidence used to construc...
3,Visual evidence,new_orleans_image_pipeline/new_orleans_image_e...,6122 selected photos across 1729 model businesses,Exact photos selected for CLIP business repres...


In [71]:
training_review_evidence.to_parquet(
    explanation_dir
    /
    "k4_training_review_evidence.parquet",

    index=False
)


visual_evidence.to_parquet(
    explanation_dir
    /
    "k4_visual_evidence.parquet",

    index=False
)


k4_evidence_registry.to_csv(
    explanation_dir
    /
    "k4_evidence_source_registry.csv",

    index=False
)

In [72]:
k42_save_check = pd.Series({

    "Training review evidence saved":
        (
            explanation_dir
            /
            "k4_training_review_evidence.parquet"
        ).exists(),

    "Visual evidence saved":
        (
            explanation_dir
            /
            "k4_visual_evidence.parquet"
        ).exists(),

    "Evidence registry saved":
        (
            explanation_dir
            /
            "k4_evidence_source_registry.csv"
        ).exists()
})


k42_save_check

Training review evidence saved    True
Visual evidence saved             True
Evidence registry saved           True
dtype: bool

### K4.3 — Representative Text Evidence

Textual explanation evidence is selected from the same training-period reviews
used to construct each business's BGE representation.

The business text vector was created by mean-pooling its training-review
embeddings and L2-normalising the resulting representation. To obtain
human-readable evidence consistent with that representation, individual
training reviews are ranked according to cosine similarity with the pooled
business text embedding.

Highly ranked reviews are therefore interpreted as representative examples
of the textual evidence encoded for a business. Similarity to the pooled
representation does not establish that an individual review causally
determined a recommendation.

In [73]:
# --------------------------------------------------
# K4.3 — Locate frozen BGE text artefacts
# --------------------------------------------------

review_embedding_candidates = list(
    processed_data_root.rglob(
        "new_orleans_bge_training_review_embeddings.npy"
    )
)


review_index_candidates = list(
    processed_data_root.rglob(
        "new_orleans_bge_training_review_embedding_index.parquet"
    )
)


business_text_embedding_candidates = list(
    processed_data_root.rglob(
        "new_orleans_bge_training_business_embeddings.npy"
    )
)


business_text_index_candidates = list(
    processed_data_root.rglob(
        "new_orleans_bge_training_business_embedding_index.parquet"
    )
)


print(
    "Review embeddings:",
    review_embedding_candidates
)

print(
    "Review index:",
    review_index_candidates
)

print(
    "Business text embeddings:",
    business_text_embedding_candidates
)

print(
    "Business text index:",
    business_text_index_candidates
)

Review embeddings: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_review_embeddings.npy')]
Review index: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_review_embedding_index.parquet')]
Business text embeddings: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_business_embeddings.npy')]
Business text index: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_business_embedding_index.parquet')]


In [74]:
k43_path_check = pd.Series({

    "One review embedding file":
        len(
            review_embedding_candidates
        ) == 1,

    "One review index":
        len(
            review_index_candidates
        ) == 1,

    "One business embedding file":
        len(
            business_text_embedding_candidates
        ) == 1,

    "One business text index":
        len(
            business_text_index_candidates
        ) == 1
})


k43_path_check

One review embedding file      True
One review index               True
One business embedding file    True
One business text index        True
dtype: bool

In [75]:
review_embedding_path = (
    review_embedding_candidates[0]
)

review_embedding_index_path = (
    review_index_candidates[0]
)

business_text_embedding_path = (
    business_text_embedding_candidates[0]
)

business_text_index_path = (
    business_text_index_candidates[0]
)

In [76]:
review_embeddings = np.load(
    review_embedding_path,
    mmap_mode="r"
)


review_embedding_index = pd.read_parquet(
    review_embedding_index_path
)


business_text_embeddings = np.load(
    business_text_embedding_path,
    mmap_mode="r"
)


business_text_embedding_index = pd.read_parquet(
    business_text_index_path
)

In [77]:
print(
    "Review embeddings shape:",
    review_embeddings.shape
)

print(
    "Review index columns:",
    review_embedding_index.columns.tolist()
)

print(
    "Business embeddings shape:",
    business_text_embeddings.shape
)

print(
    "Business index columns:",
    business_text_embedding_index.columns.tolist()
)

Review embeddings shape: (122233, 384)
Review index columns: ['review_id', 'user_id', 'business_id', 'date', 'chunk_count', 'original_token_count', 'review_embedding_row']
Business embeddings shape: (2516, 384)
Business index columns: ['business_id', 'visual_embedding_row', 'business_embedding_row', 'text_embedding_row', 'training_review_count', 'text_model', 'aggregation_method', 'training_reviewer_count', 'total_words', 'mean_review_words', 'median_review_words', 'maximum_review_words']


In [78]:
k43_review_alignment_check = pd.Series({

    "Review vectors":
        review_embeddings.shape[0],

    "Review dimension":
        review_embeddings.shape[1],

    "Review index rows":
        len(
            review_embedding_index
        ),

    "Expected reviews":
        (
            review_embeddings.shape[0]
            ==
            122233
        ),

    "Expected dimension":
        (
            review_embeddings.shape[1]
            ==
            384
        ),

    "Index aligns with vectors":
        (
            len(
                review_embedding_index
            )
            ==
            review_embeddings.shape[0]
        ),

    "Review IDs unique":
        review_embedding_index[
            "review_id"
        ].is_unique
})


k43_review_alignment_check

Review vectors               122233
Review dimension                384
Review index rows            122233
Expected reviews               True
Expected dimension             True
Index aligns with vectors      True
Review IDs unique              True
dtype: object

In [79]:
k43_business_alignment_check = pd.Series({

    "Business vectors":
        business_text_embeddings.shape[0],

    "Business dimension":
        business_text_embeddings.shape[1],

    "Business index rows":
        len(
            business_text_embedding_index
        ),

    "Expected businesses":
        (
            business_text_embeddings.shape[0]
            ==
            2516
        ),

    "Expected dimension":
        (
            business_text_embeddings.shape[1]
            ==
            384
        ),

    "Index aligns with vectors":
        (
            len(
                business_text_embedding_index
            )
            ==
            business_text_embeddings.shape[0]
        ),

    "Business IDs unique":
        business_text_embedding_index[
            "business_id"
        ].is_unique
})


k43_business_alignment_check

Business vectors             2516
Business dimension            384
Business index rows          2516
Expected businesses          True
Expected dimension           True
Index aligns with vectors    True
Business IDs unique          True
dtype: object

In [80]:
review_embedding_index = (
    review_embedding_index
    .reset_index(
        drop=True
    )
    .copy()
)


review_embedding_index[
    "review_embedding_row"
] = np.arange(
    len(
        review_embedding_index
    )
)


business_text_embedding_index = (
    business_text_embedding_index
    .reset_index(
        drop=True
    )
    .copy()
)


business_text_embedding_index[
    "business_text_embedding_row"
] = np.arange(
    len(
        business_text_embedding_index
    )
)

In [81]:
k43_evidence_alignment_check = pd.Series({

    "Review IDs exactly match training evidence":
        (
            set(
                review_embedding_index[
                    "review_id"
                ].astype(str)
            )
            ==
            set(
                training_review_evidence[
                    "review_id"
                ].astype(str)
            )
        ),

    "Business text IDs exactly match model catalogue":
        (
            set(
                business_text_embedding_index[
                    "business_id"
                ].astype(str)
            )
            ==
            model_business_ids
        )
})


k43_evidence_alignment_check

Review IDs exactly match training evidence         True
Business text IDs exactly match model catalogue    True
dtype: bool

In [82]:
review_text_lookup = (
    review_embedding_index[
        [
            "review_id",
            "business_id",
            "review_embedding_row"
        ]
    ]
    .merge(
        training_review_evidence[
            [
                "review_id",
                "business_id",
                "user_id",
                "date",
                "stars",
                "text",
                "useful",
                "funny",
                "cool"
            ]
        ],

        on=[
            "review_id",
            "business_id"
        ],

        how="left",

        validate="one_to_one"
    )
)

In [83]:
review_text_lookup_check = pd.Series({

    "Rows":
        len(
            review_text_lookup
        ),

    "Expected 122233":
        (
            len(
                review_text_lookup
            )
            ==
            122233
        ),

    "All text recovered":
        review_text_lookup[
            "text"
        ].notna().all(),

    "All embedding rows unique":
        review_text_lookup[
            "review_embedding_row"
        ].is_unique,

    "No review duplication":
        review_text_lookup[
            "review_id"
        ].is_unique
})


review_text_lookup_check

Rows                         122233
Expected 122233                True
All text recovered             True
All embedding rows unique      True
No review duplication          True
dtype: object

In [84]:
business_text_vector_lookup = (
    business_text_embedding_index[
        [
            "business_id",
            "business_text_embedding_row"
        ]
    ]
    .copy()
)

In [85]:
business_text_row_by_id = dict(
    zip(
        business_text_vector_lookup[
            "business_id"
        ].astype(str),

        business_text_vector_lookup[
            "business_text_embedding_row"
        ]
    )
)

In [86]:
def get_representative_text_evidence(
    business_id,
    top_n=3,
    max_chars=500
):

    business_id = str(
        business_id
    )


    if business_id not in business_text_row_by_id:

        raise KeyError(
            f"Business ID not found: {business_id}"
        )


    business_row = (
        business_text_row_by_id[
            business_id
        ]
    )


    business_vector = np.asarray(
        business_text_embeddings[
            business_row
        ],
        dtype=np.float32
    )


    business_reviews = (
        review_text_lookup.loc[
            review_text_lookup[
                "business_id"
            ].astype(str)
            ==
            business_id
        ]
        .copy()
    )


    if len(
        business_reviews
    ) == 0:

        return pd.DataFrame()


    review_rows = (
        business_reviews[
            "review_embedding_row"
        ]
        .to_numpy(
            dtype=int
        )
    )


    review_vectors = np.asarray(
        review_embeddings[
            review_rows
        ],
        dtype=np.float32
    )


    cosine_similarity = (
        review_vectors
        @
        business_vector
    )


    business_reviews[
        "text_representativeness"
    ] = cosine_similarity


    business_reviews = (
        business_reviews
        .sort_values(
            [
                "text_representativeness",
                "review_id"
            ],
            ascending=[
                False,
                True
            ]
        )
        .head(
            top_n
        )
        .copy()
    )


    business_reviews[
        "text_excerpt"
    ] = (
        business_reviews[
            "text"
        ]
        .astype(str)
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
        .str.slice(
            0,
            max_chars
        )
    )


    return (
        business_reviews[
            [
                "review_id",
                "date",
                "stars",
                "useful",
                "text_representativeness",
                "text_excerpt"
            ]
        ]
        .reset_index(
            drop=True
        )
    )

In [87]:
text_evidence_smoke_business_id = str(
    rq3_analysis.loc[
        rq3_analysis[
            "target_has_visual"
        ].astype(bool),
        "business_id"
    ].iloc[0]
)


text_evidence_smoke_business_id

'qb28j-FNX1_6xm7u372TZA'

In [88]:
business_display.loc[
    business_display[
        "business_id"
    ].astype(str)
    ==
    text_evidence_smoke_business_id,
    [
        "business_id",
        "name",
        "categories"
    ]
]

,business_id,name,categories
2144,qb28j-FNX1_6xm7u372TZA,Gumbo Shop,"Cajun/Creole, Seafood, Restaurants"


In [89]:
text_evidence_smoke = (
    get_representative_text_evidence(
        business_id=
            text_evidence_smoke_business_id,

        top_n=
            3
    )
)


text_evidence_smoke

,review_id,date,stars,useful,text_representativeness,text_excerpt
0,XzHPTS118jsxLLjzH_yo0w,2016-05-06 01:39:17,4,1,0.960387,"While looking for a good spot to grab dinner, ..."
1,dllYajPmSWO4J41aMi2WGQ,2019-08-13 13:36:34,5,1,0.958656,This was the first place we went to when we go...
2,5U4MbIhj0ubbafU2aniM7A,2018-01-22 20:52:27,5,6,0.957953,Take me to the Gumbo Shop! Not only is the foo...


In [90]:
text_evidence_smoke_check = pd.Series({

    "Returned three reviews":
        (
            len(
                text_evidence_smoke
            )
            ==
            min(
                3,
                len(
                    review_text_lookup.loc[
                        review_text_lookup[
                            "business_id"
                        ].astype(str)
                        ==
                        text_evidence_smoke_business_id
                    ]
                )
            )
        ),

    "Similarity finite":
        np.isfinite(
            text_evidence_smoke[
                "text_representativeness"
            ]
        ).all(),

    "Similarity <= 1":
        (
            text_evidence_smoke[
                "text_representativeness"
            ]
            <=
            1.00001
        ).all(),

    "Similarity >= -1":
        (
            text_evidence_smoke[
                "text_representativeness"
            ]
            >=
            -1.00001
        ).all(),

    "Sorted descending":
        text_evidence_smoke[
            "text_representativeness"
        ].is_monotonic_decreasing
})


text_evidence_smoke_check

Returned three reviews    True
Similarity finite         True
Similarity <= 1           True
Similarity >= -1          True
Sorted descending         True
dtype: bool

In [91]:
review_representativeness = np.empty(
    len(
        review_text_lookup
    ),
    dtype=np.float32
)

In [92]:
for business_id, group in review_text_lookup.groupby(
    "business_id",
    sort=False
):

    business_id = str(
        business_id
    )


    business_row = (
        business_text_row_by_id[
            business_id
        ]
    )


    business_vector = np.asarray(
        business_text_embeddings[
            business_row
        ],
        dtype=np.float32
    )


    review_rows = (
        group[
            "review_embedding_row"
        ]
        .to_numpy(
            dtype=int
        )
    )


    review_vectors = np.asarray(
        review_embeddings[
            review_rows
        ],
        dtype=np.float32
    )


    similarities = (
        review_vectors
        @
        business_vector
    )


    review_representativeness[
        group.index.to_numpy()
    ] = similarities

In [93]:
review_text_lookup[
    "text_representativeness"
] = review_representativeness

In [94]:
representativeness_check = pd.Series({

    "Rows":
        len(
            review_text_lookup
        ),

    "All finite":
        np.isfinite(
            review_text_lookup[
                "text_representativeness"
            ]
        ).all(),

    "Maximum <= 1":
        (
            review_text_lookup[
                "text_representativeness"
            ].max()
            <=
            1.00001
        ),

    "Minimum >= -1":
        (
            review_text_lookup[
                "text_representativeness"
            ].min()
            >=
            -1.00001
        )
})


representativeness_check

Rows             122233
All finite         True
Maximum <= 1       True
Minimum >= -1      True
dtype: object

In [95]:
business_text_evidence_strength = (
    review_text_lookup
    .groupby(
        "business_id",
        as_index=False
    )
    .agg(

        training_review_count=(
            "review_id",
            "size"
        ),

        top_review_similarity=(
            "text_representativeness",
            "max"
        ),

        median_review_similarity=(
            "text_representativeness",
            "median"
        )
    )
)

In [96]:
business_text_evidence_strength[
    [
        "training_review_count",
        "top_review_similarity",
        "median_review_similarity"
    ]
].describe()

,training_review_count,top_review_similarity,median_review_similarity
count,2516.000000,2516.000000,2516.000000
mean,48.582273,0.938715,0.900695
std,88.810075,0.012203,0.016271
min,1.000000,0.878625,0.839092
25%,8.000000,0.931862,0.890854
50%,19.000000,0.940074,0.898692
75%,51.000000,0.946857,0.909244
max,1271.000000,1.000000,1.000000


In [97]:
review_text_lookup.to_parquet(
    explanation_dir
    /
    "k4_review_text_representativeness.parquet",

    index=False
)


business_text_evidence_strength.to_parquet(
    explanation_dir
    /
    "k4_business_text_evidence_strength.parquet",

    index=False
)

In [98]:
k43_save_check = pd.Series({

    "Review representativeness saved":
        (
            explanation_dir
            /
            "k4_review_text_representativeness.parquet"
        ).exists(),

    "Business text evidence strength saved":
        (
            explanation_dir
            /
            "k4_business_text_evidence_strength.parquet"
        ).exists()
})


k43_save_check

Review representativeness saved          True
Business text evidence strength saved    True
dtype: bool

### K4.4 — Structured Knowledge-Graph Evidence

Structured explanation evidence is extracted directly from the frozen metadata
knowledge graph used by the recommender.

Each business is associated with category, scalar-attribute and nested-attribute
triples. Raw graph triples are retained for provenance while relation and tail
identifiers are converted into human-readable labels for explanation output.

Only evidence present in the frozen training knowledge graph is exposed.
Snapshot variables deliberately excluded from the model, such as aggregate
business rating, review count and operational status, are therefore not
introduced into the explanation layer.

Structured KG evidence describes information available to the model but does
not by itself establish the causal importance of an individual attribute to a
particular recommendation.

In [99]:
# --------------------------------------------------
# K4.4 — Structured KG evidence
# --------------------------------------------------

relation_family_summary = pd.DataFrame({

    "relation_family": np.select(
        [
            metadata_evidence[
                "relation"
            ].eq(
                "has_category"
            ),

            metadata_evidence[
                "relation"
            ].str.startswith(
                "attribute::",
                na=False
            ),

            metadata_evidence[
                "relation"
            ].str.startswith(
                "nested_attribute::",
                na=False
            )
        ],

        [
            "category",
            "scalar_attribute",
            "nested_attribute"
        ],

        default="other"
    )
})

In [100]:
relation_family_summary[
    "relation_family"
].value_counts()

relation_family
nested_attribute    38296
scalar_attribute    33490
category            12203
Name: count, dtype: int64

In [101]:
relation_family_counts = (
    relation_family_summary[
        "relation_family"
    ]
    .value_counts()
)


k44_relation_family_check = pd.Series({

    "Total triples":
        int(
            relation_family_counts.sum()
        ),

    "Expected 83989":
        (
            relation_family_counts.sum()
            ==
            83989
        ),

    "No unrecognised relation family":
        (
            relation_family_counts.get(
                "other",
                0
            )
            ==
            0
        ),

    "Category triples present":
        (
            relation_family_counts.get(
                "category",
                0
            )
            >
            0
        ),

    "Scalar triples present":
        (
            relation_family_counts.get(
                "scalar_attribute",
                0
            )
            >
            0
        ),

    "Nested triples present":
        (
            relation_family_counts.get(
                "nested_attribute",
                0
            )
            >
            0
        )
})


k44_relation_family_check

Total triples                      83989
Expected 83989                      True
No unrecognised relation family     True
Category triples present            True
Scalar triples present              True
Nested triples present              True
dtype: object

In [102]:
for relation_family in [
    "category",
    "scalar_attribute",
    "nested_attribute"
]:

    mask = (
        relation_family_summary[
            "relation_family"
        ]
        ==
        relation_family
    )


    print(
        "\n",
        relation_family.upper()
    )


    display(
        metadata_evidence.loc[
            mask.values,
            [
                "head",
                "relation",
                "tail",
                "source"
            ]
        ]
        .head(
            5
        )
    )


 CATEGORY


,head,relation,tail,source
0,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Cafes,category
1,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Nightlife,category
2,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Cocktail Bars,category
3,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Peruvian,category
4,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Restaurants,category



 SCALAR_ATTRIBUTE


,head,relation,tail,source
12203,business::-0__F9fnKt8uioCKztF5Ww,attribute::RestaurantsAttire,value::RestaurantsAttire::casual,scalar_attribute
12204,business::-0__F9fnKt8uioCKztF5Ww,attribute::WheelchairAccessible,value::WheelchairAccessible::true,scalar_attribute
12205,business::-0__F9fnKt8uioCKztF5Ww,attribute::BikeParking,value::BikeParking::true,scalar_attribute
12206,business::-0__F9fnKt8uioCKztF5Ww,attribute::RestaurantsReservations,value::RestaurantsReservations::false,scalar_attribute
12207,business::-0__F9fnKt8uioCKztF5Ww,attribute::RestaurantsDelivery,value::RestaurantsDelivery::false,scalar_attribute



 NESTED_ATTRIBUTE


,head,relation,tail,source
45693,business::-0__F9fnKt8uioCKztF5Ww,nested_attribute::Music::dj,value::Music::dj::false,nested_attribute
45694,business::-0__F9fnKt8uioCKztF5Ww,nested_attribute::Music::background_music,value::Music::background_music::false,nested_attribute
45695,business::-0__F9fnKt8uioCKztF5Ww,nested_attribute::Music::no_music,value::Music::no_music::false,nested_attribute
45696,business::-0__F9fnKt8uioCKztF5Ww,nested_attribute::Music::jukebox,value::Music::jukebox::false,nested_attribute
45697,business::-0__F9fnKt8uioCKztF5Ww,nested_attribute::Music::live,value::Music::live::false,nested_attribute


In [103]:
def parse_metadata_relation(
    relation
):

    relation = str(
        relation
    )


    if relation == "has_category":

        return {
            "evidence_family":
                "Category",

            "attribute":
                "Category",

            "parent_attribute":
                None
        }


    if relation.startswith(
        "attribute::"
    ):

        attribute = relation.split(
            "::",
            1
        )[1]


        return {
            "evidence_family":
                "Attribute",

            "attribute":
                attribute,

            "parent_attribute":
                None
        }


    if relation.startswith(
        "nested_attribute::"
    ):

        parts = relation.split(
            "::"
        )


        parent_attribute = (
            parts[1]
            if len(parts) > 1
            else None
        )


        nested_attribute = (
            "::".join(
                parts[2:]
            )
            if len(parts) > 2
            else None
        )


        return {
            "evidence_family":
                "Nested attribute",

            "attribute":
                nested_attribute,

            "parent_attribute":
                parent_attribute
        }


    return {
        "evidence_family":
            "Unknown",

        "attribute":
            relation,

        "parent_attribute":
            None
    }

In [104]:
def parse_metadata_tail(
    tail
):

    tail = str(
        tail
    )


    if tail.startswith(
        "category::"
    ):

        return tail.split(
            "::",
            1
        )[1]


    if tail.startswith(
        "value::"
    ):

        parts = tail.split(
            "::"
        )


        if len(parts) >= 2:

            return parts[-1]


    return tail

In [105]:
def parse_business_head(
    head
):

    head = str(
        head
    )


    prefix = "business::"


    if head.startswith(
        prefix
    ):

        return head[
            len(prefix):
        ]


    return head

In [106]:
structured_evidence = (
    metadata_evidence
    .copy()
)


structured_evidence[
    "business_id"
] = (
    structured_evidence[
        "head"
    ]
    .apply(
        parse_business_head
    )
)

In [107]:
parsed_relations = (
    structured_evidence[
        "relation"
    ]
    .apply(
        parse_metadata_relation
    )
    .apply(
        pd.Series
    )
)


structured_evidence = pd.concat(
    [
        structured_evidence,
        parsed_relations
    ],
    axis=1
)

In [108]:
structured_evidence[
    "value"
] = (
    structured_evidence[
        "tail"
    ]
    .apply(
        parse_metadata_tail
    )
)

In [109]:
structured_evidence[
    "evidence_label"
] = np.where(

    structured_evidence[
        "evidence_family"
    ]
    ==
    "Nested attribute",

    (
        structured_evidence[
            "parent_attribute"
        ].astype(str)
        +
        " — "
        +
        structured_evidence[
            "attribute"
        ].astype(str)
    ),

    structured_evidence[
        "attribute"
    ].astype(str)
)

In [110]:
structured_evidence = (
    structured_evidence[
        [
            "business_id",
            "evidence_family",
            "evidence_label",
            "value",
            "source",
            "head",
            "relation",
            "tail"
        ]
    ]
    .copy()
)

In [111]:
structured_evidence.head(
    15
)

,business_id,evidence_family,evidence_label,value,source,head,relation,tail
0,-0__F9fnKt8uioCKztF5Ww,Category,Category,Cafes,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Cafes
1,-0__F9fnKt8uioCKztF5Ww,Category,Category,Nightlife,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Nightlife
2,-0__F9fnKt8uioCKztF5Ww,Category,Category,Cocktail Bars,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Cocktail Bars
3,-0__F9fnKt8uioCKztF5Ww,Category,Category,Peruvian,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Peruvian
4,-0__F9fnKt8uioCKztF5Ww,Category,Category,Restaurants,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Restaurants
5,-0__F9fnKt8uioCKztF5Ww,Category,Category,Vegan,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Vegan
6,-0__F9fnKt8uioCKztF5Ww,Category,Category,Bars,category,business::-0__F9fnKt8uioCKztF5Ww,has_category,category::Bars
7,-1XSzguS6XLN-V6MVZMg2A,Category,Category,Cajun/Creole,category,business::-1XSzguS6XLN-V6MVZMg2A,has_category,category::Cajun/Creole
8,-1XSzguS6XLN-V6MVZMg2A,Category,Category,Soup,category,business::-1XSzguS6XLN-V6MVZMg2A,has_category,category::Soup
9,-1XSzguS6XLN-V6MVZMg2A,Category,Category,Seafood,category,business::-1XSzguS6XLN-V6MVZMg2A,has_category,category::Seafood


In [112]:
k44_structured_evidence_check = pd.Series({

    "Rows":
        len(
            structured_evidence
        ),

    "Expected 83989":
        (
            len(
                structured_evidence
            )
            ==
            83989
        ),

    "Businesses":
        structured_evidence[
            "business_id"
        ].nunique(),

    "Expected 2516 businesses":
        (
            structured_evidence[
                "business_id"
            ].nunique()
            ==
            2516
        ),

    "No unknown evidence families":
        (
            structured_evidence[
                "evidence_family"
            ]
            .eq(
                "Unknown"
            )
            .sum()
            ==
            0
        ),

    "No missing readable labels":
        structured_evidence[
            "evidence_label"
        ]
        .notna()
        .all(),

    "No missing readable values":
        structured_evidence[
            "value"
        ]
        .notna()
        .all(),

    "Business IDs match model catalogue":
        (
            set(
                structured_evidence[
                    "business_id"
                ].astype(str)
            )
            ==
            model_business_ids
        )
})


k44_structured_evidence_check

Rows                                  83989
Expected 83989                         True
Businesses                             2516
Expected 2516 businesses               True
No unknown evidence families           True
No missing readable labels             True
No missing readable values             True
Business IDs match model catalogue     True
dtype: object

In [114]:
def get_business_structured_evidence(
    business_id,
    max_items=None
):

    business_id = str(
        business_id
    )


    evidence = (
        structured_evidence.loc[
            structured_evidence[
                "business_id"
            ].astype(str)
            ==
            business_id
        ]
        .copy()
    )


    if len(
        evidence
    ) == 0:

        return pd.DataFrame()


    evidence[
        "family_order"
    ] = (
        evidence[
            "evidence_family"
        ]
        .map({
            "Category": 0,
            "Attribute": 1,
            "Nested attribute": 2
        })
    )


    evidence = (
        evidence
        .sort_values(
            [
                "family_order",
                "evidence_label",
                "value"
            ],
            ascending=True
        )
    )


    if max_items is not None:

        evidence = evidence.head(
            max_items
        )


    return (
        evidence[
            [
                "evidence_family",
                "evidence_label",
                "value",
                "relation",
                "tail"
            ]
        ]
        .reset_index(
            drop=True
        )
    )

In [115]:
structured_evidence_smoke = (
    get_business_structured_evidence(
        business_id=
            text_evidence_smoke_business_id
    )
)


structured_evidence_smoke

,evidence_family,evidence_label,value,relation,tail
0,Category,Category,Cajun/Creole,has_category,category::Cajun/Creole
1,Category,Category,Restaurants,has_category,category::Restaurants
2,Category,Category,Seafood,has_category,category::Seafood
3,Attribute,Alcohol,full_bar,attribute::Alcohol,value::Alcohol::full_bar
4,Attribute,BYOBCorkage,no,attribute::BYOBCorkage,value::BYOBCorkage::no
5,Attribute,BikeParking,true,attribute::BikeParking,value::BikeParking::true
6,Attribute,BusinessAcceptsCreditCards,false,attribute::BusinessAcceptsCreditCards,value::BusinessAcceptsCreditCards::false
7,Attribute,ByAppointmentOnly,false,attribute::ByAppointmentOnly,value::ByAppointmentOnly::false
8,Attribute,Caters,true,attribute::Caters,value::Caters::true
9,Attribute,DogsAllowed,false,attribute::DogsAllowed,value::DogsAllowed::false


In [116]:
structured_evidence_smoke[
    [
        "evidence_family",
        "evidence_label",
        "value"
    ]
].head(
    25
)

,evidence_family,evidence_label,value
0,Category,Category,Cajun/Creole
1,Category,Category,Restaurants
2,Category,Category,Seafood
3,Attribute,Alcohol,full_bar
4,Attribute,BYOBCorkage,no
5,Attribute,BikeParking,true
6,Attribute,BusinessAcceptsCreditCards,false
7,Attribute,ByAppointmentOnly,false
8,Attribute,Caters,true
9,Attribute,DogsAllowed,false


In [117]:
def get_business_categories(
    business_id
):

    business_id = str(
        business_id
    )


    categories = (
        structured_evidence.loc[
            (
                structured_evidence[
                    "business_id"
                ].astype(str)
                ==
                business_id
            )
            &
            (
                structured_evidence[
                    "evidence_family"
                ]
                ==
                "Category"
            ),
            "value"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


    return categories

In [118]:
get_business_categories(
    text_evidence_smoke_business_id
)

['Cajun/Creole', 'Restaurants', 'Seafood']

In [119]:
def get_business_attributes(
    business_id
):

    business_id = str(
        business_id
    )


    attributes = (
        structured_evidence.loc[
            (
                structured_evidence[
                    "business_id"
                ].astype(str)
                ==
                business_id
            )
            &
            (
                structured_evidence[
                    "evidence_family"
                ]
                !=
                "Category"
            ),
            [
                "evidence_family",
                "evidence_label",
                "value"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    return attributes

In [120]:
get_business_attributes(
    text_evidence_smoke_business_id
).head(
    20
)

,evidence_family,evidence_label,value
0,Attribute,GoodForKids,true
1,Attribute,RestaurantsAttire,casual
2,Attribute,Alcohol,full_bar
3,Attribute,Caters,true
4,Attribute,RestaurantsPriceRange2,2
5,Attribute,RestaurantsReservations,false
6,Attribute,RestaurantsGoodForGroups,true
7,Attribute,BYOBCorkage,no
8,Attribute,NoiseLevel,average
9,Attribute,HasTV,false


In [121]:
business_structured_evidence_strength = (
    structured_evidence
    .groupby(
        "business_id",
        as_index=False
    )
    .agg(

        metadata_evidence_count=(
            "tail",
            "size"
        ),

        category_count=(
            "evidence_family",
            lambda x: (
                x == "Category"
            ).sum()
        ),

        scalar_attribute_count=(
            "evidence_family",
            lambda x: (
                x == "Attribute"
            ).sum()
        ),

        nested_attribute_count=(
            "evidence_family",
            lambda x: (
                x
                ==
                "Nested attribute"
            ).sum()
        )
    )
)

In [122]:
business_structured_evidence_strength[
    [
        "metadata_evidence_count",
        "category_count",
        "scalar_attribute_count",
        "nested_attribute_count"
    ]
].describe()

,metadata_evidence_count,category_count,scalar_attribute_count,nested_attribute_count
count,2516.000000,2516.000000,2516.000000,2516.000000
mean,33.381955,4.850159,13.310811,15.220986
std,15.300800,2.342664,6.142604,9.220022
min,1.000000,1.000000,0.000000,0.000000
25%,20.000000,3.000000,8.000000,5.000000
50%,36.000000,4.000000,15.000000,16.000000
75%,43.000000,6.000000,18.000000,20.000000
max,72.000000,19.000000,27.000000,34.000000


In [123]:
rq3_metadata_reference = (
    rq3_analysis[
        [
            "business_id",
            "metadata_edge_count"
        ]
    ]
    .drop_duplicates(
        subset=[
            "business_id"
        ]
    )
)

In [124]:
metadata_count_crosscheck = (
    rq3_metadata_reference
    .merge(
        business_structured_evidence_strength[
            [
                "business_id",
                "metadata_evidence_count"
            ]
        ],

        on="business_id",

        how="left",

        validate="one_to_one"
    )
)

In [125]:
k44_metadata_count_crosscheck = pd.Series({

    "Target businesses checked":
        len(
            metadata_count_crosscheck
        ),

    "No missing explanation counts":
        metadata_count_crosscheck[
            "metadata_evidence_count"
        ]
        .notna()
        .all(),

    "RQ3 and explanation metadata counts identical":
        (
            metadata_count_crosscheck[
                "metadata_edge_count"
            ]
            .to_numpy()
            ==
            metadata_count_crosscheck[
                "metadata_evidence_count"
            ]
            .to_numpy()
        ).all()
})


k44_metadata_count_crosscheck

Target businesses checked                        1966
No missing explanation counts                    True
RQ3 and explanation metadata counts identical    True
dtype: object

In [126]:
structured_evidence.to_parquet(
    explanation_dir
    /
    "k4_structured_business_evidence.parquet",

    index=False
)


business_structured_evidence_strength.to_parquet(
    explanation_dir
    /
    "k4_business_structured_evidence_strength.parquet",

    index=False
)

In [127]:
k44_save_check = pd.Series({

    "Structured evidence saved":
        (
            explanation_dir
            /
            "k4_structured_business_evidence.parquet"
        ).exists(),

    "Structured evidence strength saved":
        (
            explanation_dir
            /
            "k4_business_structured_evidence_strength.parquet"
        ).exists()
})


k44_save_check

Structured evidence saved             True
Structured evidence strength saved    True
dtype: bool

### K4.5 — Representative Visual Evidence

Visual explanation evidence is restricted to photographs actually selected
for the CLIP representation used by KGRec-MM.

Individual selected-photo embeddings are compared with the final pooled
business visual embedding using cosine similarity. Because both image and
business vectors are L2-normalised, their dot product provides the similarity
measure.

Highly similar photographs are interpreted as representative examples of the
visual information encoded for the business. When a small evidence set is
required, label-diverse photographs are prioritised because the business
visual representation was constructed using label-balanced pooling.

Representative-image similarity indicates consistency with the pooled visual
representation and should not be interpreted as causal attribution of a
recommendation to an individual photograph.

In [ ]:
# --------------------------------------------------
# K4.5 — Locate frozen CLIP artefacts
# --------------------------------------------------

image_embedding_candidates = list(
    processed_data_root.rglob(
        "new_orleans_clip_vit_b32_image_embeddings.npy"
    )
)


image_index_candidates = list(
    processed_data_root.rglob(
        "new_orleans_clip_vit_b32_image_embedding_index.parquet"
    )
)


business_visual_embedding_candidates = list(
    processed_data_root.rglob(
        "new_orleans_clip_vit_b32_business_embeddings.npy"
    )
)


business_visual_index_candidates = list(
    processed_data_root.rglob(
        "new_orleans_clip_vit_b32_business_embedding_index.parquet"
    )
)


print(
    "Image embeddings:",
    image_embedding_candidates
)

print(
    "Image index:",
    image_index_candidates
)

print(
    "Business visual embeddings:",
    business_visual_embedding_candidates
)

print(
    "Business visual index:",
    business_visual_index_candidates
)

Image embeddings: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_image_embeddings.npy')]
Image index: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_image_embedding_index.parquet')]
Business visual embeddings: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_business_embeddings.npy')]
Business visual index: [PosixPath('/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_business_embedding_index.parquet')]


In [129]:
k45_path_check = pd.Series({

    "One image embedding file":
        len(
            image_embedding_candidates
        ) == 1,

    "One image index":
        len(
            image_index_candidates
        ) == 1,

    "One business visual embedding file":
        len(
            business_visual_embedding_candidates
        ) == 1,

    "One business visual index":
        len(
            business_visual_index_candidates
        ) == 1
})


k45_path_check

One image embedding file              True
One image index                       True
One business visual embedding file    True
One business visual index             True
dtype: bool

In [130]:
image_embedding_path = (
    image_embedding_candidates[0]
)

image_index_path = (
    image_index_candidates[0]
)

business_visual_embedding_path = (
    business_visual_embedding_candidates[0]
)

business_visual_index_path = (
    business_visual_index_candidates[0]
)

In [131]:
image_embeddings = np.load(
    image_embedding_path,
    mmap_mode="r"
)


image_embedding_index = pd.read_parquet(
    image_index_path
)


business_visual_embeddings = np.load(
    business_visual_embedding_path,
    mmap_mode="r"
)


business_visual_embedding_index = pd.read_parquet(
    business_visual_index_path
)

In [132]:
print(
    "Image embeddings shape:",
    image_embeddings.shape
)

print(
    "Image index columns:",
    image_embedding_index.columns.tolist()
)

print(
    "Business visual embeddings shape:",
    business_visual_embeddings.shape
)

print(
    "Business visual index columns:",
    business_visual_embedding_index.columns.tolist()
)

Image embeddings shape: (6681, 512)
Image index columns: ['manifest_position', 'photo_id', 'business_id', 'photo_filename', 'label', 'selection_rank', 'in_personalisation_core']
Business visual embeddings shape: (1977, 512)
Business visual index columns: ['business_id', 'image_count', 'unique_image_labels', 'image_labels', 'in_personalisation_core', 'aggregation_method']


In [133]:
k45_embedding_alignment_check = pd.Series({

    "Image vectors":
        image_embeddings.shape[0],

    "Image dimension":
        image_embeddings.shape[1],

    "Image index rows":
        len(
            image_embedding_index
        ),

    "Expected image vectors":
        (
            image_embeddings.shape[0]
            ==
            6681
        ),

    "Expected image dimension":
        (
            image_embeddings.shape[1]
            ==
            512
        ),

    "Image index aligns":
        (
            len(
                image_embedding_index
            )
            ==
            image_embeddings.shape[0]
        ),

    "Business visual vectors":
        business_visual_embeddings.shape[0],

    "Expected visual businesses":
        (
            business_visual_embeddings.shape[0]
            ==
            1977
        ),

    "Expected business dimension":
        (
            business_visual_embeddings.shape[1]
            ==
            512
        ),

    "Business index aligns":
        (
            len(
                business_visual_embedding_index
            )
            ==
            business_visual_embeddings.shape[0]
        )
})


k45_embedding_alignment_check

Image vectors                  6681
Image dimension                 512
Image index rows               6681
Expected image vectors         True
Expected image dimension       True
Image index aligns             True
Business visual vectors        1977
Expected visual businesses     True
Expected business dimension    True
Business index aligns          True
dtype: object

In [134]:
image_embedding_index = (
    image_embedding_index
    .reset_index(
        drop=True
    )
    .copy()
)


image_embedding_index[
    "image_embedding_row"
] = np.arange(
    len(
        image_embedding_index
    )
)


business_visual_embedding_index = (
    business_visual_embedding_index
    .reset_index(
        drop=True
    )
    .copy()
)


business_visual_embedding_index[
    "business_visual_embedding_row"
] = np.arange(
    len(
        business_visual_embedding_index
    )
)

In [135]:
personalisation_image_index = (
    image_embedding_index.loc[
        image_embedding_index[
            "in_personalisation_core"
        ].astype(bool)
    ]
    .copy()
)

In [136]:
k45_photo_alignment_check = pd.Series({

    "Personalisation image-index rows":
        len(
            personalisation_image_index
        ),

    "Expected 6122":
        (
            len(
                personalisation_image_index
            )
            ==
            6122
        ),

    "Exact selected photo-ID match":
        (
            set(
                personalisation_image_index[
                    "photo_id"
                ].astype(str)
            )
            ==
            set(
                visual_evidence[
                    "photo_id"
                ].astype(str)
            )
        ),

    "Exact visual business-ID match":
        (
            set(
                personalisation_image_index[
                    "business_id"
                ].astype(str)
            )
            ==
            visual_business_ids
        )
})


k45_photo_alignment_check

Personalisation image-index rows    6122
Expected 6122                       True
Exact selected photo-ID match       True
Exact visual business-ID match      True
dtype: object

In [137]:
visual_image_lookup = (
    personalisation_image_index[
        [
            "photo_id",
            "business_id",
            "image_embedding_row"
        ]
    ]
    .merge(
        visual_evidence[
            [
                "photo_id",
                "business_id",
                "photo_filename",
                "label",
                "caption_clean",
                "selection_rank",
                "selection_strategy"
            ]
        ],

        on=[
            "photo_id",
            "business_id"
        ],

        how="left",

        validate="one_to_one"
    )
)

In [138]:
visual_image_lookup_check = pd.Series({

    "Rows":
        len(
            visual_image_lookup
        ),

    "Expected 6122":
        (
            len(
                visual_image_lookup
            )
            ==
            6122
        ),

    "Photo IDs unique":
        visual_image_lookup[
            "photo_id"
        ].is_unique,

    "All image rows unique":
        visual_image_lookup[
            "image_embedding_row"
        ].is_unique,

    "All labels recovered":
        visual_image_lookup[
            "label"
        ].notna().all(),

    "All filenames recovered":
        visual_image_lookup[
            "photo_filename"
        ].notna().all()
})


visual_image_lookup_check

Rows                       6122
Expected 6122              True
Photo IDs unique           True
All image rows unique      True
All labels recovered       True
All filenames recovered    True
dtype: object

In [139]:
business_visual_lookup = (
    business_visual_embedding_index.loc[
        business_visual_embedding_index[
            "business_id"
        ].astype(str)
        .isin(
            visual_business_ids
        )
    ]
    [
        [
            "business_id",
            "business_visual_embedding_row"
        ]
    ]
    .copy()
)

In [140]:
business_visual_lookup_check = pd.Series({

    "Personalisation visual businesses":
        len(
            business_visual_lookup
        ),

    "Expected 1729":
        (
            len(
                business_visual_lookup
            )
            ==
            1729
        ),

    "Business IDs unique":
        business_visual_lookup[
            "business_id"
        ].is_unique,

    "Exact visual-business ID match":
        (
            set(
                business_visual_lookup[
                    "business_id"
                ].astype(str)
            )
            ==
            visual_business_ids
        )
})


business_visual_lookup_check

Personalisation visual businesses    1729
Expected 1729                        True
Business IDs unique                  True
Exact visual-business ID match       True
dtype: object

In [141]:
business_visual_row_by_id = dict(
    zip(
        business_visual_lookup[
            "business_id"
        ].astype(str),

        business_visual_lookup[
            "business_visual_embedding_row"
        ]
    )
)

In [142]:
personalisation_image_rows = (
    visual_image_lookup[
        "image_embedding_row"
    ]
    .to_numpy(
        dtype=int
    )
)


personalisation_business_rows = (
    business_visual_lookup[
        "business_visual_embedding_row"
    ]
    .to_numpy(
        dtype=int
    )
)

In [143]:
image_norms = np.linalg.norm(
    np.asarray(
        image_embeddings[
            personalisation_image_rows
        ],
        dtype=np.float32
    ),
    axis=1
)


business_visual_norms = np.linalg.norm(
    np.asarray(
        business_visual_embeddings[
            personalisation_business_rows
        ],
        dtype=np.float32
    ),
    axis=1
)

In [144]:
k45_norm_check = pd.Series({

    "All image norms finite":
        np.isfinite(
            image_norms
        ).all(),

    "All business norms finite":
        np.isfinite(
            business_visual_norms
        ).all(),

    "Image embeddings unit normalised":
        np.allclose(
            image_norms,
            1.0,
            atol=1e-4
        ),

    "Business visual embeddings unit normalised":
        np.allclose(
            business_visual_norms,
            1.0,
            atol=1e-4
        )
})


k45_norm_check

All image norms finite                        True
All business norms finite                     True
Image embeddings unit normalised              True
Business visual embeddings unit normalised    True
dtype: bool

In [145]:
def get_representative_visual_evidence(
    business_id,
    top_n=3,
    prefer_label_diversity=True
):

    business_id = str(
        business_id
    )


    if business_id not in business_visual_row_by_id:

        return pd.DataFrame(
            columns=[
                "photo_id",
                "photo_filename",
                "label",
                "caption_clean",
                "selection_rank",
                "visual_representativeness"
            ]
        )


    business_embedding_row = (
        business_visual_row_by_id[
            business_id
        ]
    )


    business_vector = np.asarray(
        business_visual_embeddings[
            business_embedding_row
        ],
        dtype=np.float32
    )


    business_images = (
        visual_image_lookup.loc[
            visual_image_lookup[
                "business_id"
            ].astype(str)
            ==
            business_id
        ]
        .copy()
    )


    image_rows = (
        business_images[
            "image_embedding_row"
        ]
        .to_numpy(
            dtype=int
        )
    )


    vectors = np.asarray(
        image_embeddings[
            image_rows
        ],
        dtype=np.float32
    )


    business_images[
        "visual_representativeness"
    ] = (
        vectors
        @
        business_vector
    )


    ranked = (
        business_images
        .sort_values(
            [
                "visual_representativeness",
                "selection_rank",
                "photo_id"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .copy()
    )


    if prefer_label_diversity:

        # Best representative image for each label
        diverse_first = (
            ranked
            .drop_duplicates(
                subset=[
                    "label"
                ],
                keep="first"
            )
            .head(
                top_n
            )
        )


        selected_ids = set(
            diverse_first[
                "photo_id"
            ]
        )


        # Fill remaining places using next-best images
        remaining_slots = (
            top_n
            -
            len(
                diverse_first
            )
        )


        if remaining_slots > 0:

            remainder = (
                ranked.loc[
                    ~ranked[
                        "photo_id"
                    ].isin(
                        selected_ids
                    )
                ]
                .head(
                    remaining_slots
                )
            )


            result = pd.concat(
                [
                    diverse_first,
                    remainder
                ],
                ignore_index=True
            )

        else:

            result = (
                diverse_first
                .copy()
            )


        result = (
            result
            .sort_values(
                "visual_representativeness",
                ascending=False
            )
        )

    else:

        result = (
            ranked
            .head(
                top_n
            )
        )


    return (
        result[
            [
                "photo_id",
                "photo_filename",
                "label",
                "caption_clean",
                "selection_rank",
                "visual_representativeness"
            ]
        ]
        .reset_index(
            drop=True
        )
    )

In [146]:
visual_evidence_smoke = (
    get_representative_visual_evidence(
        business_id=
            text_evidence_smoke_business_id,

        top_n=
            3,

        prefer_label_diversity=
            True
    )
)


visual_evidence_smoke

,photo_id,photo_filename,label,caption_clean,selection_rank,visual_representativeness
0,N_Yp6SghyrXdXqliR3irRg,N_Yp6SghyrXdXqliR3irRg.jpg,drink,This is some good hot sauce,4,0.823710
1,7x5bioN1uXRwixAwmcrasA,7x5bioN1uXRwixAwmcrasA.jpg,food,Shrimp creole,2,0.823318
2,m6HkNAHWigVOie41N4Phkg,m6HkNAHWigVOie41N4Phkg.jpg,inside,Lunch,1,0.820554


In [147]:
visual_evidence_smoke_check = pd.Series({

    "Evidence returned":
        (
            len(
                visual_evidence_smoke
            )
            >
            0
        ),

    "Maximum three images":
        (
            len(
                visual_evidence_smoke
            )
            <=
            3
        ),

    "Similarity finite":
        np.isfinite(
            visual_evidence_smoke[
                "visual_representativeness"
            ]
        ).all(),

    "Similarity <= 1":
        (
            visual_evidence_smoke[
                "visual_representativeness"
            ]
            <=
            1.00001
        ).all(),

    "Similarity >= -1":
        (
            visual_evidence_smoke[
                "visual_representativeness"
            ]
            >=
            -1.00001
        ).all(),

    "All photos belong to correct business":
        set(
            visual_evidence_smoke[
                "photo_id"
            ]
        ).issubset(
            set(
                visual_image_lookup.loc[
                    visual_image_lookup[
                        "business_id"
                    ].astype(str)
                    ==
                    text_evidence_smoke_business_id,
                    "photo_id"
                ]
            )
        )
})


visual_evidence_smoke_check

Evidence returned                        True
Maximum three images                     True
Similarity finite                        True
Similarity <= 1                          True
Similarity >= -1                         True
All photos belong to correct business    True
dtype: bool

In [148]:
visual_representativeness = np.empty(
    len(
        visual_image_lookup
    ),
    dtype=np.float32
)

In [149]:
for business_id, group in visual_image_lookup.groupby(
    "business_id",
    sort=False
):

    business_id = str(
        business_id
    )


    business_embedding_row = (
        business_visual_row_by_id[
            business_id
        ]
    )


    business_vector = np.asarray(
        business_visual_embeddings[
            business_embedding_row
        ],
        dtype=np.float32
    )


    image_rows = (
        group[
            "image_embedding_row"
        ]
        .to_numpy(
            dtype=int
        )
    )


    vectors = np.asarray(
        image_embeddings[
            image_rows
        ],
        dtype=np.float32
    )


    similarities = (
        vectors
        @
        business_vector
    )


    visual_representativeness[
        group.index.to_numpy()
    ] = similarities

In [150]:
visual_image_lookup[
    "visual_representativeness"
] = (
    visual_representativeness
)

In [151]:
visual_representativeness_check = pd.Series({

    "Rows":
        len(
            visual_image_lookup
        ),

    "Expected 6122":
        (
            len(
                visual_image_lookup
            )
            ==
            6122
        ),

    "All finite":
        np.isfinite(
            visual_image_lookup[
                "visual_representativeness"
            ]
        ).all(),

    "Maximum <= 1":
        (
            visual_image_lookup[
                "visual_representativeness"
            ].max()
            <=
            1.00001
        ),

    "Minimum >= -1":
        (
            visual_image_lookup[
                "visual_representativeness"
            ].min()
            >=
            -1.00001
        )
})


visual_representativeness_check

Rows             6122
Expected 6122    True
All finite       True
Maximum <= 1     True
Minimum >= -1    True
dtype: object

In [152]:
visual_image_lookup[
    "has_caption"
] = (
    visual_image_lookup[
        "caption_clean"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

In [153]:
business_visual_evidence_strength = (
    visual_image_lookup
    .groupby(
        "business_id",
        as_index=False
    )
    .agg(

        selected_image_count=(
            "photo_id",
            "size"
        ),

        label_diversity=(
            "label",
            "nunique"
        ),

        captioned_image_count=(
            "has_caption",
            "sum"
        ),

        top_image_similarity=(
            "visual_representativeness",
            "max"
        ),

        median_image_similarity=(
            "visual_representativeness",
            "median"
        )
    )
)

In [154]:
business_visual_evidence_strength[
    [
        "selected_image_count",
        "label_diversity",
        "captioned_image_count",
        "top_image_similarity",
        "median_image_similarity"
    ]
].describe()

,selected_image_count,label_diversity,captioned_image_count,top_image_similarity,median_image_similarity
count,1729.000000,1729.000000,1729.00000,1729.000000,1729.000000
mean,3.540775,2.177559,2.00000,0.917107,0.885179
std,1.633945,1.095680,1.49691,0.050719,0.069987
min,1.000000,1.000000,0.00000,0.788585,0.710261
25%,2.000000,1.000000,1.00000,0.881583,0.832231
50%,4.000000,2.000000,2.00000,0.906908,0.866547
75%,5.000000,3.000000,3.00000,0.947345,0.930428
max,5.000000,5.000000,5.00000,1.000000,1.000000


In [155]:
rq3_visual_reference = (
    rq3_analysis[
        [
            "business_id",
            "image_count",
            "unique_image_labels"
        ]
    ]
    .drop_duplicates(
        subset=[
            "business_id"
        ]
    )
)

In [156]:
rq3_visual_reference = (
    rq3_visual_reference.loc[
        rq3_visual_reference[
            "image_count"
        ]
        >
        0
    ]
    .copy()
)

In [157]:
visual_count_crosscheck = (
    rq3_visual_reference
    .merge(
        business_visual_evidence_strength,

        on="business_id",

        how="left",

        validate="one_to_one"
    )
)

In [158]:
k45_visual_count_crosscheck = pd.Series({

    "Visual target businesses checked":
        len(
            visual_count_crosscheck
        ),

    "Expected 1452":
        (
            len(
                visual_count_crosscheck
            )
            ==
            1452
        ),

    "No missing explanation evidence":
        visual_count_crosscheck[
            "selected_image_count"
        ].notna().all(),

    "Image counts identical":
        (
            visual_count_crosscheck[
                "image_count"
            ].to_numpy()
            ==
            visual_count_crosscheck[
                "selected_image_count"
            ].to_numpy()
        ).all(),

    "Label diversity identical":
        (
            visual_count_crosscheck[
                "unique_image_labels"
            ].to_numpy()
            ==
            visual_count_crosscheck[
                "label_diversity"
            ].to_numpy()
        ).all()
})


k45_visual_count_crosscheck

Visual target businesses checked    1452
Expected 1452                       True
No missing explanation evidence     True
Image counts identical              True
Label diversity identical           True
dtype: object

In [159]:
visual_image_lookup.to_parquet(
    explanation_dir
    /
    "k4_visual_image_representativeness.parquet",

    index=False
)


business_visual_evidence_strength.to_parquet(
    explanation_dir
    /
    "k4_business_visual_evidence_strength.parquet",

    index=False
)

In [160]:
k45_save_check = pd.Series({

    "Visual representativeness saved":
        (
            explanation_dir
            /
            "k4_visual_image_representativeness.parquet"
        ).exists(),

    "Business visual evidence strength saved":
        (
            explanation_dir
            /
            "k4_business_visual_evidence_strength.parquet"
        ).exists()
})


k45_save_check

Visual representativeness saved            True
Business visual evidence strength saved    True
dtype: bool

### K4.6 — Unified Business Evidence Profiles

The structured, textual and visual evidence channels are consolidated into a
common business-level evidence profile.

Each profile contains business identity information, structured knowledge-graph
evidence, representative training-review evidence, and, where available,
representative visual evidence from the photographs used to construct the
CLIP business representation.

The unified profile provides the evidence base from which recommendation-level
case explanations are subsequently constructed. At this stage, evidence is
business-specific rather than user-specific and should therefore not be
interpreted as a complete explanation of an individual recommendation.

In [161]:
# --------------------------------------------------
# K4.6 — Unified evidence coverage
# --------------------------------------------------

business_evidence_coverage = (
    business_display[
        [
            "business_id",
            "name",
            "categories"
        ]
    ]
    .copy()
)

In [162]:
business_evidence_coverage = (
    business_evidence_coverage
    .merge(
        business_structured_evidence_strength,

        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

In [163]:
business_evidence_coverage = (
    business_evidence_coverage
    .merge(
        business_text_evidence_strength,

        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

In [164]:
business_evidence_coverage = (
    business_evidence_coverage
    .merge(
        business_visual_evidence_strength,

        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

In [165]:
visual_count_columns = [
    "selected_image_count",
    "label_diversity",
    "captioned_image_count"
]


business_evidence_coverage[
    visual_count_columns
] = (
    business_evidence_coverage[
        visual_count_columns
    ]
    .fillna(0)
    .astype(int)
)

In [166]:
business_evidence_coverage[
    "has_visual_evidence"
] = (
    business_evidence_coverage[
        "selected_image_count"
    ]
    >
    0
)

In [167]:
k46_coverage_check = pd.Series({

    "Rows":
        len(
            business_evidence_coverage
        ),

    "Expected 2516":
        (
            len(
                business_evidence_coverage
            )
            ==
            2516
        ),

    "Business IDs unique":
        business_evidence_coverage[
            "business_id"
        ].is_unique,

    "All businesses have structured evidence":
        business_evidence_coverage[
            "metadata_evidence_count"
        ].notna().all(),

    "All businesses have text evidence":
        business_evidence_coverage[
            "training_review_count"
        ].notna().all(),

    "Visual businesses":
        int(
            business_evidence_coverage[
                "has_visual_evidence"
            ].sum()
        ),

    "Expected visual businesses":
        (
            business_evidence_coverage[
                "has_visual_evidence"
            ].sum()
            ==
            1729
        ),

    "Expected image-less businesses":
        (
            (
                ~business_evidence_coverage[
                    "has_visual_evidence"
                ]
            ).sum()
            ==
            787
        ),

    "Image-less similarity is missing":
        business_evidence_coverage.loc[
            ~business_evidence_coverage[
                "has_visual_evidence"
            ],
            "top_image_similarity"
        ]
        .isna()
        .all()
})


k46_coverage_check

Rows                                       2516
Expected 2516                              True
Business IDs unique                        True
All businesses have structured evidence    True
All businesses have text evidence          True
Visual businesses                          1729
Expected visual businesses                 True
Expected image-less businesses             True
Image-less similarity is missing           True
dtype: object

In [168]:
evidence_coverage_summary = pd.Series({

    "Businesses":
        len(
            business_evidence_coverage
        ),

    "Businesses with structured KG evidence":
        int(
            business_evidence_coverage[
                "metadata_evidence_count"
            ]
            .notna()
            .sum()
        ),

    "Businesses with training text evidence":
        int(
            business_evidence_coverage[
                "training_review_count"
            ]
            .notna()
            .sum()
        ),

    "Businesses with visual evidence":
        int(
            business_evidence_coverage[
                "has_visual_evidence"
            ]
            .sum()
        ),

    "Businesses without visual evidence":
        int(
            (
                ~business_evidence_coverage[
                    "has_visual_evidence"
                ]
            )
            .sum()
        ),

    "Median metadata evidence count":
        float(
            business_evidence_coverage[
                "metadata_evidence_count"
            ].median()
        ),

    "Median training reviews":
        float(
            business_evidence_coverage[
                "training_review_count"
            ].median()
        ),

    "Median selected images among visual businesses":
        float(
            business_evidence_coverage.loc[
                business_evidence_coverage[
                    "has_visual_evidence"
                ],
                "selected_image_count"
            ].median()
        )
})


evidence_coverage_summary

Businesses                                        2516.0
Businesses with structured KG evidence            2516.0
Businesses with training text evidence            2516.0
Businesses with visual evidence                   1729.0
Businesses without visual evidence                 787.0
Median metadata evidence count                      36.0
Median training reviews                             19.0
Median selected images among visual businesses       4.0
dtype: float64

In [169]:
def get_business_identity(
    business_id
):

    business_id = str(
        business_id
    )


    result = (
        business_display.loc[
            business_display[
                "business_id"
            ].astype(str)
            ==
            business_id
        ]
        .copy()
    )


    if len(
        result
    ) == 0:

        raise KeyError(
            f"Business ID not found: {business_id}"
        )


    return (
        result
        .iloc[0]
        .to_dict()
    )

In [170]:
def get_compact_structured_evidence(
    business_id,
    max_attributes=8
):

    business_id = str(
        business_id
    )


    evidence = (
        get_business_structured_evidence(
            business_id
        )
    )


    if len(
        evidence
    ) == 0:

        return {
            "categories": [],
            "attributes": []
        }


    categories = (
        evidence.loc[
            evidence[
                "evidence_family"
            ]
            ==
            "Category",
            "value"
        ]
        .drop_duplicates()
        .tolist()
    )


    attributes = (
        evidence.loc[
            evidence[
                "evidence_family"
            ]
            !=
            "Category",
            [
                "evidence_family",
                "evidence_label",
                "value"
            ]
        ]
        .copy()
    )

In [172]:
def get_compact_structured_evidence(
    business_id,
    max_attributes=8
):

    business_id = str(
        business_id
    )


    evidence = (
        get_business_structured_evidence(
            business_id
        )
    )


    if len(evidence) == 0:

        return {
            "categories": [],
            "attributes": []
        }


    # ---------------------------------------------
    # Categories
    # ---------------------------------------------

    categories = (
        evidence.loc[
            evidence[
                "evidence_family"
            ]
            ==
            "Category",
            "value"
        ]
        .drop_duplicates()
        .tolist()
    )


    # ---------------------------------------------
    # Descriptive attributes
    # ---------------------------------------------

    attributes = (
        evidence.loc[
            evidence[
                "evidence_family"
            ]
            !=
            "Category",
            [
                "evidence_family",
                "evidence_label",
                "value"
            ]
        ]
        .copy()
    )


    # Presentation heuristic only:
    # positive/descriptive values shown before
    # explicit false/no values.
    #
    # This is NOT model importance.
    attributes[
        "presentation_priority"
    ] = np.where(
        attributes[
            "value"
        ]
        .astype(str)
        .str.lower()
        .isin(
            [
                "false",
                "none",
                "no"
            ]
        ),
        1,
        0
    )


    attributes = (
        attributes
        .sort_values(
            [
                "presentation_priority",
                "evidence_family",
                "evidence_label"
            ]
        )
        .head(
            max_attributes
        )
    )


    attribute_records = (
        attributes[
            [
                "evidence_family",
                "evidence_label",
                "value"
            ]
        ]
        .to_dict(
            orient="records"
        )
    )


    return {
        "categories":
            categories,

        "attributes":
            attribute_records
    }

In [173]:
get_compact_structured_evidence(
    text_evidence_smoke_business_id
)

{'categories': ['Cajun/Creole', 'Restaurants', 'Seafood'],
 'attributes': [{'evidence_family': 'Attribute',
   'evidence_label': 'Alcohol',
   'value': 'full_bar'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'BikeParking',
   'value': 'true'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'Caters',
   'value': 'true'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'GoodForKids',
   'value': 'true'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'NoiseLevel',
   'value': 'average'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'RestaurantsAttire',
   'value': 'casual'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'RestaurantsGoodForGroups',
   'value': 'true'},
  {'evidence_family': 'Attribute',
   'evidence_label': 'RestaurantsPriceRange2',
   'value': '2'}]}

In [174]:
def build_business_evidence_profile(
    business_id,
    top_text_reviews=3,
    top_visual_images=3,
    max_structured_attributes=8
):

    business_id = str(
        business_id
    )


    identity = (
        get_business_identity(
            business_id
        )
    )


    structured = (
        get_compact_structured_evidence(
            business_id=
                business_id,

            max_attributes=
                max_structured_attributes
        )
    )


    text_evidence = (
        get_representative_text_evidence(
            business_id=
                business_id,

            top_n=
                top_text_reviews
        )
    )


    visual_evidence_profile = (
        get_representative_visual_evidence(
            business_id=
                business_id,

            top_n=
                top_visual_images,

            prefer_label_diversity=
                True
        )
    )


    coverage_row = (
        business_evidence_coverage.loc[
            business_evidence_coverage[
                "business_id"
            ].astype(str)
            ==
            business_id
        ]
        .iloc[0]
    )


    profile = {

        "business_id":
            business_id,

        "name":
            identity[
                "name"
            ],

        "location":
            {
                "address":
                    identity[
                        "address"
                    ],

                "city":
                    identity[
                        "city"
                    ],

                "state":
                    identity[
                        "state"
                    ]
            },

        "categories":
            structured[
                "categories"
            ],

        "structured_attributes":
            structured[
                "attributes"
            ],

        "representative_text_reviews":
            text_evidence,

        "has_visual_evidence":
            bool(
                coverage_row[
                    "has_visual_evidence"
                ]
            ),

        "representative_visual_images":
            visual_evidence_profile,

        "evidence_summary":
            {
                "metadata_evidence_count":
                    int(
                        coverage_row[
                            "metadata_evidence_count"
                        ]
                    ),

                "training_review_count":
                    int(
                        coverage_row[
                            "training_review_count"
                        ]
                    ),

                "top_review_similarity":
                    float(
                        coverage_row[
                            "top_review_similarity"
                        ]
                    ),

                "selected_image_count":
                    int(
                        coverage_row[
                            "selected_image_count"
                        ]
                    ),

                "label_diversity":
                    int(
                        coverage_row[
                            "label_diversity"
                        ]
                    ),

                "top_image_similarity":
                    (
                        None
                        if pd.isna(
                            coverage_row[
                                "top_image_similarity"
                            ]
                        )
                        else
                        float(
                            coverage_row[
                                "top_image_similarity"
                            ]
                        )
                    )
            }
    }


    return profile

In [175]:
gumbo_shop_profile = (
    build_business_evidence_profile(
        business_id=
            text_evidence_smoke_business_id
    )
)

In [176]:
print(
    "Business:",
    gumbo_shop_profile[
        "name"
    ]
)

print(
    "\nCategories:",
    gumbo_shop_profile[
        "categories"
    ]
)

print(
    "\nHas visual evidence:",
    gumbo_shop_profile[
        "has_visual_evidence"
    ]
)

print(
    "\nEvidence summary:",
    gumbo_shop_profile[
        "evidence_summary"
    ]
)

Business: Gumbo Shop

Categories: ['Cajun/Creole', 'Restaurants', 'Seafood']

Has visual evidence: True

Evidence summary: {'metadata_evidence_count': 36, 'training_review_count': 568, 'top_review_similarity': 0.9603865742683411, 'selected_image_count': 5, 'label_diversity': 4, 'top_image_similarity': 0.8237102031707764}


In [178]:
pd.DataFrame(
    gumbo_shop_profile[
        "structured_attributes"
    ]
)

,evidence_family,evidence_label,value
0,Attribute,Alcohol,full_bar
1,Attribute,BikeParking,true
2,Attribute,Caters,true
3,Attribute,GoodForKids,true
4,Attribute,NoiseLevel,average
5,Attribute,RestaurantsAttire,casual
6,Attribute,RestaurantsGoodForGroups,true
7,Attribute,RestaurantsPriceRange2,2


In [179]:
gumbo_shop_profile[
    "representative_text_reviews"
]

,review_id,date,stars,useful,text_representativeness,text_excerpt
0,XzHPTS118jsxLLjzH_yo0w,2016-05-06 01:39:17,4,1,0.960387,"While looking for a good spot to grab dinner, ..."
1,dllYajPmSWO4J41aMi2WGQ,2019-08-13 13:36:34,5,1,0.958656,This was the first place we went to when we go...
2,5U4MbIhj0ubbafU2aniM7A,2018-01-22 20:52:27,5,6,0.957953,Take me to the Gumbo Shop! Not only is the foo...


In [180]:
gumbo_shop_profile[
    "representative_visual_images"
]

,photo_id,photo_filename,label,caption_clean,selection_rank,visual_representativeness
0,N_Yp6SghyrXdXqliR3irRg,N_Yp6SghyrXdXqliR3irRg.jpg,drink,This is some good hot sauce,4,0.823710
1,7x5bioN1uXRwixAwmcrasA,7x5bioN1uXRwixAwmcrasA.jpg,food,Shrimp creole,2,0.823318
2,m6HkNAHWigVOie41N4Phkg,m6HkNAHWigVOie41N4Phkg.jpg,inside,Lunch,1,0.820554


In [181]:
image_less_smoke_business_id = str(
    business_evidence_coverage.loc[
        ~business_evidence_coverage[
            "has_visual_evidence"
        ],
        "business_id"
    ].iloc[0]
)

In [182]:
image_less_profile = (
    build_business_evidence_profile(
        image_less_smoke_business_id
    )
)

In [183]:
image_less_profile_check = pd.Series({

    "Has structured evidence":
        (
            len(
                image_less_profile[
                    "categories"
                ]
            )
            >
            0
        ),

    "Has text evidence":
        (
            len(
                image_less_profile[
                    "representative_text_reviews"
                ]
            )
            >
            0
        ),

    "Correctly marked image-less":
        (
            image_less_profile[
                "has_visual_evidence"
            ]
            is False
        ),

    "Visual evidence empty":
        (
            len(
                image_less_profile[
                    "representative_visual_images"
                ]
            )
            ==
            0
        ),

    "Selected image count zero":
        (
            image_less_profile[
                "evidence_summary"
            ][
                "selected_image_count"
            ]
            ==
            0
        ),

    "Top image similarity absent":
        (
            image_less_profile[
                "evidence_summary"
            ][
                "top_image_similarity"
            ]
            is None
        )
})


image_less_profile_check

Has structured evidence        True
Has text evidence              True
Correctly marked image-less    True
Visual evidence empty          True
Selected image count zero      True
Top image similarity absent    True
dtype: bool

In [184]:
k46_global_profile_check = pd.Series({

    "All businesses named":
        business_evidence_coverage[
            "name"
        ].notna().all(),

    "All have at least one KG edge":
        (
            business_evidence_coverage[
                "metadata_evidence_count"
            ]
            >=
            1
        ).all(),

    "All have at least one training review":
        (
            business_evidence_coverage[
                "training_review_count"
            ]
            >=
            1
        ).all(),

    "Visual availability internally consistent":
        (
            business_evidence_coverage[
                "has_visual_evidence"
            ]
            ==
            (
                business_evidence_coverage[
                    "selected_image_count"
                ]
                >
                0
            )
        ).all(),

    "Visual businesses have similarity":
        business_evidence_coverage.loc[
            business_evidence_coverage[
                "has_visual_evidence"
            ],
            "top_image_similarity"
        ]
        .notna()
        .all(),

    "Image-less businesses have no visual similarity":
        business_evidence_coverage.loc[
            ~business_evidence_coverage[
                "has_visual_evidence"
            ],
            "top_image_similarity"
        ]
        .isna()
        .all()
})


k46_global_profile_check

All businesses named                               True
All have at least one KG edge                      True
All have at least one training review              True
Visual availability internally consistent          True
Visual businesses have similarity                  True
Image-less businesses have no visual similarity    True
dtype: bool

In [185]:
business_evidence_coverage.to_parquet(
    explanation_dir
    /
    "k4_unified_business_evidence_coverage.parquet",

    index=False
)

In [186]:
gumbo_shop_profile[
    "representative_text_reviews"
].to_csv(
    explanation_dir
    /
    "k4_smoke_text_evidence.csv",

    index=False
)


gumbo_shop_profile[
    "representative_visual_images"
].to_csv(
    explanation_dir
    /
    "k4_smoke_visual_evidence.csv",

    index=False
)


pd.DataFrame(
    gumbo_shop_profile[
        "structured_attributes"
    ]
).to_csv(
    explanation_dir
    /
    "k4_smoke_structured_evidence.csv",

    index=False
)

In [187]:
k46_save_check = pd.Series({

    "Unified evidence coverage saved":
        (
            explanation_dir
            /
            "k4_unified_business_evidence_coverage.parquet"
        ).exists(),

    "Smoke text evidence saved":
        (
            explanation_dir
            /
            "k4_smoke_text_evidence.csv"
        ).exists(),

    "Smoke visual evidence saved":
        (
            explanation_dir
            /
            "k4_smoke_visual_evidence.csv"
        ).exists(),

    "Smoke structured evidence saved":
        (
            explanation_dir
            /
            "k4_smoke_structured_evidence.csv"
        ).exists()
})


k46_save_check

Unified evidence coverage saved    True
Smoke text evidence saved          True
Smoke visual evidence saved        True
Smoke structured evidence saved    True
dtype: bool

## K5 — Recommendation-Level Case Explanations

Recommendation-level explanation cases are selected systematically from the
frozen test results rather than manually chosen after inspecting individual
businesses.

Five complementary recommendation conditions are considered:

1. a visual-supported case where KGRec-MM moves the held-out target into the
   top-20 recommendation set;

2. a visually diverse target showing a top-20 multimodal gain, reflecting the
   visual-diversity pattern identified in RQ3;

3. a counterexample where visual information is available but KGRec-MM ranks
   the held-out target worse than KGRec-NV;

4. an image-less target exhibiting the competitive missing-modality behaviour
   identified through frozen-model perturbation;

5. a sparse-history user for whom multimodal recommendation moves the held-out
   target into the top-20 set.

Within each eligible group, the case whose relevant effect is closest to the
group median is selected. This avoids deliberately choosing the most extreme
positive or negative observation.

These cases are illustrative model-behaviour examples and are not intended to
provide population-level statistical evidence independently of the aggregate
evaluation.

In [188]:
# --------------------------------------------------
# K5.1 — Systematic recommendation-case selection
# --------------------------------------------------

k5_cases = (
    rq3_analysis
    .copy()
)

In [189]:
k5_cases[
    "mm_enters_top10"
] = (
    (k5_cases["nv_rank"] > 10)
    &
    (k5_cases["mm_rank"] <= 10)
)


k5_cases[
    "mm_enters_top20"
] = (
    (k5_cases["nv_rank"] > 20)
    &
    (k5_cases["mm_rank"] <= 20)
)


k5_cases[
    "mm_worsens_vs_nv"
] = (
    k5_cases[
        "mm_rank"
    ]
    >
    k5_cases[
        "nv_rank"
    ]
)


k5_cases[
    "competitive_visual_rank_effect"
] = (
    k5_cases[
        "mm_all_visual_off_rank"
    ]
    -
    k5_cases[
        "mm_rank"
    ]
)


k5_cases[
    "sparse_user"
] = (
    k5_cases[
        "user_training_interaction_count"
    ]
    .between(
        3,
        5
    )
)

In [190]:
k5_cases = (
    k5_cases
    .merge(
        business_evidence_coverage[
            [
                "business_id",
                "name",
                "categories",
                "metadata_evidence_count",
                "training_review_count",
                "selected_image_count",
                "label_diversity",
                "top_review_similarity",
                "top_image_similarity"
            ]
        ],

        on="business_id",

        how="left",

        validate="many_to_one",

        suffixes=(
            "",
            "_evidence"
        )
    )
)

In [191]:
k51_base_check = pd.Series({

    "Rows":
        len(
            k5_cases
        ),

    "Expected test users":
        (
            len(
                k5_cases
            )
            ==
            14991
        ),

    "Unique users":
        k5_cases[
            "user_row"
        ].nunique(),

    "All businesses named":
        k5_cases[
            "name"
        ].notna().all(),

    "All have text evidence":
        k5_cases[
            "training_review_count_evidence"
        ].notna().all()
        if "training_review_count_evidence"
        in k5_cases.columns
        else
        k5_cases[
            "training_review_count"
        ].notna().all()
})


k51_base_check

Rows                      14991
Expected test users        True
Unique users              14991
All businesses named       True
All have text evidence     True
dtype: object

In [192]:
def select_median_case(
    candidates,
    effect_column,
    case_type,
    used_user_rows=None
):

    candidates = (
        candidates
        .copy()
    )


    if used_user_rows is not None:

        candidates = (
            candidates.loc[
                ~candidates[
                    "user_row"
                ].isin(
                    used_user_rows
                )
            ]
            .copy()
        )


    if len(
        candidates
    ) == 0:

        raise ValueError(
            f"No eligible candidates for {case_type}"
        )


    median_effect = (
        candidates[
            effect_column
        ]
        .median()
    )


    candidates[
        "_median_distance"
    ] = (
        candidates[
            effect_column
        ]
        -
        median_effect
    ).abs()


    selected = (
        candidates
        .sort_values(
            [
                "_median_distance",
                "user_row",
                "business_id"
            ]
        )
        .iloc[0]
        .copy()
    )


    selected[
        "case_type"
    ] = case_type


    selected[
        "candidate_pool_size"
    ] = len(
        candidates
    )


    selected[
        "candidate_median_effect"
    ] = median_effect


    return selected

In [193]:
case_a_candidates = (
    k5_cases.loc[
        (
            k5_cases[
                "target_has_visual"
            ].astype(bool)
        )
        &
        (
            k5_cases[
                "mm_enters_top20"
            ]
        )
    ]
    .copy()
)

In [220]:
case_b_candidates = (
    k5_cases.loc[
        (
            k5_cases[
                "target_has_visual"
            ].astype(bool)
        )
        &
        (
            k5_cases[
                "unique_image_labels"
            ]
            >=
            4
        )
        &
        (
            k5_cases[
                "mm_enters_top20"
            ]
        )
    ]
    .copy()
)

In [221]:
case_c_candidates = (
    k5_cases.loc[
        (
            k5_cases[
                "target_has_visual"
            ].astype(bool)
        )
        &
        (
            k5_cases[
                "mm_worsens_vs_nv"
            ]
        )
        &
        (
            k5_cases[
                "nv_rank"
            ]
            <=
            100
        )
    ]
    .copy()
)

In [222]:
case_d_candidates = (
    k5_cases.loc[
        (
            ~k5_cases[
                "target_has_visual"
            ].astype(bool)
        )
        &
        (
            k5_cases[
                "competitive_visual_rank_effect"
            ]
            <
            0
        )
    ]
    .copy()
)

In [223]:
case_e_candidates = (
    k5_cases.loc[
        (
            k5_cases[
                "sparse_user"
            ]
        )
        &
        (
            k5_cases[
                "target_has_visual"
            ].astype(bool)
        )
        &
        (
            k5_cases[
                "mm_enters_top20"
            ]
        )
    ]
    .copy()
)

In [224]:
k51_candidate_pool_summary = pd.Series({

    "A — Visual Top-20 Gain":
        len(
            case_a_candidates
        ),

    "B — High-Diversity Top-20 Gain":
        len(
            case_b_candidates
        ),

    "C — Visual Counterexample":
        len(
            case_c_candidates
        ),

    "D — Image-less Competitive Penalty":
        len(
            case_d_candidates
        ),

    "E — Sparse-User Top-20 Gain":
        len(
            case_e_candidates
        )
})


k51_candidate_pool_summary

A — Visual Top-20 Gain                 707
B — High-Diversity Top-20 Gain         666
C — Visual Counterexample             2311
D — Image-less Competitive Penalty    1723
E — Sparse-User Top-20 Gain            489
dtype: int64

In [225]:
selected_cases = []

used_user_rows = set()

In [226]:
case_a = select_median_case(
    candidates=
        case_a_candidates,

    effect_column=
        "rank_improvement",

    case_type=
        "A — Visual Top-20 Gain",

    used_user_rows=
        used_user_rows
)


selected_cases.append(
    case_a
)


used_user_rows.add(
    case_a[
        "user_row"
    ]
)

In [227]:
case_b = select_median_case(
    candidates=
        case_b_candidates,

    effect_column=
        "rank_improvement",

    case_type=
        "B — High-Diversity Visual Gain",

    used_user_rows=
        used_user_rows
)


selected_cases.append(
    case_b
)


used_user_rows.add(
    case_b[
        "user_row"
    ]
)

In [228]:
case_c = select_median_case(
    candidates=
        case_c_candidates,

    effect_column=
        "rank_improvement",

    case_type=
        "C — Visual Counterexample",

    used_user_rows=
        used_user_rows
)


selected_cases.append(
    case_c
)


used_user_rows.add(
    case_c[
        "user_row"
    ]
)

In [229]:
case_d = select_median_case(
    candidates=
        case_d_candidates,

    effect_column=
        "competitive_visual_rank_effect",

    case_type=
        "D — Image-less Competitive Penalty",

    used_user_rows=
        used_user_rows
)


selected_cases.append(
    case_d
)


used_user_rows.add(
    case_d[
        "user_row"
    ]
)

In [230]:
case_e = select_median_case(
    candidates=
        case_e_candidates,

    effect_column=
        "rank_improvement",

    case_type=
        "E — Sparse-User Visual Gain",

    used_user_rows=
        used_user_rows
)


selected_cases.append(
    case_e
)

In [231]:
selected_case_table = pd.DataFrame(
    selected_cases
)

In [232]:
selected_case_summary = (
    selected_case_table[
        [
            "case_type",
            "candidate_pool_size",
            "candidate_median_effect",
            "user_row",
            "business_id",
            "name",
            "target_has_visual",
            "nv_rank",
            "mm_rank",
            "rank_improvement",
            "user_training_interaction_count",
            "training_interaction_count",
            "metadata_edge_count",
            "image_count",
            "unique_image_labels",
            "target_visual_score_gain",
            "target_visual_rank_gain",
            "competitive_visual_rank_effect"
        ]
    ]
    .copy()
)

In [233]:
selected_case_summary

,case_type,candidate_pool_size,candidate_median_effect,user_row,business_id,name,target_has_visual,nv_rank,mm_rank,rank_improvement,user_training_interaction_count,training_interaction_count,metadata_edge_count,image_count,unique_image_labels,target_visual_score_gain,target_visual_rank_gain,competitive_visual_rank_effect
1483,A — Visual Top-20 Gain,707,38.0,1483,iSRTaT9WngzB8JJ2YKJUig,Mother's Restaurant,True,56,18,38,5,601,54,5,3,0.722194,635,11
3279,B — High-Diversity Visual Gain,666,38.0,3279,ZTctPm8-lBy0iJ9dFhYhyQ,Herbsaint,True,44,6,38,10,216,40,5,4,0.930911,762,141
672,C — Visual Counterexample,2311,-30.0,672,u7uFQCoHFtBKCtbWUm6yZw,Emeril's,True,19,49,-30,3,288,35,5,3,0.756464,622,-32
7806,D — Image-less Competitive Penalty,1723,-578.0,7806,oOSMOJptLRbmEDjp0YnEPw,Witches Brew Tours,False,542,759,-217,11,57,10,0,0,0.000000,0,-578
3072,E — Sparse-User Visual Gain,488,39.0,3072,oBNrLz4EDhiscSlbOl8uAw,Ruby Slipper - New Orleans,True,52,13,39,5,676,44,5,5,0.857512,713,12


In [234]:
k51_selection_check = pd.Series({

    "Five cases selected":
        (
            len(
                selected_case_summary
            )
            ==
            5
        ),

    "Five unique users":
        (
            selected_case_summary[
                "user_row"
            ].nunique()
            ==
            5
        ),

    "All businesses named":
        selected_case_summary[
            "name"
        ].notna().all(),

    "Case A enters Top-20":
        bool(
            (
                case_a[
                    "nv_rank"
                ]
                >
                20
            )
            and
            (
                case_a[
                    "mm_rank"
                ]
                <=
                20
            )
        ),

    "Case B high visual diversity":
        bool(
            case_b[
                "unique_image_labels"
            ]
            >=
            4
        ),

    "Case C MM worse than NV":
        bool(
            case_c[
                "mm_rank"
            ]
            >
            case_c[
                "nv_rank"
            ]
        ),

    "Case D image-less":
        bool(
            not bool(
                case_d[
                    "target_has_visual"
                ]
            )
        ),

    "Case D competitive penalty":
        bool(
            case_d[
                "competitive_visual_rank_effect"
            ]
            <
            0
        ),

    "Case E sparse history":
        bool(
            3
            <=
            case_e[
                "user_training_interaction_count"
            ]
            <=
            5
        ),

    "Case E enters Top-20":
        bool(
            (
                case_e[
                    "nv_rank"
                ]
                >
                20
            )
            and
            (
                case_e[
                    "mm_rank"
                ]
                <=
                20
            )
        )
})


k51_selection_check

Five cases selected             True
Five unique users               True
All businesses named            True
Case A enters Top-20            True
Case B high visual diversity    True
Case C MM worse than NV         True
Case D image-less               True
Case D competitive penalty      True
Case E sparse history           True
Case E enters Top-20            True
dtype: bool

In [235]:
k51_case_protocol = pd.DataFrame([
    {
        "case_type":
            "A — Visual Top-20 Gain",

        "eligibility":
            (
                "Visual-supported target; NV rank > 20; "
                "MM rank <= 20"
            ),

        "selection_rule":
            (
                "Case closest to median rank improvement "
                "within eligible pool"
            ),

        "purpose":
            (
                "Illustrate a concrete multimodal top-20 gain"
            )
    },

    {
        "case_type":
            "B — High-Diversity Visual Gain",

        "eligibility":
            (
                "Visual-supported target; >=4 image labels; "
                "NV rank > 20; MM rank <= 20"
            ),

        "selection_rule":
            (
                "Case closest to median rank improvement "
                "within eligible pool"
            ),

        "purpose":
            (
                "Connect case evidence to RQ3 visual-diversity finding"
            )
    },

    {
        "case_type":
            "C — Visual Counterexample",

        "eligibility":
            (
                "Visual-supported target; NV rank <=100; "
                "MM rank worse than NV"
            ),

        "selection_rule":
            (
                "Case closest to median rank change "
                "within eligible pool"
            ),

        "purpose":
            (
                "Show that visual information does not "
                "guarantee improved ranking"
            )
    },

    {
        "case_type":
            "D — Image-less Competitive Penalty",

        "eligibility":
            (
                "Image-less target; rank worsens when "
                "competitors retain visual evidence"
            ),

        "selection_rule":
            (
                "Case closest to median competitive "
                "rank effect"
            ),

        "purpose":
            (
                "Illustrate competitive missing-modality behaviour"
            )
    },

    {
        "case_type":
            "E — Sparse-User Visual Gain",

        "eligibility":
            (
                "3–5 training interactions; visual target; "
                "NV rank >20; MM rank <=20"
            ),

        "selection_rule":
            (
                "Case closest to median rank improvement "
                "within eligible pool"
            ),

        "purpose":
            (
                "Connect case evidence to RQ3 user-history heterogeneity"
            )
    }
])


k51_case_protocol

,case_type,eligibility,selection_rule,purpose
0,A — Visual Top-20 Gain,Visual-supported target; NV rank > 20; MM rank...,Case closest to median rank improvement within...,Illustrate a concrete multimodal top-20 gain
1,B — High-Diversity Visual Gain,Visual-supported target; >=4 image labels; NV ...,Case closest to median rank improvement within...,Connect case evidence to RQ3 visual-diversity ...
2,C — Visual Counterexample,Visual-supported target; NV rank <=100; MM ran...,Case closest to median rank change within elig...,Show that visual information does not guarante...
3,D — Image-less Competitive Penalty,Image-less target; rank worsens when competito...,Case closest to median competitive rank effect,Illustrate competitive missing-modality behaviour
4,E — Sparse-User Visual Gain,3–5 training interactions; visual target; NV r...,Case closest to median rank improvement within...,Connect case evidence to RQ3 user-history hete...


In [236]:
selected_case_summary.to_csv(
    explanation_dir
    /
    "k5_selected_recommendation_cases.csv",

    index=False
)


k51_case_protocol.to_csv(
    explanation_dir
    /
    "k5_case_selection_protocol.csv",

    index=False
)

In [237]:
k51_save_check = pd.Series({

    "Selected cases saved":
        (
            explanation_dir
            /
            "k5_selected_recommendation_cases.csv"
        ).exists(),

    "Case protocol saved":
        (
            explanation_dir
            /
            "k5_case_selection_protocol.csv"
        ).exists()
})


k51_save_check

Selected cases saved    True
Case protocol saved     True
dtype: bool

### K5.2 — Personalised User–Target Evidence Assembly

Business-level evidence alone does not explain the personalised context of a
recommendation. User representations in the recommender are learned from the
training interaction graph, making historical user-business interactions an
important source of recommendation context.

For each selected case, the target business's frozen KG categories are compared
with the categories of businesses appearing in that user's training history.
The analysis records category overlap frequencies and identifies previously
interacted businesses sharing those categories.

This overlap is interpreted as human-readable interaction and knowledge-graph
evidence associated with the recommendation. It is not treated as a learned
attention weight, explicit reasoning path, or causal attribution.

All user-history evidence is restricted to the frozen training interaction
split.

In [238]:
# --------------------------------------------------
# K5.2 — User-row to external user-ID alignment
# --------------------------------------------------

k52_user_index = (
    user_index
    .reset_index(drop=True)
    .copy()
)


if "user_row" not in k52_user_index.columns:

    k52_user_index[
        "user_row"
    ] = np.arange(
        len(k52_user_index)
    )

In [239]:
k52_user_index_check = pd.Series({

    "Rows":
        len(k52_user_index),

    "Expected 14991":
        (
            len(k52_user_index)
            ==
            14991
        ),

    "user_id present":
        (
            "user_id"
            in k52_user_index.columns
        ),

    "user_row unique":
        k52_user_index[
            "user_row"
        ].is_unique,

    "user_id unique":
        k52_user_index[
            "user_id"
        ].is_unique,

    "user rows cover 0..14990":
        (
            set(
                k52_user_index[
                    "user_row"
                ].astype(int)
            )
            ==
            set(
                range(14991)
            )
        )
})


k52_user_index_check

Rows                        14991
Expected 14991               True
user_id present              True
user_row unique              True
user_id unique               True
user rows cover 0..14990     True
dtype: object

In [240]:
user_id_by_row = dict(
    zip(
        k52_user_index[
            "user_row"
        ].astype(int),

        k52_user_index[
            "user_id"
        ].astype(str)
    )
)

In [241]:
kg_category_evidence = (
    structured_evidence.loc[
        structured_evidence[
            "evidence_family"
        ]
        ==
        "Category",
        [
            "business_id",
            "value"
        ]
    ]
    .rename(
        columns={
            "value":
                "category"
        }
    )
    .drop_duplicates()
    .copy()
)

In [242]:
k52_category_check = pd.Series({

    "Category edges":
        len(
            kg_category_evidence
        ),

    "Expected 12203":
        (
            len(
                kg_category_evidence
            )
            ==
            12203
        ),

    "Businesses represented":
        kg_category_evidence[
            "business_id"
        ].nunique(),

    "Expected 2516 businesses":
        (
            kg_category_evidence[
                "business_id"
            ].nunique()
            ==
            2516
        ),

    "No duplicate business-category pairs":
        (
            ~kg_category_evidence.duplicated(
                subset=[
                    "business_id",
                    "category"
                ]
            ).any()
        )
})


k52_category_check

Category edges                          12203
Expected 12203                           True
Businesses represented                   2516
Expected 2516 businesses                 True
No duplicate business-category pairs     True
dtype: object

In [243]:
training_user_business = (
    training_interactions[
        [
            "user_id",
            "business_id"
        ]
    ]
    .astype(str)
    .drop_duplicates()
    .copy()
)

In [244]:
training_user_business = (
    training_user_business
    .merge(
        business_display[
            [
                "business_id",
                "name"
            ]
        ].astype({
            "business_id":
                str
        }),

        on="business_id",

        how="left",

        validate="many_to_one"
    )
)

In [245]:
k52_training_history_check = pd.Series({

    "Unique user-business pairs":
        len(
            training_user_business
        ),

    "Users":
        training_user_business[
            "user_id"
        ].nunique(),

    "Businesses":
        training_user_business[
            "business_id"
        ].nunique(),

    "Expected users":
        (
            training_user_business[
                "user_id"
            ].nunique()
            ==
            14991
        ),

    "Expected businesses":
        (
            training_user_business[
                "business_id"
            ].nunique()
            ==
            2516
        ),

    "All business names recovered":
        training_user_business[
            "name"
        ].notna().all()
})


k52_training_history_check

Unique user-business pairs      122233
Users                            14991
Businesses                        2516
Expected users                    True
Expected businesses               True
All business names recovered      True
dtype: object

In [246]:
user_category_history = (
    training_user_business
    .merge(
        kg_category_evidence,

        on="business_id",

        how="left",

        validate="many_to_many"
    )
)

In [247]:
user_category_counts = (
    user_category_history
    .groupby(
        [
            "user_id",
            "category"
        ],
        as_index=False
    )
    .agg(
        historical_business_count=(
            "business_id",
            "nunique"
        )
    )
)

In [248]:
k52_user_category_check = pd.Series({

    "Rows":
        len(
            user_category_counts
        ),

    "Users represented":
        user_category_counts[
            "user_id"
        ].nunique(),

    "Expected 14991 users":
        (
            user_category_counts[
                "user_id"
            ].nunique()
            ==
            14991
        ),

    "No missing categories":
        user_category_counts[
            "category"
        ].notna().all(),

    "All counts positive":
        (
            user_category_counts[
                "historical_business_count"
            ]
            >
            0
        ).all()
})


k52_user_category_check

Rows                     325421
Users represented         14991
Expected 14991 users       True
No missing categories      True
All counts positive        True
dtype: object

In [249]:
def get_user_target_category_evidence(
    user_id,
    target_business_id
):

    user_id = str(
        user_id
    )

    target_business_id = str(
        target_business_id
    )


    target_categories = (
        kg_category_evidence.loc[
            kg_category_evidence[
                "business_id"
            ].astype(str)
            ==
            target_business_id,
            "category"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


    category_overlap = (
        user_category_counts.loc[
            (
                user_category_counts[
                    "user_id"
                ].astype(str)
                ==
                user_id
            )
            &
            (
                user_category_counts[
                    "category"
                ].isin(
                    target_categories
                )
            )
        ]
        .copy()
    )


    # Add target categories absent from history
    existing_categories = set(
        category_overlap[
            "category"
        ]
    )


    missing_categories = [
        category
        for category
        in target_categories
        if category not in existing_categories
    ]


    if len(
        missing_categories
    ) > 0:

        missing_rows = pd.DataFrame({

            "user_id":
                user_id,

            "category":
                missing_categories,

            "historical_business_count":
                0
        })


        category_overlap = pd.concat(
            [
                category_overlap,
                missing_rows
            ],
            ignore_index=True
        )


    category_overlap = (
        category_overlap
        .sort_values(
            [
                "historical_business_count",
                "category"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return category_overlap

In [250]:
def get_user_target_matching_history(
    user_id,
    target_business_id,
    top_n=5
):

    user_id = str(
        user_id
    )

    target_business_id = str(
        target_business_id
    )


    target_categories = set(
        kg_category_evidence.loc[
            kg_category_evidence[
                "business_id"
            ].astype(str)
            ==
            target_business_id,
            "category"
        ]
    )


    user_history = (
        training_user_business.loc[
            training_user_business[
                "user_id"
            ].astype(str)
            ==
            user_id
        ]
        .copy()
    )


    history_categories = (
        user_history
        .merge(
            kg_category_evidence,

            on="business_id",

            how="left",

            validate="many_to_many"
        )
    )


    history_categories = (
        history_categories.loc[
            history_categories[
                "category"
            ].isin(
                target_categories
            )
        ]
        .copy()
    )


    if len(
        history_categories
    ) == 0:

        return pd.DataFrame(
            columns=[
                "business_id",
                "name",
                "shared_category_count",
                "shared_categories"
            ]
        )


    matching_history = (
        history_categories
        .groupby(
            [
                "business_id",
                "name"
            ],
            as_index=False
        )
        .agg(
            shared_category_count=(
                "category",
                "nunique"
            ),

            shared_categories=(
                "category",
                lambda values:
                    ", ".join(
                        sorted(
                            set(values)
                        )
                    )
            )
        )
        .sort_values(
            [
                "shared_category_count",
                "name",
                "business_id"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .head(
            top_n
        )
        .reset_index(
            drop=True
        )
    )


    return matching_history

In [251]:
case_behaviour_rows = []

case_category_overlap_frames = []

case_history_match_frames = []

case_structured_frames = []

case_text_frames = []

case_visual_frames = []

In [252]:
for _, case in selected_case_summary.iterrows():

    case_type = str(
        case[
            "case_type"
        ]
    )


    user_row = int(
        case[
            "user_row"
        ]
    )


    user_id = (
        user_id_by_row[
            user_row
        ]
    )


    business_id = str(
        case[
            "business_id"
        ]
    )


    business_profile = (
        build_business_evidence_profile(
            business_id=
                business_id,

            top_text_reviews=
                3,

            top_visual_images=
                3,

            max_structured_attributes=
                8
        )
    )


    # ---------------------------------------------
    # Personalised category evidence
    # ---------------------------------------------

    category_overlap = (
        get_user_target_category_evidence(
            user_id=
                user_id,

            target_business_id=
                business_id
        )
    )


    category_overlap.insert(
        0,
        "case_type",
        case_type
    )


    category_overlap.insert(
        1,
        "business_id",
        business_id
    )


    case_category_overlap_frames.append(
        category_overlap
    )


    # ---------------------------------------------
    # Concrete matching historical businesses
    # ---------------------------------------------

    matching_history = (
        get_user_target_matching_history(
            user_id=
                user_id,

            target_business_id=
                business_id,

            top_n=
                5
        )
    )


    matching_history.insert(
        0,
        "case_type",
        case_type
    )


    matching_history.insert(
        1,
        "user_id",
        user_id
    )


    matching_history.insert(
        2,
        "target_business_id",
        business_id
    )


    case_history_match_frames.append(
        matching_history
    )


    # ---------------------------------------------
    # Structured evidence
    # ---------------------------------------------

    structured_case = pd.DataFrame(
        business_profile[
            "structured_attributes"
        ]
    )


    structured_case.insert(
        0,
        "case_type",
        case_type
    )


    structured_case.insert(
        1,
        "business_id",
        business_id
    )


    structured_case.insert(
        2,
        "business_name",
        business_profile[
            "name"
        ]
    )


    case_structured_frames.append(
        structured_case
    )


    # ---------------------------------------------
    # Text evidence
    # ---------------------------------------------

    text_case = (
        business_profile[
            "representative_text_reviews"
        ]
        .copy()
    )


    text_case.insert(
        0,
        "case_type",
        case_type
    )


    text_case.insert(
        1,
        "business_id",
        business_id
    )


    text_case.insert(
        2,
        "business_name",
        business_profile[
            "name"
        ]
    )


    case_text_frames.append(
        text_case
    )


    # ---------------------------------------------
    # Visual evidence
    # ---------------------------------------------

    visual_case = (
        business_profile[
            "representative_visual_images"
        ]
        .copy()
    )


    if len(
        visual_case
    ) > 0:

        visual_case.insert(
            0,
            "case_type",
            case_type
        )


        visual_case.insert(
            1,
            "business_id",
            business_id
        )


        visual_case.insert(
            2,
            "business_name",
            business_profile[
                "name"
        ])


        case_visual_frames.append(
            visual_case
        )


    # ---------------------------------------------
    # Flat behavioural summary
    # ---------------------------------------------

    actual_training_history_count = (
        training_user_business.loc[
            training_user_business[
                "user_id"
            ].astype(str)
            ==
            user_id,
            "business_id"
        ]
        .nunique()
    )


    supported_target_categories = int(
        (
            category_overlap[
                "historical_business_count"
            ]
            >
            0
        ).sum()
    )


    total_target_categories = int(
        len(
            category_overlap
        )
    )


    case_behaviour_rows.append({

        "case_type":
            case_type,

        "user_row":
            user_row,

        "user_id":
            user_id,

        "business_id":
            business_id,

        "business_name":
            business_profile[
                "name"
            ],

        "nv_rank":
            int(
                case[
                    "nv_rank"
                ]
            ),

        "mm_rank":
            int(
                case[
                    "mm_rank"
                ]
            ),

        "rank_improvement":
            int(
                case[
                    "rank_improvement"
                ]
            ),

        "training_history_count":
            int(
                actual_training_history_count
            ),

        "target_category_count":
            total_target_categories,

        "target_categories_supported_by_history":
            supported_target_categories,

        "target_category_support_fraction":
            (
                supported_target_categories
                /
                total_target_categories
                if total_target_categories > 0
                else np.nan
            ),

        "target_has_visual":
            bool(
                case[
                    "target_has_visual"
                ]
            ),

        "selected_image_count":
            int(
                business_profile[
                    "evidence_summary"
                ][
                    "selected_image_count"
                ]
            ),

        "label_diversity":
            int(
                business_profile[
                    "evidence_summary"
                ][
                    "label_diversity"
                ]
            ),

        "target_visual_score_gain":
            (
                float(
                    case[
                        "target_visual_score_gain"
                    ]
                )
                if bool(
                    case[
                        "target_has_visual"
                    ]
                )
                else np.nan
            ),

        "target_visual_rank_gain":
            (
                float(
                    case[
                        "target_visual_rank_gain"
                    ]
                )
                if bool(
                    case[
                        "target_has_visual"
                    ]
                )
                else np.nan
            ),

        "competitive_visual_rank_effect":
            (
                float(
                    case[
                        "competitive_visual_rank_effect"
                    ]
                )
                if not bool(
                    case[
                        "target_has_visual"
                    ]
                )
                else np.nan
            )
    })

In [253]:
k52_case_behaviour = pd.DataFrame(
    case_behaviour_rows
)


k52_case_category_overlap = pd.concat(
    case_category_overlap_frames,
    ignore_index=True
)


k52_case_history_matches = pd.concat(
    case_history_match_frames,
    ignore_index=True
)


k52_case_structured_evidence = pd.concat(
    case_structured_frames,
    ignore_index=True
)


k52_case_text_evidence = pd.concat(
    case_text_frames,
    ignore_index=True
)


k52_case_visual_evidence = pd.concat(
    case_visual_frames,
    ignore_index=True
)

In [254]:
k52_case_behaviour

,case_type,user_row,user_id,business_id,business_name,nv_rank,mm_rank,rank_improvement,training_history_count,target_category_count,target_categories_supported_by_history,target_category_support_fraction,target_has_visual,selected_image_count,label_diversity,target_visual_score_gain,target_visual_rank_gain,competitive_visual_rank_effect
0,A — Visual Top-20 Gain,1483,5Nl6vX1LSbd3K1KBkOVtxQ,iSRTaT9WngzB8JJ2YKJUig,Mother's Restaurant,56,18,38,5,12,6,0.500000,True,5,3,0.722194,635.0,NaN
1,B — High-Diversity Visual Gain,3279,D76EFdD5F7XoIIyQmWTQhA,ZTctPm8-lBy0iJ9dFhYhyQ,Herbsaint,44,6,38,10,3,3,1.000000,True,5,4,0.930911,762.0,NaN
2,C — Visual Counterexample,672,1j0ef7b89BUBByKc5BNWDg,u7uFQCoHFtBKCtbWUm6yZw,Emeril's,19,49,-30,3,4,4,1.000000,True,5,3,0.756464,622.0,NaN
3,D — Image-less Competitive Penalty,7806,WZUeLyOo2Y_GnlEOLMUjpg,oOSMOJptLRbmEDjp0YnEPw,Witches Brew Tours,542,759,-217,11,7,1,0.142857,False,0,0,NaN,NaN,-578.0
4,E — Sparse-User Visual Gain,3072,CJ8yYeLi1qaBbxc5xezyaA,oBNrLz4EDhiscSlbOl8uAw,Ruby Slipper - New Orleans,52,13,39,5,5,3,0.600000,True,5,5,0.857512,713.0,NaN


In [255]:
k52_case_category_overlap

,case_type,business_id,user_id,category,historical_business_count
0,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Cajun/Creole,4
1,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Restaurants,4
2,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Southern,2
3,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,American (New),1
4,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Breakfast & Brunch,1
5,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Food,1
6,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Caterers,0
7,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Ethnic Food,0
8,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Event Planning & Services,0
9,A — Visual Top-20 Gain,iSRTaT9WngzB8JJ2YKJUig,5Nl6vX1LSbd3K1KBkOVtxQ,Sandwiches,0


In [256]:
k52_case_history_matches

,case_type,user_id,target_business_id,business_id,name,shared_category_count,shared_categories
0,A — Visual Top-20 Gain,5Nl6vX1LSbd3K1KBkOVtxQ,iSRTaT9WngzB8JJ2YKJUig,oe-iacnzvyHV8zxaNqCf7A,Coterie Restaurant & Oyster Bar,4,"Breakfast & Brunch, Cajun/Creole, Restaurants,..."
1,A — Visual Top-20 Gain,5Nl6vX1LSbd3K1KBkOVtxQ,iSRTaT9WngzB8JJ2YKJUig,NHdE_ObFj7OjQBB3jjqBNQ,Lula Restaurant Distillery,4,"Cajun/Creole, Food, Restaurants, Southern"
2,A — Visual Top-20 Gain,5Nl6vX1LSbd3K1KBkOVtxQ,iSRTaT9WngzB8JJ2YKJUig,eCikDKFbaeYVNdCD2beX0Q,Dat Dog,3,"American (New), Cajun/Creole, Restaurants"
3,A — Visual Top-20 Gain,5Nl6vX1LSbd3K1KBkOVtxQ,iSRTaT9WngzB8JJ2YKJUig,TVDe34aKHkjaEOtCqffwbw,BB King's Blues Club,2,"Cajun/Creole, Restaurants"
4,B — High-Diversity Visual Gain,D76EFdD5F7XoIIyQmWTQhA,ZTctPm8-lBy0iJ9dFhYhyQ,_C7QiQQc47AOEv4PE3Kong,Commander's Palace,2,"French, Restaurants"
5,B — High-Diversity Visual Gain,D76EFdD5F7XoIIyQmWTQhA,ZTctPm8-lBy0iJ9dFhYhyQ,mhrW9O0O5hXGXGnEYBVoag,Jacques-Imo's Cafe,2,"Restaurants, Seafood"
6,B — High-Diversity Visual Gain,D76EFdD5F7XoIIyQmWTQhA,ZTctPm8-lBy0iJ9dFhYhyQ,8KnMSrMuTI2AtHZ2LOI8OQ,Maurepas Foods,2,"Restaurants, Seafood"
7,B — High-Diversity Visual Gain,D76EFdD5F7XoIIyQmWTQhA,ZTctPm8-lBy0iJ9dFhYhyQ,JhpRI9m71ybWHkAqRdL0Tg,The Avenue Pub,2,"French, Restaurants"
8,B — High-Diversity Visual Gain,D76EFdD5F7XoIIyQmWTQhA,ZTctPm8-lBy0iJ9dFhYhyQ,kdqoDMuvyNedsouc1i33vQ,Araña Taqueria y Cantina,1,Restaurants
9,C — Visual Counterexample,1j0ef7b89BUBByKc5BNWDg,u7uFQCoHFtBKCtbWUm6yZw,GBTPC53ZrG1ZBY3DT8Mbcw,Luke,4,"American (New), Cajun/Creole, Restaurants, Sea..."


In [257]:
k52_history_count_check = (
    k52_case_behaviour[
        [
            "case_type",
            "user_row",
            "training_history_count"
        ]
    ]
    .merge(
        selected_case_summary[
            [
                "case_type",
                "user_row",
                "user_training_interaction_count"
            ]
        ],

        on=[
            "case_type",
            "user_row"
        ],

        how="left",

        validate="one_to_one"
    )
)

In [258]:
k52_history_count_check[
    "counts_identical"
] = (
    k52_history_count_check[
        "training_history_count"
    ]
    ==
    k52_history_count_check[
        "user_training_interaction_count"
    ]
)


k52_history_count_check

,case_type,user_row,training_history_count,user_training_interaction_count,counts_identical
0,A — Visual Top-20 Gain,1483,5,5,True
1,B — High-Diversity Visual Gain,3279,10,10,True
2,C — Visual Counterexample,672,3,3,True
3,D — Image-less Competitive Penalty,7806,11,11,True
4,E — Sparse-User Visual Gain,3072,5,5,True


In [259]:
k52_evidence_check = pd.Series({

    "Five recommendation cases":
        (
            len(
                k52_case_behaviour
            )
            ==
            5
        ),

    "Five unique users":
        (
            k52_case_behaviour[
                "user_id"
            ].nunique()
            ==
            5
        ),

    "History counts reproduce RQ3":
        k52_history_count_check[
            "counts_identical"
        ].all(),

    "Every case has target categories":
        (
            k52_case_behaviour[
                "target_category_count"
            ]
            >
            0
        ).all(),

    "Every case has representative text":
        (
            k52_case_text_evidence[
                "case_type"
            ].nunique()
            ==
            5
        ),

    "Four visual cases have visual evidence":
        (
            k52_case_visual_evidence[
                "case_type"
            ].nunique()
            ==
            4
        ),

    "Image-less case absent from visual evidence":
        (
            "D — Image-less Competitive Penalty"
            not in set(
                k52_case_visual_evidence[
                    "case_type"
                ]
            )
        ),

    "No category support fraction below 0":
        (
            k52_case_behaviour[
                "target_category_support_fraction"
            ]
            >=
            0
        ).all(),

    "No category support fraction above 1":
        (
            k52_case_behaviour[
                "target_category_support_fraction"
            ]
            <=
            1
        ).all()
})


k52_evidence_check

Five recommendation cases                      True
Five unique users                              True
History counts reproduce RQ3                   True
Every case has target categories               True
Every case has representative text             True
Four visual cases have visual evidence         True
Image-less case absent from visual evidence    True
No category support fraction below 0           True
No category support fraction above 1           True
dtype: bool

In [260]:
k52_case_behaviour.to_csv(
    explanation_dir
    /
    "k5_case_behaviour_summary.csv",

    index=False
)


k52_case_category_overlap.to_csv(
    explanation_dir
    /
    "k5_case_user_category_overlap.csv",

    index=False
)


k52_case_history_matches.to_csv(
    explanation_dir
    /
    "k5_case_matching_training_businesses.csv",

    index=False
)


k52_case_structured_evidence.to_csv(
    explanation_dir
    /
    "k5_case_structured_evidence.csv",

    index=False
)


k52_case_text_evidence.to_csv(
    explanation_dir
    /
    "k5_case_text_evidence.csv",

    index=False
)


k52_case_visual_evidence.to_csv(
    explanation_dir
    /
    "k5_case_visual_evidence.csv",

    index=False
)

In [261]:
k52_save_check = pd.Series({

    "Behaviour summary saved":
        (
            explanation_dir
            /
            "k5_case_behaviour_summary.csv"
        ).exists(),

    "Category overlap saved":
        (
            explanation_dir
            /
            "k5_case_user_category_overlap.csv"
        ).exists(),

    "Historical matches saved":
        (
            explanation_dir
            /
            "k5_case_matching_training_businesses.csv"
        ).exists(),

    "Structured evidence saved":
        (
            explanation_dir
            /
            "k5_case_structured_evidence.csv"
        ).exists(),

    "Text evidence saved":
        (
            explanation_dir
            /
            "k5_case_text_evidence.csv"
        ).exists(),

    "Visual evidence saved":
        (
            explanation_dir
            /
            "k5_case_visual_evidence.csv"
        ).exists()
})


k52_save_check

Behaviour summary saved      True
Category overlap saved       True
Historical matches saved     True
Structured evidence saved    True
Text evidence saved          True
Visual evidence saved        True
dtype: bool

### K5.3 — Grounded Recommendation Explanations

The verified recommendation evidence is now consolidated into structured,
human-readable case explanations.

Each explanation distinguishes between:

- recommendation outcome evidence, including KGRec-NV and KGRec-MM ranks;
- personalised knowledge evidence derived from the user's training history;
- representative textual evidence associated with the target business;
- representative visual evidence where available;
- frozen-model visual sensitivity evidence.

The resulting explanations describe evidence associated with a recommendation
rather than asserting that individual evidence items causally determined the
model prediction.

In [265]:
# --------------------------------------------------
# K5.3 — Grounded case explanation summaries
# --------------------------------------------------

k53_explanation_rows = []


for _, case in k52_case_behaviour.iterrows():

    case_type = str(
        case["case_type"]
    )

    business_id = str(
        case["business_id"]
    )

    business_name = str(
        case["business_name"]
    )


    # ---------------------------------------------
    # Category-history evidence
    # ---------------------------------------------

    category_evidence = (
        k52_case_category_overlap.loc[
            k52_case_category_overlap[
                "case_type"
            ]
            ==
            case_type
        ]
        .copy()
    )


    supported_categories = (
        category_evidence.loc[
            category_evidence[
                "historical_business_count"
            ]
            >
            0
        ]
        .sort_values(
            [
                "historical_business_count",
                "category"
            ],
            ascending=[
                False,
                True
            ]
        )
    )


    supported_category_text = ", ".join(
        [
            (
                f"{row['category']} "
                f"({int(row['historical_business_count'])})"
            )

            for _, row
            in supported_categories.iterrows()
        ]
    )


    # ---------------------------------------------
    # Strongest historical business match
    # ---------------------------------------------

    history_matches = (
        k52_case_history_matches.loc[
            k52_case_history_matches[
                "case_type"
            ]
            ==
            case_type
        ]
        .copy()
    )


    if len(history_matches) > 0:

        strongest_history = (
            history_matches
            .sort_values(
                [
                    "shared_category_count",
                    "name"
                ],
                ascending=[
                    False,
                    True
                ]
            )
            .iloc[0]
        )


        strongest_history_text = (
            f"{strongest_history['name']} "
            f"({int(strongest_history['shared_category_count'])} "
            f"shared categories: "
            f"{strongest_history['shared_categories']})"
        )

    else:

        strongest_history_text = (
            "No historical business shared a frozen-KG category."
        )


    # ---------------------------------------------
    # Representative text evidence
    # ---------------------------------------------

    text_evidence = (
        k52_case_text_evidence.loc[
            k52_case_text_evidence[
                "case_type"
            ]
            ==
            case_type
        ]
        .sort_values(
            "text_representativeness",
            ascending=False
        )
    )


    if len(text_evidence) > 0:

        top_text = (
            text_evidence.iloc[0]
        )


        text_evidence_summary = (
            str(
                top_text[
                    "text_excerpt"
                ]
            )
        )


        text_similarity = float(
            top_text[
                "text_representativeness"
            ]
        )

    else:

        text_evidence_summary = None
        text_similarity = np.nan


    # ---------------------------------------------
    # Visual evidence
    # ---------------------------------------------

    visual_evidence = (
        k52_case_visual_evidence.loc[
            k52_case_visual_evidence[
                "case_type"
            ]
            ==
            case_type
        ]
        .copy()
    )


    if len(visual_evidence) > 0:

        visual_labels = (
            visual_evidence[
                "label"
            ]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .tolist()
        )


        visual_label_text = ", ".join(
            visual_labels
        )


        top_visual_similarity = float(
            visual_evidence[
                "visual_representativeness"
            ]
            .max()
        )

    else:

        visual_label_text = (
            "Visual evidence unavailable"
        )

        top_visual_similarity = np.nan


    # ---------------------------------------------
    # Behaviour
    # ---------------------------------------------

    nv_rank = int(
        case["nv_rank"]
    )

    mm_rank = int(
        case["mm_rank"]
    )

    rank_improvement = int(
        case["rank_improvement"]
    )


    if rank_improvement > 0:

        outcome_label = (
            "MM improved target rank"
        )

    elif rank_improvement < 0:

        outcome_label = (
            "MM worsened target rank"
        )

    else:

        outcome_label = (
            "NV and MM produced the same target rank"
        )


    # ---------------------------------------------
    # Explanation record
    # ---------------------------------------------

    k53_explanation_rows.append({

        "case_type":
            case_type,

        "business_name":
            business_name,

        "nv_rank":
            nv_rank,

        "mm_rank":
            mm_rank,

        "rank_improvement":
            rank_improvement,

        "outcome":
            outcome_label,

        "training_history_count":
            int(
                case[
                    "training_history_count"
                ]
            ),

        "category_support_fraction":
            float(
                case[
                    "target_category_support_fraction"
                ]
            ),

        "supported_categories":
            supported_category_text,

        "strongest_historical_match":
            strongest_history_text,

        "representative_text_excerpt":
            text_evidence_summary,

        "representative_text_similarity":
            text_similarity,

        "visual_available":
            bool(
                case[
                    "target_has_visual"
                ]
            ),

        "representative_visual_labels":
            visual_label_text,

        "representative_visual_similarity":
            top_visual_similarity,

        "target_visual_score_sensitivity":
            case[
                "target_visual_score_gain"
            ],

        "target_visual_rank_sensitivity":
            case[
                "target_visual_rank_gain"
            ],

        "competitive_visual_rank_effect":
            case[
                "competitive_visual_rank_effect"
            ]
    })

In [266]:
k53_explanation_summary = pd.DataFrame(
    k53_explanation_rows
)


k53_explanation_summary

,case_type,business_name,nv_rank,mm_rank,rank_improvement,outcome,training_history_count,category_support_fraction,supported_categories,strongest_historical_match,representative_text_excerpt,representative_text_similarity,visual_available,representative_visual_labels,representative_visual_similarity,target_visual_score_sensitivity,target_visual_rank_sensitivity,competitive_visual_rank_effect
0,A — Visual Top-20 Gain,Mother's Restaurant,56,18,38,MM improved target rank,5,0.500000,"Cajun/Creole (4), Restaurants (4), Southern (2...",Coterie Restaurant & Oyster Bar (4 shared cate...,Pro tip: Go early (8:00 a.m.) during the weeke...,0.943879,True,"inside, outside, food",0.869333,0.722194,635.0,NaN
1,B — High-Diversity Visual Gain,Herbsaint,44,6,38,MM improved target rank,10,1.000000,"Restaurants (8), French (2), Seafood (2)",Commander's Palace (2 shared categories: Frenc...,Herbsaint has been on a running list of NOLA r...,0.944530,True,"inside, outside, food",0.902639,0.930911,762.0,NaN
2,C — Visual Counterexample,Emeril's,19,49,-30,MM worsened target rank,3,1.000000,"Restaurants (3), American (New) (1), Cajun/Cre...","Luke (4 shared categories: American (New), Caj...",What a great dining experience here overall. W...,0.960541,True,"inside, drink, food",0.876507,0.756464,622.0,NaN
3,D — Image-less Competitive Penalty,Witches Brew Tours,542,759,-217,MM worsened target rank,11,0.142857,Nightlife (3),New Orleans Original Daiquiris (1 shared categ...,My friends and I recently got home from a week...,0.937130,False,Visual evidence unavailable,NaN,NaN,NaN,-578.0
4,E — Sparse-User Visual Gain,Ruby Slipper - New Orleans,52,13,39,MM improved target rank,5,0.600000,"Restaurants (4), Breakfast & Brunch (2), Ameri...",Russell's Marina Grill (3 shared categories: A...,This was our first time in NOLA and per some r...,0.940481,True,"inside, menu, drink",0.854159,0.857512,713.0,NaN


### K5.4 — Grounded Recommendation-Level Explanations

Structured case evidence is converted into concise human-readable explanation
summaries.

The explanations distinguish between personalised historical evidence,
representative textual and visual evidence, recommendation outcomes and
frozen-model sensitivity.

The wording deliberately avoids causal expressions such as "recommended
because". Instead, each explanation reports evidence associated with the
recommendation and highlights where different evidence sources agree or
conflict.

In [267]:
# --------------------------------------------------
# K5.4 — Grounded explanation generator
# --------------------------------------------------

def format_percentage(
    value
):
    return (
        f"{value * 100:.1f}%"
    )


def build_grounded_case_explanation(
    row
):

    business_name = str(
        row[
            "business_name"
        ]
    )


    nv_rank = int(
        row[
            "nv_rank"
        ]
    )


    mm_rank = int(
        row[
            "mm_rank"
        ]
    )


    rank_improvement = int(
        row[
            "rank_improvement"
        ]
    )


    history_count = int(
        row[
            "training_history_count"
        ]
    )


    support_fraction = float(
        row[
            "category_support_fraction"
        ]
    )


    supported_categories = str(
        row[
            "supported_categories"
        ]
    )


    historical_match = str(
        row[
            "strongest_historical_match"
        ]
    )


    # ---------------------------------------------
    # Recommendation outcome
    # ---------------------------------------------

    if rank_improvement > 0:

        outcome_sentence = (
            f"KGRec-MM ranked {business_name} at position "
            f"{mm_rank}, compared with position {nv_rank} "
            f"under KGRec-NV, an improvement of "
            f"{rank_improvement} positions."
        )

    elif rank_improvement < 0:

        outcome_sentence = (
            f"KGRec-MM ranked {business_name} at position "
            f"{mm_rank}, compared with position {nv_rank} "
            f"under KGRec-NV, a deterioration of "
            f"{abs(rank_improvement)} positions."
        )

    else:

        outcome_sentence = (
            f"KGRec-MM and KGRec-NV both ranked "
            f"{business_name} at position {mm_rank}."
        )


    # ---------------------------------------------
    # Personalised historical evidence
    # ---------------------------------------------

    history_sentence = (
        f"The user had {history_count} businesses in the "
        f"training history. "
        f"{format_percentage(support_fraction)} of the "
        f"target's frozen-KG categories were also represented "
        f"in that history. Supported categories included "
        f"{supported_categories}. The strongest historical "
        f"business match was {historical_match}."
    )


    # ---------------------------------------------
    # Text evidence
    # ---------------------------------------------

    text_similarity = float(
        row[
            "representative_text_similarity"
        ]
    )


    text_sentence = (
        f"The most representative training review had "
        f"cosine similarity {text_similarity:.3f} to the "
        f"pooled BGE business representation. Its excerpt was: "
        f"\"{row['representative_text_excerpt']}\""
    )


    # ---------------------------------------------
    # Visual / missing-modality evidence
    # ---------------------------------------------

    if bool(
        row[
            "visual_available"
        ]
    ):

        visual_similarity = float(
            row[
                "representative_visual_similarity"
            ]
        )


        visual_score_sensitivity = float(
            row[
                "target_visual_score_sensitivity"
            ]
        )


        visual_rank_sensitivity = float(
            row[
                "target_visual_rank_sensitivity"
            ]
        )


        visual_sentence = (
            f"Representative CLIP evidence covered "
            f"{row['representative_visual_labels']}, with "
            f"maximum image-to-business similarity "
            f"{visual_similarity:.3f}. Within the frozen "
            f"multimodal model, removing the target's visual "
            f"evidence reduced its score by "
            f"{visual_score_sensitivity:.3f} and its rank by "
            f"{visual_rank_sensitivity:.0f} positions."
        )

    else:

        competitive_effect = float(
            row[
                "competitive_visual_rank_effect"
            ]
        )


        visual_sentence = (
            f"No visual representation was available for "
            f"{business_name}. When competing businesses retained "
            f"their visual evidence, the target's rank was "
            f"{abs(competitive_effect):.0f} positions lower than "
            f"in the all-visuals-masked condition."
        )


    # ---------------------------------------------
    # Interpretation
    # ---------------------------------------------

    if (
        rank_improvement > 0
        and bool(
            row[
                "visual_available"
            ]
        )
    ):

        interpretation = (
            "The personalised category evidence, representative "
            "business evidence and visual sensitivity are therefore "
            "consistent with the observed multimodal rank improvement. "
            "However, these signals are associative model evidence "
            "rather than causal attribution."
        )

    elif (
        rank_improvement < 0
        and bool(
            row[
                "visual_available"
            ]
        )
    ):

        interpretation = (
            "Despite substantial personalised, textual and visual "
            "evidence, the multimodal model ranked the target worse "
            "than the non-visual model. This demonstrates that "
            "evidence availability and visual sensitivity do not "
            "guarantee an overall multimodal recommendation gain."
        )

    else:

        interpretation = (
            "The case is consistent with the competitive "
            "missing-modality pattern identified in the aggregate "
            "analysis, although visual availability is also associated "
            "with differences in business support and metadata richness."
        )


    explanation = " ".join([
        outcome_sentence,
        history_sentence,
        text_sentence,
        visual_sentence,
        interpretation
    ])


    return explanation

In [268]:
k54_case_explanations = (
    k53_explanation_summary
    .copy()
)


k54_case_explanations[
    "grounded_explanation"
] = (
    k54_case_explanations
    .apply(
        build_grounded_case_explanation,
        axis=1
    )
)

In [269]:
for _, row in k54_case_explanations.iterrows():

    print(
        "\n"
        +
        "=" * 90
    )

    print(
        row[
            "case_type"
        ]
    )

    print(
        "=" * 90
    )

    print(
        row[
            "grounded_explanation"
        ]
    )


A — Visual Top-20 Gain
KGRec-MM ranked Mother's Restaurant at position 18, compared with position 56 under KGRec-NV, an improvement of 38 positions. The user had 5 businesses in the training history. 50.0% of the target's frozen-KG categories were also represented in that history. Supported categories included Cajun/Creole (4), Restaurants (4), Southern (2), American (New) (1), Breakfast & Brunch (1), Food (1). The strongest historical business match was Coterie Restaurant & Oyster Bar (4 shared categories: Breakfast & Brunch, Cajun/Creole, Restaurants, Southern). The most representative training review had cosine similarity 0.944 to the pooled BGE business representation. Its excerpt was: "Pro tip: Go early (8:00 a.m.) during the weekend to avoid long lines. Totes worth it, even if you're hungover. My friend and I ate breakfast here on our way out of NOLA. I got the ham po'boy. I know, I know, it's pretty tame, but after several days of heavy Cajun cuisine, I was really looking forwa

In [270]:
k54_explanation_check = pd.Series({

    "Five explanations generated":
        (
            len(
                k54_case_explanations
            )
            ==
            5
        ),

    "No missing explanations":
        k54_case_explanations[
            "grounded_explanation"
        ]
        .notna()
        .all(),

    "All explanations non-empty":
        (
            k54_case_explanations[
                "grounded_explanation"
            ]
            .str.len()
            >
            0
        ).all(),

    "Image-less case does not claim visual representation":
        (
            "No visual representation was available"
            in
            k54_case_explanations.loc[
                k54_case_explanations[
                    "case_type"
                ]
                ==
                "D — Image-less Competitive Penalty",
                "grounded_explanation"
            ].iloc[0]
        ),

    "Counterexample explicitly reports limitation":
        (
            "do not guarantee"
            in
            k54_case_explanations.loc[
                k54_case_explanations[
                    "case_type"
                ]
                ==
                "C — Visual Counterexample",
                "grounded_explanation"
            ].iloc[0]
        )
})


k54_explanation_check

Five explanations generated                             True
No missing explanations                                 True
All explanations non-empty                              True
Image-less case does not claim visual representation    True
Counterexample explicitly reports limitation            True
dtype: bool

In [273]:
# --------------------------------------------------
# Restore correct explanation output paths
# --------------------------------------------------

final_model_dir = (
    processed_data_root
    /
    "new_orleans_model_outputs"
)

explanation_dir = (
    final_model_dir
    /
    "explanations"
)

explanation_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Final model directory:",
    final_model_dir
)

print(
    "Explanation directory:",
    explanation_dir
)

print(
    "Final model directory exists:",
    final_model_dir.exists()
)

print(
    "Explanation directory exists:",
    explanation_dir.exists()
)

Final model directory: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_model_outputs
Explanation directory: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_model_outputs/explanations
Final model directory exists: True
Explanation directory exists: True


In [274]:
k54_case_explanations.to_csv(
    explanation_dir
    /
    "k5_grounded_case_explanations.csv",

    index=False
)

In [275]:
k54_save_check = pd.Series({

    "Grounded explanations saved":
        (
            explanation_dir
            /
            "k5_grounded_case_explanations.csv"
        ).exists()
})


k54_save_check

Grounded explanations saved    True
dtype: bool

## K6 — Quantitative and Explanation-Evidence Consistency

The final explanation analysis compares the quantitative recommendation
findings with evidence obtained from global fusion analysis, frozen-model
perturbation, representative multimodal evidence and systematically selected
recommendation cases.

The objective is not to require perfect agreement between quantitative and
explanation results. Instead, the analysis identifies where explanation
evidence:

- supports a quantitative pattern;
- qualifies its interpretation;
- reveals heterogeneity or counterexamples.

This synthesis directly addresses RQ4.

In [276]:
# --------------------------------------------------
# K6.1 — RQ4 quantitative ↔ explanation consistency
# --------------------------------------------------

rq4_consistency_matrix = pd.DataFrame([
    {
        "theme":
            "Overall effect of visual information",

        "quantitative_evidence":
            (
                "KGRec-MM improved 8 of 9 top-K metrics relative "
                "to KGRec-NV. The primary NDCG@10 effect was "
                "positive (+2.52%) but its paired 95% CI included zero. "
                "Clear positive differences were observed for Recall@20 "
                "and NDCG@20."
            ),

        "explanation_evidence":
            (
                "Within the frozen KGRec-MM model, disabling visual "
                "evidence reduced all nine top-K metrics. Seven of nine "
                "visual-ablation confidence intervals excluded zero, "
                "including NDCG@10."
            ),

        "case_evidence":
            (
                "Mother's Restaurant, Herbsaint and Ruby Slipper moved "
                "substantially upward under KGRec-MM, while Emeril's "
                "provided a counterexample."
            ),

        "consistency":
            "Consistent with qualification",

        "interpretation":
            (
                "The trained multimodal model actively uses visual "
                "information, but visual utilisation does not guarantee "
                "a statistically clear MM-over-NV advantage at every cutoff."
            )
    },

    {
        "theme":
            "Global modality balance",

        "quantitative_evidence":
            (
                "Strong target-level visual sensitivity was observed "
                "during frozen-model perturbation."
            ),

        "explanation_evidence":
            (
                "Global fusion weights were approximately balanced: "
                "graph 0.335, text 0.330 and visual 0.335, with "
                "normalised fusion entropy approximately 1."
            ),

        "case_evidence":
            (
                "Large visual-ablation rank sensitivities occurred "
                "despite the absence of a globally dominant visual weight."
            ),

        "consistency":
            "Consistent with qualification",

        "interpretation":
            (
                "Visual sensitivity is not explained by simple scalar "
                "domination of the fusion layer. Global fusion weight "
                "should therefore not be interpreted as modality importance."
            )
    },

    {
        "theme":
            "Visual label diversity",

        "quantitative_evidence":
            (
                "After adjustment and business-clustered inference, visual "
                "label diversity was positively associated with NDCG@10, "
                "Recall@20 and NDCG@20 multimodal gains, whereas image "
                "quantity alone was not clearly associated with top-K gain."
            ),

        "explanation_evidence":
            (
                "Representative visual evidence preserved multiple visual "
                "labels rather than selecting redundant photographs."
            ),

        "case_evidence":
            (
                "Herbsaint had four visual labels and moved from rank 44 "
                "to 6. Ruby Slipper had five labels and moved from 52 to 13. "
                "Emeril's had three labels but nevertheless worsened."
            ),

        "consistency":
            "Consistent with qualification",

        "interpretation":
            (
                "The case evidence is compatible with the aggregate "
                "visual-diversity result, while the Emeril's counterexample "
                "shows that diversity is associated with stronger gains "
                "rather than deterministically producing them."
            )
    },

    {
        "theme":
            "Image quantity",

        "quantitative_evidence":
            (
                "After controlling for business support, metadata richness "
                "and user history, raw image quantity did not show clear "
                "top-K multimodal effects."
            ),

        "explanation_evidence":
            (
                "All four visually supported case businesses had five "
                "selected images, yet their recommendation outcomes differed."
            ),

        "case_evidence":
            (
                "Mother's Restaurant, Herbsaint and Ruby Slipper improved, "
                "whereas Emeril's worsened despite also having five images."
            ),

        "consistency":
            "Consistent",

        "interpretation":
            (
                "The cases reinforce the quantitative result that simply "
                "having more selected images is insufficient to explain "
                "multimodal recommendation improvement."
            )
    },

    {
        "theme":
            "User-history sparsity",

        "quantitative_evidence":
            (
                "Sparse-history users showed clearer Recall@20 and NDCG@20 "
                "multimodal gains than dense-history users."
            ),

        "explanation_evidence":
            (
                "Personalised category-overlap analysis exposes how much "
                "structured historical evidence is available for each target."
            ),

        "case_evidence":
            (
                "Mother's Restaurant and Ruby Slipper both involved users "
                "with five training interactions and entered the top-20 "
                "under KGRec-MM. Emeril's involved only three interactions "
                "but still worsened."
            ),

        "consistency":
            "Consistent with heterogeneity",

        "interpretation":
            (
                "Sparse-history users can benefit from multimodal evidence, "
                "but sparsity alone does not determine the direction of "
                "an individual recommendation outcome."
            )
    },

    {
        "theme":
            "Business interaction support",

        "quantitative_evidence":
            (
                "Low-support visual businesses exhibited large average rank "
                "movements but generally remained outside the top-20, while "
                "top-K effects were concentrated among higher-support businesses."
            ),

        "explanation_evidence":
            (
                "Case profiles combine business support with structured, "
                "textual and visual evidence rather than treating rank movement "
                "alone as recommendation success."
            ),

        "case_evidence":
            (
                "Successful top-20 cases included Mother's Restaurant "
                "(601 training interactions), Herbsaint (216) and Ruby Slipper "
                "(676), illustrating how established support can help translate "
                "representation changes into top-K exposure."
            ),

        "consistency":
            "Consistent with qualification",

        "interpretation":
            (
                "Business support appears relevant to whether visual-induced "
                "rank movement translates into top-K exposure, although it "
                "does not guarantee improvement."
            )
    },

    {
        "theme":
            "Missing visual modality",

        "quantitative_evidence":
            (
                "Image-less targets were substantially weaker and 1723 of "
                "1725 ranked lower when visually supported competitors retained "
                "their visual representations compared with the all-visuals-off "
                "condition."
            ),

        "explanation_evidence":
            (
                "The model correctly masks unavailable visual information "
                "and renormalises graph and text weights rather than treating "
                "zero placeholder vectors as genuine visual evidence."
            ),

        "case_evidence":
            (
                "Witches Brew Tours had no visual representation and ranked "
                "578 positions lower when competitors retained visual evidence "
                "than in the all-visuals-masked condition."
            ),

        "consistency":
            "Consistent",

        "interpretation":
            (
                "The case-level explanation reproduces the aggregate "
                "competitive missing-modality pattern, while business-support "
                "and metadata confounding prevent a causal fairness interpretation."
            )
    },

    {
        "theme":
            "Evidence availability versus recommendation success",

        "quantitative_evidence":
            (
                "Aggregate multimodal effects were heterogeneous across users "
                "and businesses."
            ),

        "explanation_evidence":
            (
                "High category overlap, representative text and strong visual "
                "sensitivity can coexist without an MM-over-NV rank improvement."
            ),

        "case_evidence":
            (
                "Emeril's had 100% historical target-category coverage, "
                "representative text similarity 0.961 and target visual-ablation "
                "sensitivity of 622 positions, yet its rank worsened from "
                "19 under KGRec-NV to 49 under KGRec-MM."
            ),

        "consistency":
            "Counterexample / heterogeneity",

        "interpretation":
            (
                "Explanation evidence should not be converted into a simple "
                "post-hoc justification of recommendation success. The model "
                "combines interacting signals whose net effect can differ from "
                "the apparent strength of individual evidence channels."
            )
    }
])


rq4_consistency_matrix

,theme,quantitative_evidence,explanation_evidence,case_evidence,consistency,interpretation
0,Overall effect of visual information,KGRec-MM improved 8 of 9 top-K metrics relativ...,"Within the frozen KGRec-MM model, disabling vi...","Mother's Restaurant, Herbsaint and Ruby Slippe...",Consistent with qualification,The trained multimodal model actively uses vis...
1,Global modality balance,Strong target-level visual sensitivity was obs...,Global fusion weights were approximately balan...,Large visual-ablation rank sensitivities occur...,Consistent with qualification,Visual sensitivity is not explained by simple ...
2,Visual label diversity,After adjustment and business-clustered infere...,Representative visual evidence preserved multi...,Herbsaint had four visual labels and moved fro...,Consistent with qualification,The case evidence is compatible with the aggre...
3,Image quantity,"After controlling for business support, metada...",All four visually supported case businesses ha...,"Mother's Restaurant, Herbsaint and Ruby Slippe...",Consistent,The cases reinforce the quantitative result th...
4,User-history sparsity,Sparse-history users showed clearer Recall@20 ...,Personalised category-overlap analysis exposes...,Mother's Restaurant and Ruby Slipper both invo...,Consistent with heterogeneity,Sparse-history users can benefit from multimod...
5,Business interaction support,Low-support visual businesses exhibited large ...,Case profiles combine business support with st...,Successful top-20 cases included Mother's Rest...,Consistent with qualification,Business support appears relevant to whether v...
6,Missing visual modality,Image-less targets were substantially weaker a...,The model correctly masks unavailable visual i...,Witches Brew Tours had no visual representatio...,Consistent,The case-level explanation reproduces the aggr...
7,Evidence availability versus recommendation su...,Aggregate multimodal effects were heterogeneou...,"High category overlap, representative text and...",Emeril's had 100% historical target-category c...,Counterexample / heterogeneity,Explanation evidence should not be converted i...


In [277]:
rq4_consistency_counts = (
    rq4_consistency_matrix[
        "consistency"
    ]
    .value_counts()
)


rq4_consistency_counts

consistency
Consistent with qualification     4
Consistent                        2
Consistent with heterogeneity     1
Counterexample / heterogeneity    1
Name: count, dtype: int64

In [278]:
rq4_consistency_counts.sum()

8

In [279]:
rq4_final_synthesis = pd.DataFrame([
    {
        "rq":
            "RQ4",

        "research_question":
            (
                "To what extent are the patterns observed in quantitative "
                "recommendation performance consistent with the evidence "
                "revealed through explanation and model-analysis methods?"
            ),

        "answer":
            (
                "The explanation and model-analysis evidence was broadly "
                "consistent with the quantitative recommendation findings, "
                "but also revealed important heterogeneity and limitations. "
                "Frozen-model perturbation confirmed that the multimodal "
                "recommender actively used visual information, while balanced "
                "global fusion weights showed that this sensitivity was not "
                "caused by a globally dominant visual coefficient. "
                "Recommendation-level evidence was compatible with the "
                "quantitative findings on visual diversity, user-history "
                "sparsity, business support and missing visual information. "
                "However, counterexamples such as Emeril's demonstrated that "
                "strong structured, textual and visual evidence does not "
                "necessarily translate into superior multimodal ranking. "
                "The results therefore support interpreting the explanation "
                "layer as complementary evidence about model behaviour rather "
                "than as deterministic or causal justification of individual "
                "recommendations."
            ),

        "overall_conclusion":
            (
                "Broad consistency with meaningful qualification "
                "and recommendation-level heterogeneity."
            )
    }
])


rq4_final_synthesis

,rq,research_question,answer,overall_conclusion
0,RQ4,To what extent are the patterns observed in qu...,The explanation and model-analysis evidence wa...,Broad consistency with meaningful qualificatio...


In [281]:
rq4_consistency_matrix.to_csv(
    explanation_dir
    /
    "k6_rq4_consistency_matrix.csv",

    index=False
)


rq4_final_synthesis.to_csv(
    explanation_dir
    /
    "k6_rq4_final_synthesis.csv",

    index=False
)

In [282]:
k6_save_check = pd.Series({

    "RQ4 consistency matrix saved":
        (
            explanation_dir
            /
            "k6_rq4_consistency_matrix.csv"
        ).exists(),

    "RQ4 synthesis saved":
        (
            explanation_dir
            /
            "k6_rq4_final_synthesis.csv"
        ).exists()
})


k6_save_check

RQ4 consistency matrix saved    True
RQ4 synthesis saved             True
dtype: bool

In [283]:
k6_integrity_check = pd.Series({

    "Eight synthesis themes":
        (
            len(
                rq4_consistency_matrix
            )
            ==
            8
        ),

    "No missing quantitative evidence":
        rq4_consistency_matrix[
            "quantitative_evidence"
        ]
        .notna()
        .all(),

    "No missing explanation evidence":
        rq4_consistency_matrix[
            "explanation_evidence"
        ]
        .notna()
        .all(),

    "No missing case evidence":
        rq4_consistency_matrix[
            "case_evidence"
        ]
        .notna()
        .all(),

    "No missing interpretation":
        rq4_consistency_matrix[
            "interpretation"
        ]
        .notna()
        .all(),

    "RQ4 synthesis exists":
        (
            len(
                rq4_final_synthesis
            )
            ==
            1
        )
})


k6_integrity_check

Eight synthesis themes              True
No missing quantitative evidence    True
No missing explanation evidence     True
No missing case evidence            True
No missing interpretation           True
RQ4 synthesis exists                True
dtype: bool

# Final Experimental Integrity and Reproducibility Audit

This final audit verifies that the dissertation experiment terminates with a
consistent set of frozen recommendation models, evaluation outputs, analysis
artefacts and explanation results.

The audit does not perform additional training, model selection or test-set
analysis. Its purpose is to confirm the provenance, completeness and internal
consistency of the artefacts used to answer RQ1–RQ4.

In [284]:
# --------------------------------------------------
# F1 — Core experimental dimensions
# --------------------------------------------------

final_scope_check = pd.Series({

    "Personalisation businesses":
        len(
            business_index
        ),

    "Expected 2516 businesses":
        (
            len(
                business_index
            )
            ==
            2516
        ),

    "Users":
        len(
            user_index
        ),

    "Expected 14991 users":
        (
            len(
                user_index
            )
            ==
            14991
        ),

    "KG entities":
        len(
            entity_index
        ),

    "Expected 17819 entities":
        (
            len(
                entity_index
            )
            ==
            17819
        ),

    "Base KG relations":
        len(
            relation_index
        ),

    "Expected 64 relations":
        (
            len(
                relation_index
            )
            ==
            64
        ),

    "Test cases":
        len(
            rq3_analysis
        ),

    "One test case per user":
        (
            len(
                rq3_analysis
            )
            ==
            14991
        )
})


final_scope_check

Personalisation businesses     2516
Expected 2516 businesses       True
Users                         14991
Expected 14991 users           True
KG entities                   17819
Expected 17819 entities        True
Base KG relations                64
Expected 64 relations          True
Test cases                    14991
One test case per user         True
dtype: object

In [285]:
final_modality_check = pd.Series({

    "Visual-supported businesses":
        int(
            business_evidence_coverage[
                "has_visual_evidence"
            ].sum()
        ),

    "Expected visual businesses":
        (
            business_evidence_coverage[
                "has_visual_evidence"
            ].sum()
            ==
            1729
        ),

    "Image-less businesses":
        int(
            (
                ~business_evidence_coverage[
                    "has_visual_evidence"
                ]
            ).sum()
        ),

    "Expected image-less businesses":
        (
            (
                ~business_evidence_coverage[
                    "has_visual_evidence"
                ]
            ).sum()
            ==
            787
        ),

    "Selected personalisation images":
        len(
            visual_image_lookup
        ),

    "Expected selected images":
        (
            len(
                visual_image_lookup
            )
            ==
            6122
        ),

    "Training review evidence":
        len(
            training_review_evidence
        ),

    "Expected training reviews":
        (
            len(
                training_review_evidence
            )
            ==
            122233
        )
})


final_modality_check

Visual-supported businesses          1729
Expected visual businesses           True
Image-less businesses                 787
Expected image-less businesses       True
Selected personalisation images      6122
Expected selected images             True
Training review evidence           122233
Expected training reviews            True
dtype: object

In [286]:
final_checkpoint_check = pd.Series({

    "KGRec-NV checkpoint":
        nv_checkpoint_path.exists(),

    "KGRec-MM checkpoint":
        mm_checkpoint_path.exists(),

    "NV checkpoint non-empty":
        (
            nv_checkpoint_path.stat().st_size
            >
            0
        ),

    "MM checkpoint non-empty":
        (
            mm_checkpoint_path.stat().st_size
            >
            0
        )
})


final_checkpoint_check

KGRec-NV checkpoint        True
KGRec-MM checkpoint        True
NV checkpoint non-empty    True
MM checkpoint non-empty    True
dtype: bool

In [287]:
import hashlib


def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )


            if not chunk:
                break


            digest.update(
                chunk
            )


    return digest.hexdigest()

In [288]:
checkpoint_hash_manifest = pd.DataFrame([
    {
        "model":
            "KGRec-NV",

        "checkpoint":
            nv_checkpoint_path.name,

        "sha256":
            sha256_file(
                nv_checkpoint_path
            )
    },

    {
        "model":
            "KGRec-MM",

        "checkpoint":
            mm_checkpoint_path.name,

        "sha256":
            sha256_file(
                mm_checkpoint_path
            )
    }
])


checkpoint_hash_manifest

,model,checkpoint,sha256
0,KGRec-NV,kgrec_nv_final_checkpoint.pt,254b564ec3a1fd601c0a328d66f855985cee161dc4fd2b...
1,KGRec-MM,kgrec_mm_final_checkpoint.pt,6faba05bf03c9fa3363b266a9f5a3d227f58a1f789afcf...


In [289]:
def final_metrics_from_ranks(
    ranks
):

    ranks = np.asarray(
        ranks,
        dtype=float
    )


    results = {}


    for k in [
        5,
        10,
        20
    ]:

        hit = (
            ranks
            <=
            k
        )


        results[
            f"Recall@{k}"
        ] = float(
            hit.mean()
        )


        results[
            f"NDCG@{k}"
        ] = float(
            np.where(
                hit,
                1.0
                /
                np.log2(
                    ranks + 1
                ),
                0.0
            ).mean()
        )


        results[
            f"MAP@{k}"
        ] = float(
            np.where(
                hit,
                1.0
                /
                ranks,
                0.0
            ).mean()
        )


    results[
        "MeanRank"
    ] = float(
        ranks.mean()
    )


    results[
        "MedianRank"
    ] = float(
        np.median(
            ranks
        )
    )


    return results

In [290]:
final_nv_metrics = (
    final_metrics_from_ranks(
        rq3_analysis[
            "nv_rank"
        ]
    )
)


final_mm_metrics = (
    final_metrics_from_ranks(
        rq3_analysis[
            "mm_rank"
        ]
    )
)


final_metric_audit = pd.DataFrame({

    "metric":
        list(
            final_nv_metrics.keys()
        ),

    "KGRec-NV":
        list(
            final_nv_metrics.values()
        ),

    "KGRec-MM":
        [
            final_mm_metrics[
                metric
            ]
            for metric
            in final_nv_metrics.keys()
        ]
})


final_metric_audit

,metric,KGRec-NV,KGRec-MM
0,Recall@5,0.040424,0.041692
1,NDCG@5,0.026427,0.026689
2,MAP@5,0.021831,0.021785
3,Recall@10,0.064305,0.067574
4,NDCG@10,0.034106,0.034965
5,MAP@10,0.024975,0.025146
6,Recall@20,0.100861,0.111000
7,NDCG@20,0.043217,0.045860
8,MAP@20,0.027407,0.028093
9,MeanRank,485.083050,492.093456


In [291]:
expected_final_metrics = {

    "Recall@5":
        (0.040424, 0.041692),

    "NDCG@5":
        (0.026427, 0.026689),

    "MAP@5":
        (0.021831, 0.021785),

    "Recall@10":
        (0.064305, 0.067574),

    "NDCG@10":
        (0.034106, 0.034965),

    "MAP@10":
        (0.024975, 0.025146),

    "Recall@20":
        (0.100861, 0.111000),

    "NDCG@20":
        (0.043217, 0.045860),

    "MAP@20":
        (0.027407, 0.028093)
}

In [292]:
metric_reproduction_rows = []


for metric, (
    expected_nv,
    expected_mm
) in expected_final_metrics.items():

    observed_nv = (
        final_nv_metrics[
            metric
        ]
    )


    observed_mm = (
        final_mm_metrics[
            metric
        ]
    )


    metric_reproduction_rows.append({

        "metric":
            metric,

        "expected_nv":
            expected_nv,

        "observed_nv":
            observed_nv,

        "nv_matches":
            np.isclose(
                observed_nv,
                expected_nv,
                atol=1e-6
            ),

        "expected_mm":
            expected_mm,

        "observed_mm":
            observed_mm,

        "mm_matches":
            np.isclose(
                observed_mm,
                expected_mm,
                atol=1e-6
            )
    })


metric_reproduction_check = pd.DataFrame(
    metric_reproduction_rows
)


metric_reproduction_check

,metric,expected_nv,observed_nv,nv_matches,expected_mm,observed_mm,mm_matches
0,Recall@5,0.040424,0.040424,True,0.041692,0.041692,True
1,NDCG@5,0.026427,0.026427,True,0.026689,0.026689,True
2,MAP@5,0.021831,0.021831,True,0.021785,0.021785,True
3,Recall@10,0.064305,0.064305,True,0.067574,0.067574,True
4,NDCG@10,0.034106,0.034106,True,0.034965,0.034965,True
5,MAP@10,0.024975,0.024975,True,0.025146,0.025146,True
6,Recall@20,0.100861,0.100861,True,0.111000,0.111000,True
7,NDCG@20,0.043217,0.043217,True,0.045860,0.045860,True
8,MAP@20,0.027407,0.027407,True,0.028093,0.028093,True


In [293]:
core_analysis_files = {

    "Paired user test results":
        final_model_dir
        /
        "paired_user_test_results.parquet",

    "Paired bootstrap":
        final_model_dir
        /
        "paired_test_bootstrap_summary.csv",

    "RQ3 paired analysis":
        final_model_dir
        /
        "rq3_paired_test_analysis.parquet",

    "Visual perturbation":
        final_model_dir
        /
        "rq3_mm_visual_perturbation_metrics.csv",

    "Business-level RQ3":
        final_model_dir
        /
        "rq3_visual_business_level_analysis.parquet",

    "Adjusted rank effects":
        final_model_dir
        /
        "rq3_adjusted_rank_visual_effects.csv",

    "Clustered top-K effects":
        final_model_dir
        /
        "rq3_clustered_topk_visual_effects.csv",

    "Interaction support analysis":
        final_model_dir
        /
        "rq3_interaction_support_bootstrap.csv",

    "User-history analysis":
        final_model_dir
        /
        "rq3_user_history_bootstrap.csv"
}


core_analysis_file_check = pd.Series({

    name:
        path.exists()

    for name, path
    in core_analysis_files.items()
})


core_analysis_file_check

Paired user test results        True
Paired bootstrap                True
RQ3 paired analysis             True
Visual perturbation             True
Business-level RQ3              True
Adjusted rank effects           True
Clustered top-K effects         True
Interaction support analysis    True
User-history analysis           True
dtype: bool

In [294]:
final_explanation_files = {

    "K2 global fusion":
        explanation_dir
        /
        "k2_global_fusion_summary.csv",

    "K3 visual ablation":
        explanation_dir
        /
        "k3_catalogue_visual_ablation_summary.csv",

    "K3 bootstrap":
        explanation_dir
        /
        "k3_catalogue_visual_ablation_bootstrap.csv",

    "K4 training review evidence":
        explanation_dir
        /
        "k4_training_review_evidence.parquet",

    "K4 text representativeness":
        explanation_dir
        /
        "k4_review_text_representativeness.parquet",

    "K4 structured evidence":
        explanation_dir
        /
        "k4_structured_business_evidence.parquet",

    "K4 visual representativeness":
        explanation_dir
        /
        "k4_visual_image_representativeness.parquet",

    "K4 unified coverage":
        explanation_dir
        /
        "k4_unified_business_evidence_coverage.parquet",

    "K5 selected cases":
        explanation_dir
        /
        "k5_selected_recommendation_cases.csv",

    "K5 behaviour":
        explanation_dir
        /
        "k5_case_behaviour_summary.csv",

    "K5 user-category evidence":
        explanation_dir
        /
        "k5_case_user_category_overlap.csv",

    "K5 grounded explanations":
        explanation_dir
        /
        "k5_grounded_case_explanations.csv",

    "K6 RQ4 matrix":
        explanation_dir
        /
        "k6_rq4_consistency_matrix.csv",

    "K6 final synthesis":
        explanation_dir
        /
        "k6_rq4_final_synthesis.csv"
}


final_explanation_file_check = pd.Series({

    name:
        path.exists()

    for name, path
    in final_explanation_files.items()
})


final_explanation_file_check

K2 global fusion                True
K3 visual ablation              True
K3 bootstrap                    True
K4 training review evidence     True
K4 text representativeness      True
K4 structured evidence          True
K4 visual representativeness    True
K4 unified coverage             True
K5 selected cases               True
K5 behaviour                    True
K5 user-category evidence       True
K5 grounded explanations        True
K6 RQ4 matrix                   True
K6 final synthesis              True
dtype: bool

In [295]:
final_explanation_integrity = pd.Series({

    "Five systematic cases":
        (
            len(
                selected_case_summary
            )
            ==
            5
        ),

    "Five grounded explanations":
        (
            len(
                k54_case_explanations
            )
            ==
            5
        ),

    "Five unique case users":
        (
            selected_case_summary[
                "user_row"
            ].nunique()
            ==
            5
        ),

    "Four visual cases":
        (
            k52_case_behaviour[
                "target_has_visual"
            ].sum()
            ==
            4
        ),

    "One image-less case":
        (
            (
                ~k52_case_behaviour[
                    "target_has_visual"
                ]
            ).sum()
            ==
            1
        ),

    "Eight RQ4 synthesis themes":
        (
            len(
                rq4_consistency_matrix
            )
            ==
            8
        ),

    "One final RQ4 synthesis":
        (
            len(
                rq4_final_synthesis
            )
            ==
            1
        )
})


final_explanation_integrity

Five systematic cases         True
Five grounded explanations    True
Five unique case users        True
Four visual cases             True
One image-less case           True
Eight RQ4 synthesis themes    True
One final RQ4 synthesis       True
dtype: bool

In [296]:
final_experiment_manifest = pd.DataFrame([
    {
        "component":
            "Personalisation catalogue",

        "value":
            2516,

        "status":
            "Frozen"
    },

    {
        "component":
            "Users",

        "value":
            14991,

        "status":
            "Frozen"
    },

    {
        "component":
            "Training interactions",

        "value":
            122233,

        "status":
            "Frozen"
    },

    {
        "component":
            "Validation interactions",

        "value":
            14991,

        "status":
            "Frozen"
    },

    {
        "component":
            "Test interactions",

        "value":
            14991,

        "status":
            "Frozen"
    },

    {
        "component":
            "Metadata KG triples",

        "value":
            83989,

        "status":
            "Frozen"
    },

    {
        "component":
            "Unified training KG triples",

        "value":
            206222,

        "status":
            "Frozen"
    },

    {
        "component":
            "Businesses with genuine visual features",

        "value":
            1729,

        "status":
            "Frozen"
    },

    {
        "component":
            "Image-less businesses",

        "value":
            787,

        "status":
            "Frozen"
    },

    {
        "component":
            "Selected personalisation images",

        "value":
            6122,

        "status":
            "Frozen"
    },

    {
        "component":
            "Selected explanation cases",

        "value":
            5,

        "status":
            "Frozen"
    },

    {
        "component":
            "RQ4 synthesis themes",

        "value":
            8,

        "status":
            "Complete"
    }
])


final_experiment_manifest

,component,value,status
0,Personalisation catalogue,2516,Frozen
1,Users,14991,Frozen
2,Training interactions,122233,Frozen
3,Validation interactions,14991,Frozen
4,Test interactions,14991,Frozen
5,Metadata KG triples,83989,Frozen
6,Unified training KG triples,206222,Frozen
7,Businesses with genuine visual features,1729,Frozen
8,Image-less businesses,787,Frozen
9,Selected personalisation images,6122,Frozen


In [297]:
final_experiment_manifest.to_csv(
    explanation_dir
    /
    "final_experiment_manifest.csv",

    index=False
)


checkpoint_hash_manifest.to_csv(
    explanation_dir
    /
    "final_model_checkpoint_hashes.csv",

    index=False
)


final_metric_audit.to_csv(
    explanation_dir
    /
    "final_test_metric_reproduction.csv",

    index=False
)


metric_reproduction_check.to_csv(
    explanation_dir
    /
    "final_test_metric_reproduction_check.csv",

    index=False
)

In [298]:
final_master_check = pd.Series({

    "Scope audit passed":
        bool(
            final_scope_check[
                final_scope_check.index.str.startswith(
                    "Expected"
                )
                |
                final_scope_check.index.str.startswith(
                    "One test"
                )
            ]
            .astype(bool)
            .all()
        ),

    "Modality audit passed":
        bool(
            final_modality_check[
                [
                    "Expected visual businesses",
                    "Expected image-less businesses",
                    "Expected selected images",
                    "Expected training reviews"
                ]
            ]
            .astype(bool)
            .all()
        ),

    "Frozen checkpoints valid":
        bool(
            final_checkpoint_check
            .astype(bool)
            .all()
        ),

    "Final metrics reproduce":
        bool(
            metric_reproduction_check[
                [
                    "nv_matches",
                    "mm_matches"
                ]
            ]
            .to_numpy()
            .all()
        ),

    "RQ2/RQ3 artefacts complete":
        bool(
            core_analysis_file_check
            .all()
        ),

    "K2-K6 artefacts complete":
        bool(
            final_explanation_file_check
            .all()
        ),

    "Explanation integrity passed":
        bool(
            final_explanation_integrity
            .astype(bool)
            .all()
        )
})


final_master_check

Scope audit passed              True
Modality audit passed           True
Frozen checkpoints valid        True
Final metrics reproduce         True
RQ2/RQ3 artefacts complete      True
K2-K6 artefacts complete        True
Explanation integrity passed    True
dtype: bool